## there are also bugs and error of analysis here

# 2_shared_multiCore_XLM-R_diffWeightsBoth

In [1]:

# IMPORTS


import json
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns


# CONFIGURATION


OUTPUT_DIR = Path(
    r"C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth"
)

# ============================================================
# INPUT FILES
# ============================================================

LOGITS_FILE = (
    OUTPUT_DIR / "test_logits.npy"
)

LABELS_FILE = (
    OUTPUT_DIR / "test_labels.npy"
)

LABEL_NAMES_FILE = (
    OUTPUT_DIR / "hierarchical_labels.json"
)


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

ANALYSIS_DIR = (
    OUTPUT_DIR
    / "multilabel_error_analysis"
)

ANALYSIS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# LOAD DATA
# ============================================================

print("\n" + "=" * 100)
print("LOADING TEST DATA")
print("=" * 100)


test_logits = np.load(
    LOGITS_FILE
)

test_labels = np.load(
    LABELS_FILE
)


with open(
    LABEL_NAMES_FILE,
    "r",
    encoding="utf-8",
) as f:

    hierarchical_labels = json.load(f)


print(
    "Logits shape:",
    test_logits.shape,
)

print(
    "Labels shape:",
    test_labels.shape,
)

print(
    "Number of labels:",
    len(hierarchical_labels),
)


# ============================================================
# BASIC CHECKS
# ============================================================

if test_logits.shape != test_labels.shape:

    raise ValueError(
        "\nLogits and labels have different shapes:\n"
        f"logits = {test_logits.shape}\n"
        f"labels = {test_labels.shape}"
    )


NUM_LABELS = len(
    hierarchical_labels
)


if test_logits.shape[1] != NUM_LABELS:

    raise ValueError(
        "\nNumber of labels does not match "
        "logits columns."
    )


# ============================================================
# HIERARCHICAL STRUCTURE
# ============================================================

NUM_PARENTS = 9


parents = hierarchical_labels[
    :NUM_PARENTS
]


children = hierarchical_labels[
    NUM_PARENTS:
]


parent_indices = list(
    range(
        NUM_PARENTS
    )
)


child_indices = list(
    range(
        NUM_PARENTS,
        NUM_LABELS
    )
)


print("\nParents:")
print(parents)

print("\nChildren:")
print(children)


# ============================================================
# SIGMOID
# ============================================================

def sigmoid(x):

    return 1.0 / (
        1.0 + np.exp(-x)
    )


test_probabilities = sigmoid(
    test_logits
)


# THRESHOLD CONFIGURATION


# Global tuned threshold


GLOBAL_TUNED_THRESHOLD = 0.35



# Per-label tuned thresholds


PER_LABEL_THRESHOLDS = {

"MT"  : 0.10,
"LY"  : 0.50,
"SP"  : 0.53,
"ID"  : 0.66,
"NA"  : 0.44,
"HI"  : 0.24,
"IN"  : 0.46,
"OP"  : 0.35,
"IP"  : 0.28,
"it"  : 0.43,
"ne"  : 0.28,
"sr"  : 0.39,
"nb"  : 0.26,
"re"  : 0.49,
"en"  : 0.44,
"ra"  : 0.16,
"dtp" : 0.25,
"fi"  : 0.33,
"lt"  : 0.38,
"rv"  : 0.21,
"ob"  : 0.26,
"rs"  : 0.47,
"av"  : 0.45,
"ds"  : 0.53,
"ed"  : 0.32

}

# ------------------------------------------------------------
# Verify per-label thresholds
# ------------------------------------------------------------

missing_thresholds = [
    label
    for label in hierarchical_labels
    if label not in PER_LABEL_THRESHOLDS
]


if missing_thresholds:

    raise ValueError(
        "Missing thresholds for:\n"
        f"{missing_thresholds}"
    )


extra_thresholds = [
    label
    for label in PER_LABEL_THRESHOLDS
    if label not in hierarchical_labels
]


if extra_thresholds:

    raise ValueError(
        "Unknown labels in PER_LABEL_THRESHOLDS:\n"
        f"{extra_thresholds}"
    )


per_label_threshold_array = np.array(
    [
        PER_LABEL_THRESHOLDS[label]
        for label in hierarchical_labels
    ],
    dtype=np.float32,
)


# ============================================================
# MAKE PREDICTIONS
# ============================================================

def make_predictions(
    probabilities,
    thresholds,
):
    """
    probabilities:
        [N, number_of_labels]

    thresholds:
        either

        - one scalar
        - one threshold per label
    """

    thresholds = np.asarray(
        thresholds,
        dtype=np.float32,
    )


    if thresholds.ndim == 0:

        thresholds = np.repeat(
            thresholds,
            NUM_LABELS,
        )


    if len(thresholds) != NUM_LABELS:

        raise ValueError(
            "Threshold count does not match "
            f"number of labels ({NUM_LABELS})."
        )


    predictions = (
        probabilities
        >= thresholds.reshape(1, -1)
    ).astype(np.int32)


    return predictions


# ============================================================
# THREE SETS OF PREDICTIONS
# ============================================================

predictions_05 = make_predictions(
    test_probabilities,
    0.50,
)


predictions_global = make_predictions(
    test_probabilities,
    GLOBAL_TUNED_THRESHOLD,
)


predictions_per_label = make_predictions(
    test_probabilities,
    per_label_threshold_array,
)


# ============================================================
# HELPER:
# CREATE LABEL MATRIX
# ============================================================

def initialize_matrix(
    n_rows,
    n_cols,
):

    return np.zeros(
        (n_rows, n_cols),
        dtype=np.int64,
    )


# ============================================================
# 1. TRUE -> PREDICTED ASSOCIATION
# ============================================================
#
# Example:
#
# TRUE:
#     IN = 1
#
# PREDICTED:
#     IN = 1
#     OP = 1
#
# This contributes:
#
#     IN -> IN
#     IN -> OP
#
# It does NOT mean that IN was confused with OP.
#
# It means OP was also predicted in an IN-containing
# document.
#
# This is a CO-OCCURRENCE / ASSOCIATION matrix.
# ============================================================

def true_predicted_association(
    true_labels,
    predicted_labels,
):

    n_labels = true_labels.shape[1]


    matrix = initialize_matrix(
        n_labels,
        n_labels,
    )


    for i in range(
        true_labels.shape[0]
    ):

        true_indices = np.where(
            true_labels[i] == 1
        )[0]


        predicted_indices = np.where(
            predicted_labels[i] == 1
        )[0]


        for true_idx in true_indices:

            for pred_idx in predicted_indices:

                matrix[
                    true_idx,
                    pred_idx
                ] += 1


    return matrix


# ============================================================
# 2. GENUINE SUBSTITUTION ERROR
# ============================================================
#
# We count:
#
# TRUE label A = 1
# TRUE label B = 0
#
# PREDICTED A = 0
# PREDICTED B = 1
#
# Therefore:
#
# A was missed
# B was falsely predicted
#
# This is much closer to what we normally mean by:
#
#     "A was confused with B"
#
# in a multilabel setting.
# ============================================================

def substitution_error_matrix(
    true_labels,
    predicted_labels,
):

    n_labels = true_labels.shape[1]


    matrix = initialize_matrix(
        n_labels,
        n_labels,
    )


    for i in range(
        true_labels.shape[0]
    ):

        true_row = true_labels[i]

        predicted_row = (
            predicted_labels[i]
        )


        # Labels that were truly present
        # but missed.
        missed_indices = np.where(
            (
                true_row == 1
            )
            &
            (
                predicted_row == 0
            )
        )[0]


        # Labels that were not truly present
        # but falsely predicted.
        false_positive_indices = np.where(
            (
                true_row == 0
            )
            &
            (
                predicted_row == 1
            )
        )[0]


        for missed_idx in missed_indices:

            for fp_idx in false_positive_indices:

                matrix[
                    missed_idx,
                    fp_idx
                ] += 1


    return matrix


# ============================================================
# 3. FALSE-POSITIVE ASSOCIATION
# ============================================================
#
# Here we ask:
#
# "When label B was falsely predicted,
#  which true labels were present?"
#
# Example:
#
# TRUE:
#     OP = 1
#     IN = 0
#
# PREDICTED:
#     IN = 1
#
# Then:
#
#     OP -> IN
#
# is counted.
#
# This is particularly useful for answering:
#
#     "What true labels tend to accompany false IN
#      predictions?"
#
# ============================================================

def false_positive_association(
    true_labels,
    predicted_labels,
):

    n_labels = true_labels.shape[1]


    matrix = initialize_matrix(
        n_labels,
        n_labels,
    )


    for i in range(
        true_labels.shape[0]
    ):

        true_row = true_labels[i]

        predicted_row = (
            predicted_labels[i]
        )


        # Truly present labels
        true_indices = np.where(
            true_row == 1
        )[0]


        # False-positive predictions
        false_positive_indices = np.where(
            (
                true_row == 0
            )
            &
            (
                predicted_row == 1
            )
        )[0]


        for true_idx in true_indices:

            for fp_idx in false_positive_indices:

                matrix[
                    true_idx,
                    fp_idx
                ] += 1


    return matrix


# ============================================================
# 4. NOTHING PREDICTED ANALYSIS
# ============================================================
#
# For each true label:
#
# How often was that label present but the model predicted
# absolutely NOTHING?
#
# Example:
#
# TRUE:
#     IN = 1
#     OP = 1
#
# PREDICTED:
#     nothing
#
# Then both:
#
#     IN -> nothing
#     OP -> nothing
#
# are counted.
#
# ============================================================

def nothing_predicted_counts(
    true_labels,
    predicted_labels,
):

    n_labels = true_labels.shape[1]


    nothing_predicted = (
        predicted_labels.sum(
            axis=1
        ) == 0
    )


    counts = np.zeros(
        n_labels,
        dtype=np.int64,
    )


    true_support = np.zeros(
        n_labels,
        dtype=np.int64,
    )


    for label_idx in range(
        n_labels
    ):

        true_positive_examples = (
            true_labels[:, label_idx]
            == 1
        )


        true_support[label_idx] = (
            true_positive_examples.sum()
        )


        counts[label_idx] = (
            (
                true_positive_examples
                &
                nothing_predicted
            )
            .sum()
        )


    return (
        counts,
        true_support,
    )


# ============================================================
# 5. PER-LABEL BASIC OUTCOMES
# ============================================================
#
# TP / FP / FN / TN for each label.
#
# This is still important because it gives the conventional
# multilabel evaluation for every label.
#
# ============================================================

def per_label_outcomes(
    true_labels,
    predicted_labels,
):

    tp = (
        (
            true_labels == 1
        )
        &
        (
            predicted_labels == 1
        )
    ).sum(axis=0)


    fp = (
        (
            true_labels == 0
        )
        &
        (
            predicted_labels == 1
        )
    ).sum(axis=0)


    fn = (
        (
            true_labels == 1
        )
        &
        (
            predicted_labels == 0
        )
    ).sum(axis=0)


    tn = (
        (
            true_labels == 0
        )
        &
        (
            predicted_labels == 0
        )
    ).sum(axis=0)


    return (
        tp,
        fp,
        fn,
        tn,
    )


# ============================================================
# PERCENTAGE HELPERS
# ============================================================

def row_percentages(
    matrix,
    denominators,
):

    result = np.zeros_like(
        matrix,
        dtype=np.float64,
    )


    for i in range(
        matrix.shape[0]
    ):

        if denominators[i] > 0:

            result[i] = (
                matrix[i]
                /
                denominators[i]
                *
                100.0
            )


    return result


def column_percentages(
    matrix,
    denominators,
):

    result = np.zeros_like(
        matrix,
        dtype=np.float64,
    )


    for j in range(
        matrix.shape[1]
    ):

        if denominators[j] > 0:

            result[:, j] = (
                matrix[:, j]
                /
                denominators[j]
                *
                100.0
            )


    return result


# ============================================================
# SAVE CSV MATRIX
# ============================================================

def save_matrix_csv(
    matrix,
    labels,
    filename,
):

    df = pd.DataFrame(
        matrix,
        index=labels,
        columns=labels,
    )


    df.to_csv(
        ANALYSIS_DIR / filename,
        encoding="utf-8-sig",
    )


# ============================================================
# HEATMAP FUNCTION
# ============================================================

def plot_heatmap(
    values,
    counts,
    labels,
    filename,
    title,
    percentage_label,
):

    n_labels = len(labels)


    if n_labels <= 9:

        figsize = (
            12,
            10,
        )

        fontsize = 11

    elif n_labels <= 16:

        figsize = (
            16,
            14,
        )

        fontsize = 9

    else:

        figsize = (
            23,
            20,
        )

        fontsize = 7


    # --------------------------------------------------------
    # Annotation
    # --------------------------------------------------------

    annotations = np.empty(
        counts.shape,
        dtype=object,
    )


    for i in range(
        counts.shape[0]
    ):

        for j in range(
            counts.shape[1]
        ):

            annotations[i, j] = (
                f"{counts[i, j]:,}\n"
                f"{values[i, j]:.1f}%"
            )


    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=figsize
    )


    sns.heatmap(

        values,

        annot=annotations,

        fmt="",

        cmap="Blues",

        linewidths=0.5,

        linecolor="white",

        xticklabels=labels,

        yticklabels=labels,

        vmin=0,

        vmax=np.nanmax(
            values
        )
        if np.nanmax(values) > 0
        else 1,

        cbar_kws={
            "label":
            percentage_label
        },

        ax=ax,
    )


    ax.set_xlabel(
        "Predicted Label",
        fontsize=14,
        fontweight="bold",
    )


    ax.set_ylabel(
        "True Label",
        fontsize=14,
        fontweight="bold",
    )


    ax.set_title(
        title,
        fontsize=17,
        fontweight="bold",
        pad=18,
    )


    ax.set_xticklabels(
        labels,
        rotation=45,
        ha="right",
        fontsize=fontsize,
    )


    ax.set_yticklabels(
        labels,
        rotation=0,
        fontsize=fontsize,
    )


    plt.tight_layout()


    png_file = (
        ANALYSIS_DIR
        / filename
    )


    plt.savefig(
        png_file,
        dpi=400,
        bbox_inches="tight",
    )


    pdf_file = (
        ANALYSIS_DIR
        / filename.replace(
            ".png",
            ".pdf",
        )
    )


    plt.savefig(
        pdf_file,
        bbox_inches="tight",
    )


    plt.close()


    print(
        "Saved:",
        png_file,
    )


# ============================================================
# CREATE ASSOCIATION ANALYSIS
# ============================================================

def create_association_analysis(
    true_labels,
    predicted_labels,
    labels,
    prefix,
    description,
):

    print(
        "\nCreating TRUE -> PREDICTED association:"
    )

    print(
        description
    )


    # --------------------------------------------------------
    # Count matrix
    # --------------------------------------------------------

    counts = (
        true_predicted_association(
            true_labels,
            predicted_labels,
        )
    )


    # --------------------------------------------------------
    # Denominator:
    #
    # Number of examples in which each TRUE label
    # was present.
    # --------------------------------------------------------

    true_support = (
        true_labels.sum(
            axis=0
        )
    )


    percentages = (
        row_percentages(
            counts,
            true_support,
        )
    )


    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    save_matrix_csv(
        counts,
        labels,
        f"{prefix}_association_counts.csv",
    )


    save_matrix_csv(
        percentages,
        labels,
        f"{prefix}_association_percentages.csv",
    )


    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------

    plot_heatmap(

        values=percentages,

        counts=counts,

        labels=labels,

        filename=(
            f"{prefix}_association.png"
        ),

        title=(
            "True → Predicted Label Association\n"
            f"{description}"
        ),

        percentage_label=(
            "Percentage of true-label occurrences"
        ),
    )


    return (
        counts,
        percentages,
    )


# ============================================================
# CREATE SUBSTITUTION ANALYSIS
# ============================================================

def create_substitution_analysis(
    true_labels,
    predicted_labels,
    labels,
    prefix,
    description,
):

    print(
        "\nCreating GENUINE SUBSTITUTION matrix:"
    )

    print(
        description
    )


    counts = (
        substitution_error_matrix(
            true_labels,
            predicted_labels,
        )
    )


    # --------------------------------------------------------
    # Number of times each TRUE label was missed.
    #
    # This is the denominator.
    # --------------------------------------------------------

    missed_counts = (
        (
            (
                true_labels == 1
            )
            &
            (
                predicted_labels == 0
            )
        )
        .sum(axis=0)
    )


    percentages = (
        row_percentages(
            counts,
            missed_counts,
        )
    )


    # --------------------------------------------------------
    # Diagonal does not represent a substitution.
    # --------------------------------------------------------

    np.fill_diagonal(
        percentages,
        0,
    )


    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    save_matrix_csv(
        counts,
        labels,
        f"{prefix}_substitution_counts.csv",
    )


    save_matrix_csv(
        percentages,
        labels,
        f"{prefix}_substitution_percentages.csv",
    )


    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------

    plot_heatmap(

        values=percentages,

        counts=counts,

        labels=labels,

        filename=(
            f"{prefix}_substitution.png"
        ),

        title=(
            "Genuine Multilabel Substitution Errors\n"
            f"{description}"
        ),

        percentage_label=(
            "% of missed true-label cases"
        ),
    )


    return (
        counts,
        percentages,
    )


# ============================================================
# CREATE FALSE-POSITIVE ANALYSIS
# ============================================================

def create_false_positive_analysis(
    true_labels,
    predicted_labels,
    labels,
    prefix,
    description,
):

    print(
        "\nCreating FALSE-POSITIVE association:"
    )

    print(
        description
    )


    counts = (
        false_positive_association(
            true_labels,
            predicted_labels,
        )
    )


    # --------------------------------------------------------
    # Number of false-positive predictions for each
    # predicted label.
    #
    # This is the column denominator.
    # --------------------------------------------------------

    false_positive_counts = (
        (
            (
                true_labels == 0
            )
            &
            (
                predicted_labels == 1
            )
        )
        .sum(axis=0)
    )


    percentages = (
        column_percentages(
            counts,
            false_positive_counts,
        )
    )


    # --------------------------------------------------------
    # Diagonal is impossible by definition because a
    # false-positive label has true value = 0.
    # --------------------------------------------------------

    np.fill_diagonal(
        percentages,
        0,
    )


    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    save_matrix_csv(
        counts,
        labels,
        f"{prefix}_false_positive_counts.csv",
    )


    save_matrix_csv(
        percentages,
        labels,
        f"{prefix}_false_positive_percentages.csv",
    )


    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------

    plot_heatmap(

        values=percentages,

        counts=counts,

        labels=labels,

        filename=(
            f"{prefix}_false_positive.png"
        ),

        title=(
            "False-Positive Label Association\n"
            f"{description}"
        ),

        percentage_label=(
            "% of false-positive predictions"
        ),
    )


    return (
        counts,
        percentages,
    )


# ============================================================
# CREATE NOTHING-PREDICTED ANALYSIS
# ============================================================

def create_nothing_analysis(
    true_labels,
    predicted_labels,
    labels,
    prefix,
    description,
):

    print(
        "\nCreating NOTHING-PREDICTED analysis:"
    )

    print(
        description
    )


    (
        nothing_counts,
        true_support,
    ) = nothing_predicted_counts(
        true_labels,
        predicted_labels,
    )


    nothing_percentages = np.zeros(
        len(labels),
        dtype=np.float64,
    )


    for i in range(
        len(labels)
    ):

        if true_support[i] > 0:

            nothing_percentages[i] = (
                nothing_counts[i]
                /
                true_support[i]
                *
                100.0
            )


    # --------------------------------------------------------
    # Also calculate missed-label counts
    # --------------------------------------------------------

    (
        tp,
        fp,
        fn,
        tn,
    ) = per_label_outcomes(
        true_labels,
        predicted_labels,
    )


    # --------------------------------------------------------
    # Save table
    # --------------------------------------------------------

    df = pd.DataFrame({

        "label":
            labels,

        "true_support":
            true_support,

        "nothing_predicted":
            nothing_counts,

        "nothing_predicted_percent":
            nothing_percentages,

        "TP":
            tp,

        "FP":
            fp,

        "FN":
            fn,

        "TN":
            tn,
    })


    df.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_nothing_predicted.csv",

        index=False,

        encoding="utf-8-sig",
    )


    # --------------------------------------------------------
    # Bar plot
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(16, 8)
    )


    bars = ax.bar(
        labels,
        nothing_percentages,
        color="#d95f02",
    )


    ax.set_xlabel(
        "Label",
        fontsize=13,
        fontweight="bold",
    )


    ax.set_ylabel(
        "Percentage (%)",
        fontsize=13,
        fontweight="bold",
    )


    ax.set_title(
        (
            "True Labels with No Predicted Labels\n"
            f"{description}"
        ),
        fontsize=16,
        fontweight="bold",
    )


    ax.tick_params(
        axis="x",
        rotation=45,
    )


    for bar, value in zip(
        bars,
        nothing_percentages,
    ):

        ax.text(
            bar.get_x()
            + bar.get_width()
            / 2,

            bar.get_height(),

            f"{value:.1f}%",

            ha="center",

            va="bottom",

            fontsize=8,
        )


    plt.tight_layout()


    png_file = (
        ANALYSIS_DIR
        / f"{prefix}_nothing_predicted.png"
    )


    plt.savefig(
        png_file,
        dpi=400,
        bbox_inches="tight",
    )


    pdf_file = (
        ANALYSIS_DIR
        / f"{prefix}_nothing_predicted.pdf"
    )


    plt.savefig(
        pdf_file,
        bbox_inches="tight",
    )


    plt.close()


    print(
        "Saved:",
        png_file,
    )


    return df


# ============================================================
# CREATE PER-LABEL SUMMARY
# ============================================================

def create_label_summary(
    true_labels,
    predicted_labels,
    labels,
    prefix,
    description,
):

    (
        tp,
        fp,
        fn,
        tn,
    ) = per_label_outcomes(
        true_labels,
        predicted_labels,
    )


    precision = np.divide(
        tp,
        tp + fp,
        out=np.zeros(
            len(tp),
            dtype=float,
        ),
        where=(
            tp + fp
        ) != 0,
    )


    recall = np.divide(
        tp,
        tp + fn,
        out=np.zeros(
            len(tp),
            dtype=float,
        ),
        where=(
            tp + fn
        ) != 0,
    )


    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros(
            len(tp),
            dtype=float,
        ),
        where=(
            precision + recall
        ) != 0,
    )


    specificity = np.divide(
        tn,
        tn + fp,
        out=np.zeros(
            len(tp),
            dtype=float,
        ),
        where=(
            tn + fp
        ) != 0,
    )


    false_positive_rate = (
        1 - specificity
    )


    df = pd.DataFrame({

        "label":
            labels,

        "TP":
            tp,

        "FP":
            fp,

        "FN":
            fn,

        "TN":
            tn,

        "precision":
            precision,

        "recall":
            recall,

        "F1":
            f1,

        "specificity":
            specificity,

        "false_positive_rate":
            false_positive_rate,

        "support":
            tp + fn,
    })


    df.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_per_label_summary.csv",

        index=False,

        encoding="utf-8-sig",
    )


    return df


# ============================================================
# RUN ONE COMPLETE ANALYSIS
# ============================================================

def run_analysis(
    predictions,
    prefix,
    description,
):

    print("\n")
    print("=" * 100)
    print(description)
    print("=" * 100)


    # ========================================================
    # ALL 25 LABELS
    # ========================================================

    print("\n")
    print("-" * 100)
    print("ALL 25 LABELS")
    print("-" * 100)


    create_association_analysis(

        true_labels=test_labels,

        predicted_labels=predictions,

        labels=hierarchical_labels,

        prefix=(
            f"{prefix}_all25"
        ),

        description=description,
    )


    create_substitution_analysis(

        true_labels=test_labels,

        predicted_labels=predictions,

        labels=hierarchical_labels,

        prefix=(
            f"{prefix}_all25"
        ),

        description=description,
    )


    create_false_positive_analysis(

        true_labels=test_labels,

        predicted_labels=predictions,

        labels=hierarchical_labels,

        prefix=(
            f"{prefix}_all25"
        ),

        description=description,
    )


    create_nothing_analysis(

        true_labels=test_labels,

        predicted_labels=predictions,

        labels=hierarchical_labels,

        prefix=(
            f"{prefix}_all25"
        ),

        description=description,
    )


    create_label_summary(

        true_labels=test_labels,

        predicted_labels=predictions,

        labels=hierarchical_labels,

        prefix=(
            f"{prefix}_all25"
        ),

        description=description,
    )


    # ========================================================
    # PARENTS
    # ========================================================

    print("\n")
    print("-" * 100)
    print("9 PARENT LABELS")
    print("-" * 100)


    parent_true = (
        test_labels[
            :,
            parent_indices
        ]
    )


    parent_predictions = (
        predictions[
            :,
            parent_indices
        ]
    )


    create_association_analysis(

        true_labels=parent_true,

        predicted_labels=parent_predictions,

        labels=parents,

        prefix=(
            f"{prefix}_parents"
        ),

        description=(
            description
            + " | Parents"
        ),
    )


    create_substitution_analysis(

        true_labels=parent_true,

        predicted_labels=parent_predictions,

        labels=parents,

        prefix=(
            f"{prefix}_parents"
        ),

        description=(
            description
            + " | Parents"
        ),
    )


    create_false_positive_analysis(

        true_labels=parent_true,

        predicted_labels=parent_predictions,

        labels=parents,

        prefix=(
            f"{prefix}_parents"
        ),

        description=(
            description
            + " | Parents"
        ),
    )


    create_nothing_analysis(

        true_labels=parent_true,

        predicted_labels=parent_predictions,

        labels=parents,

        prefix=(
            f"{prefix}_parents"
        ),

        description=(
            description
            + " | Parents"
        ),
    )


    create_label_summary(

        true_labels=parent_true,

        predicted_labels=parent_predictions,

        labels=parents,

        prefix=(
            f"{prefix}_parents"
        ),

        description=(
            description
            + " | Parents"
        ),
    )


    # ========================================================
    # CHILDREN
    # ========================================================

    print("\n")
    print("-" * 100)
    print("16 CHILD LABELS")
    print("-" * 100)


    child_true = (
        test_labels[
            :,
            child_indices
        ]
    )


    child_predictions = (
        predictions[
            :,
            child_indices
        ]
    )


    create_association_analysis(

        true_labels=child_true,

        predicted_labels=child_predictions,

        labels=children,

        prefix=(
            f"{prefix}_children"
        ),

        description=(
            description
            + " | Children"
        ),
    )


    create_substitution_analysis(

        true_labels=child_true,

        predicted_labels=child_predictions,

        labels=children,

        prefix=(
            f"{prefix}_children"
        ),

        description=(
            description
            + " | Children"
        ),
    )


    create_false_positive_analysis(

        true_labels=child_true,

        predicted_labels=child_predictions,

        labels=children,

        prefix=(
            f"{prefix}_children"
        ),

        description=(
            description
            + " | Children"
        ),
    )


    create_nothing_analysis(

        true_labels=child_true,

        predicted_labels=child_predictions,

        labels=children,

        prefix=(
            f"{prefix}_children"
        ),

        description=(
            description
            + " | Children"
        ),
    )


    create_label_summary(

        true_labels=child_true,

        predicted_labels=child_predictions,

        labels=children,

        prefix=(
            f"{prefix}_children"
        ),

        description=(
            description
            + " | Children"
        ),
    )


# ============================================================
# RUN ANALYSIS 1:
# ORIGINAL 0.50
# ============================================================

run_analysis(

    predictions=predictions_05,

    prefix="threshold_0.50",

    description=(
        "Threshold = 0.50"
    ),
)


# ============================================================
# RUN ANALYSIS 2:
# GLOBAL TUNED
# ============================================================

run_analysis(

    predictions=predictions_global,

    prefix="global_tuned",

    description=(
        "Global Tuned Threshold = "
        f"{GLOBAL_TUNED_THRESHOLD:.3f}"
    ),
)


# ============================================================
# RUN ANALYSIS 3:
# PER-LABEL TUNED
# ============================================================

run_analysis(

    predictions=predictions_per_label,

    prefix="per_label_tuned",

    description=(
        "Per-Label Tuned Thresholds"
    ),
)


# ============================================================
# SAVE THRESHOLDS
# ============================================================

threshold_df = pd.DataFrame({

    "label":
        hierarchical_labels,

    "per_label_threshold":
        per_label_threshold_array,

})


threshold_df.to_csv(

    ANALYSIS_DIR
    / "thresholds_used.csv",

    index=False,

    encoding="utf-8-sig",
)


with open(

    ANALYSIS_DIR
    / "threshold_configuration.json",

    "w",

    encoding="utf-8",

) as f:

    json.dump(

        {
            "original_threshold": 0.50,

            "global_tuned_threshold":
                GLOBAL_TUNED_THRESHOLD,

            "per_label_thresholds":
                PER_LABEL_THRESHOLDS,
        },

        f,

        indent=2,
    )


# ============================================================
# FINAL MESSAGE
# ============================================================

print("\n" + "=" * 100)
print("MULTILABEL ERROR ANALYSIS COMPLETE")
print("=" * 100)

print(
    "\nResults saved to:"
)

print(
    ANALYSIS_DIR
)

print("\n")
print(
    "You now have:"
)

print(
    "  1. True -> Predicted association matrices"
)

print(
    "  2. Genuine substitution-error matrices"
)

print(
    "  3. False-positive association matrices"
)

print(
    "  4. Nothing-predicted analysis"
)

print(
    "  5. Per-label TP/FP/FN/TN summaries"
)

print(
    "  6. Separate parent and child analyses"
)

print(
    "  7. Original, global-tuned, and per-label-tuned versions"
)

print("=" * 100)



LOADING TEST DATA
Logits shape: (633, 25)
Labels shape: (633, 25)
Number of labels: 25

Parents:
['MT', 'LY', 'SP', 'ID', 'NA', 'HI', 'IN', 'OP', 'IP']

Children:
['it', 'ne', 'sr', 'nb', 're', 'en', 'ra', 'dtp', 'fi', 'lt', 'rv', 'ob', 'rs', 'av', 'ds', 'ed']


Threshold = 0.50


----------------------------------------------------------------------------------------------------
ALL 25 LABELS
----------------------------------------------------------------------------------------------------

Creating TRUE -> PREDICTED association:
Threshold = 0.50
Saved: C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth\multilabel_error_analysis\threshold_0.50_all25_association.png

Creating GENUINE SUBSTITUTION matrix:
Threshold = 0.50
Saved: C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth\multilabel_error_analysis\threshold_0.50_all25_substitution.png

Creating FALSE-POSITIVE association:
Threshold = 0.50
Saved: C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffW

## New approach codes

In [2]:
# ============================================================
# HIERARCHICAL MULTILABEL ERROR ANALYSIS
#
# Designed for:
#
#   9 parent labels
#   16 child labels
#   25 total labels
#
# The analysis explicitly distinguishes:
#
#   1. Correct labels
#   2. False positives
#   3. False negatives
#   4. Missed-label -> false-positive associations
#   5. Completely missed documents
#   6. Partially correct documents
#   7. Exact label-set matches
#   8. Hierarchical violations
#   9. Child predicted but parent missing
#  10. Parent predicted without corresponding child
#
# ============================================================


# ============================================================
# IMPORTS
# ============================================================

import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    multilabel_confusion_matrix,
)


# ============================================================
# CONFIGURATION
# ============================================================

OUTPUT_DIR = Path(
    r"C:\Users\alrazz\Downloads\shared_multiCore_XLM-R_diffWeightsBoth"
)


LOGITS_FILE = (
    OUTPUT_DIR / "test_logits.npy"
)

LABELS_FILE = (
    OUTPUT_DIR / "test_labels.npy"
)

LABEL_NAMES_FILE = (
    OUTPUT_DIR / "hierarchical_labels.json"
)


ANALYSIS_DIR = (
    OUTPUT_DIR / "hierarchical_error_analysis"
)

ANALYSIS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# THRESHOLD CONFIGURATION
# ============================================================

GLOBAL_TUNED_THRESHOLD = 0.35


PER_LABEL_THRESHOLDS = {

    "MT":  0.10,
    "LY":  0.50,
    "SP":  0.53,
    "ID":  0.66,
    "NA":  0.44,
    "HI":  0.24,
    "IN":  0.46,
    "OP":  0.35,
    "IP":  0.28,

    "it":  0.43,
    "ne":  0.28,
    "sr":  0.39,
    "nb":  0.26,
    "re":  0.49,

    "en":  0.44,
    "ra":  0.16,
    "dtp": 0.25,
    "fi":  0.33,
    "lt":  0.38,

    "rv":  0.21,
    "ob":  0.26,
    "rs":  0.47,
    "av":  0.45,

    "ds":  0.53,
    "ed":  0.32,
}


# ============================================================
# LOAD DATA
# ============================================================

print("\n" + "=" * 100)
print("LOADING TEST DATA")
print("=" * 100)


test_logits = np.load(
    LOGITS_FILE
)

test_labels = np.load(
    LABELS_FILE
)


with open(
    LABEL_NAMES_FILE,
    "r",
    encoding="utf-8",
) as f:

    hierarchical_labels = json.load(f)


print(
    "Logits shape:",
    test_logits.shape,
)

print(
    "Labels shape:",
    test_labels.shape,
)

print(
    "Labels:",
    hierarchical_labels,
)


# ============================================================
# BASIC CHECKS
# ============================================================

if test_logits.shape != test_labels.shape:

    raise ValueError(
        "Logits and labels have different shapes."
    )


NUM_LABELS = len(
    hierarchical_labels
)


if test_logits.shape[1] != NUM_LABELS:

    raise ValueError(
        "Number of logits does not match number of labels."
    )


# ============================================================
# HIERARCHICAL STRUCTURE
# ============================================================

NUM_PARENTS = 9


parents = hierarchical_labels[
    :NUM_PARENTS
]

children = hierarchical_labels[
    NUM_PARENTS:
]


parent_indices = list(
    range(NUM_PARENTS)
)

child_indices = list(
    range(NUM_PARENTS, NUM_LABELS)
)


# ------------------------------------------------------------
# Explicit child -> parent mapping
# ------------------------------------------------------------

child_to_parent = {

    "it": "SP",

    "ne": "NA",
    "sr": "NA",
    "nb": "NA",

    "re": "HI",

    "en": "IN",
    "ra": "IN",
    "dtp": "IN",
    "fi": "IN",
    "lt": "IN",

    "rv": "OP",
    "ob": "OP",
    "rs": "OP",
    "av": "OP",

    "ds": "IP",
    "ed": "IP",
}


print("\nParents:")
print(parents)

print("\nChildren:")
print(children)


# ============================================================
# VERIFY CHILD -> PARENT MAPPING
# ============================================================

for child in children:

    if child not in child_to_parent:

        raise ValueError(
            f"No parent defined for child: {child}"
        )

    parent = child_to_parent[child]

    if parent not in parents:

        raise ValueError(
            f"Unknown parent {parent} for child {child}"
        )


# ============================================================
# SIGMOID
# ============================================================

def sigmoid(x):

    # Numerically safer implementation
    x = np.clip(
        x,
        -50,
        50,
    )

    return 1.0 / (
        1.0 + np.exp(-x)
    )


test_probabilities = sigmoid(
    test_logits
)


# ============================================================
# THRESHOLDS
# ============================================================

missing_thresholds = [

    label

    for label in hierarchical_labels

    if label not in PER_LABEL_THRESHOLDS
]


if missing_thresholds:

    raise ValueError(
        "Missing thresholds for: "
        f"{missing_thresholds}"
    )


threshold_array = np.array(

    [
        PER_LABEL_THRESHOLDS[label]
        for label in hierarchical_labels
    ],

    dtype=np.float32,
)


# ============================================================
# PREDICTIONS
# ============================================================

def make_predictions(
    probabilities,
    thresholds,
):

    thresholds = np.asarray(
        thresholds,
        dtype=np.float32,
    )


    if thresholds.ndim == 0:

        thresholds = np.repeat(
            thresholds,
            probabilities.shape[1],
        )


    return (
        probabilities
        >= thresholds.reshape(1, -1)
    ).astype(np.int32)


predictions_05 = make_predictions(
    test_probabilities,
    0.50,
)


predictions_global = make_predictions(
    test_probabilities,
    GLOBAL_TUNED_THRESHOLD,
)


predictions_per_label = make_predictions(
    test_probabilities,
    threshold_array,
)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def label_set(
    row,
    labels=hierarchical_labels,
):

    return [
        labels[i]
        for i in np.where(
            row == 1
        )[0]
    ]


def label_string(
    row,
    labels=hierarchical_labels,
):

    values = label_set(
        row,
        labels,
    )

    if not values:

        return "NONE"

    return ", ".join(values)


def set_difference_string(
    true_row,
    pred_row,
):

    true_set = set(
        label_set(true_row)
    )

    pred_set = set(
        label_set(pred_row)
    )


    missed = sorted(
        true_set - pred_set
    )

    false_positive = sorted(
        pred_set - true_set
    )


    return (
        ", ".join(missed)
        if missed
        else "NONE"
    ), (
        ", ".join(false_positive)
        if false_positive
        else "NONE"
    )


# ============================================================
# HIERARCHICAL VIOLATIONS
# ============================================================
#
# A hierarchy violation occurs when:
#
# child = 1
# parent = 0
#
# Example:
#
# fi = 1
# IN = 0
#
# This is an invalid hierarchical prediction.
# ============================================================

def get_hierarchy_violations(
    prediction_row,
):

    violations = []


    for child, parent in child_to_parent.items():

        child_idx = hierarchical_labels.index(
            child
        )

        parent_idx = hierarchical_labels.index(
            parent
        )


        if (
            prediction_row[child_idx] == 1
            and
            prediction_row[parent_idx] == 0
        ):

            violations.append(
                f"{child}->{parent}"
            )


    return violations


# ============================================================
# TRUE HIERARCHY CHECK
# ============================================================
#
# This checks whether the GOLD labels themselves are
# hierarchically consistent.
#
# Normally all violations should be zero because your
# make_hierarchical_labels() function explicitly propagates
# child labels to parents.
# ============================================================

def get_true_hierarchy_violations(
    true_row,
):

    violations = []


    for child, parent in child_to_parent.items():

        child_idx = hierarchical_labels.index(
            child
        )

        parent_idx = hierarchical_labels.index(
            parent
        )


        if (
            true_row[child_idx] == 1
            and
            true_row[parent_idx] == 0
        ):

            violations.append(
                f"{child}->{parent}"
            )


    return violations


# ============================================================
# PER-DOCUMENT ERROR ANALYSIS
# ============================================================

def create_document_error_table(
    true_labels,
    predicted_labels,
    probabilities,
):

    rows = []


    for i in range(
        len(true_labels)
    ):

        true_row = true_labels[i]

        pred_row = predicted_labels[i]

        prob_row = probabilities[i]


        true_set = set(
            label_set(true_row)
        )

        pred_set = set(
            label_set(pred_row)
        )


        correct = sorted(
            true_set & pred_set
        )

        missed = sorted(
            true_set - pred_set
        )

        false_positive = sorted(
            pred_set - true_set
        )


        true_count = len(
            true_set
        )

        predicted_count = len(
            pred_set
        )

        correct_count = len(
            correct
        )

        missed_count = len(
            missed
        )

        false_positive_count = len(
            false_positive
        )


        # ----------------------------------------------------
        # Exact set match
        # ----------------------------------------------------

        exact_match = (
            true_set == pred_set
        )


        # ----------------------------------------------------
        # Completely missed
        # ----------------------------------------------------

        nothing_predicted = (
            predicted_count == 0
        )


        # ----------------------------------------------------
        # No true labels
        # ----------------------------------------------------

        no_true_labels = (
            true_count == 0
        )


        # ----------------------------------------------------
        # Completely wrong
        #
        # There is no overlap between true and predicted
        # labels, while both contain at least one label.
        # ----------------------------------------------------

        completely_wrong = (

            true_count > 0

            and

            predicted_count > 0

            and

            correct_count == 0
        )


        # ----------------------------------------------------
        # Partially correct
        # ----------------------------------------------------

        partially_correct = (

            correct_count > 0

            and

            not exact_match
        )


        # ----------------------------------------------------
        # Hierarchical violations
        # ----------------------------------------------------

        hierarchy_violations = (
            get_hierarchy_violations(
                pred_row
            )
        )


        true_hierarchy_violations = (
            get_true_hierarchy_violations(
                true_row
            )
        )


        # ----------------------------------------------------
        # Probability of true labels
        # ----------------------------------------------------

        true_probabilities = [

            float(
                prob_row[
                    hierarchical_labels.index(label)
                ]
            )

            for label in sorted(true_set)

        ]


        # ----------------------------------------------------
        # Probability of missed labels
        # ----------------------------------------------------

        missed_probabilities = [

            float(
                prob_row[
                    hierarchical_labels.index(label)
                ]
            )

            for label in missed

        ]


        # ----------------------------------------------------
        # Probability of false positives
        # ----------------------------------------------------

        false_positive_probabilities = [

            float(
                prob_row[
                    hierarchical_labels.index(label)
                ]
            )

            for label in false_positive

        ]


        rows.append({

            "document_index": i,

            "true_labels":
                ", ".join(
                    sorted(true_set)
                )
                if true_set
                else "NONE",

            "predicted_labels":
                ", ".join(
                    sorted(pred_set)
                )
                if pred_set
                else "NONE",

            "correct_labels":
                ", ".join(correct)
                if correct
                else "NONE",

            "missed_labels":
                ", ".join(missed)
                if missed
                else "NONE",

            "false_positive_labels":
                ", ".join(false_positive)
                if false_positive
                else "NONE",

            "true_label_count":
                true_count,

            "predicted_label_count":
                predicted_count,

            "correct_label_count":
                correct_count,

            "missed_label_count":
                missed_count,

            "false_positive_count":
                false_positive_count,

            "exact_match":
                exact_match,

            "partially_correct":
                partially_correct,

            "completely_wrong":
                completely_wrong,

            "nothing_predicted":
                nothing_predicted,

            "hierarchy_violation":
                len(
                    hierarchy_violations
                ) > 0,

            "hierarchy_violations":
                "; ".join(
                    hierarchy_violations
                )
                if hierarchy_violations
                else "NONE",

            "true_hierarchy_violation":
                len(
                    true_hierarchy_violations
                ) > 0,

            "true_hierarchy_violations":
                "; ".join(
                    true_hierarchy_violations
                )
                if true_hierarchy_violations
                else "NONE",

            "true_probabilities":
                "; ".join(
                    f"{p:.4f}"
                    for p in true_probabilities
                ),

            "missed_probabilities":
                "; ".join(
                    f"{p:.4f}"
                    for p in missed_probabilities
                ),

            "false_positive_probabilities":
                "; ".join(
                    f"{p:.4f}"
                    for p in false_positive_probabilities
                ),
        })


    return pd.DataFrame(rows)


# ============================================================
# PER-LABEL TP / FP / FN / TN
# ============================================================

def create_per_label_summary(
    true_labels,
    predicted_labels,
):

    mcm = multilabel_confusion_matrix(
        true_labels,
        predicted_labels,
    )


    rows = []


    for i, label in enumerate(
        hierarchical_labels
    ):

        tn = mcm[i, 0, 0]

        fp = mcm[i, 0, 1]

        fn = mcm[i, 1, 0]

        tp = mcm[i, 1, 1]


        precision = (
            tp / (tp + fp)
            if (tp + fp) > 0
            else 0.0
        )


        recall = (
            tp / (tp + fn)
            if (tp + fn) > 0
            else 0.0
        )


        f1 = (

            2 * precision * recall
            /
            (precision + recall)

            if (precision + recall) > 0

            else 0.0
        )


        rows.append({

            "label": label,

            "TP": int(tp),

            "FP": int(fp),

            "FN": int(fn),

            "TN": int(tn),

            "precision":
                precision,

            "recall":
                recall,

            "F1":
                f1,

            "support":
                int(tp + fn),

        })


    return pd.DataFrame(rows)


# ============================================================
# MISSED -> FALSE POSITIVE ASSOCIATION
# ============================================================
#
# Example:
#
# TRUE:
#     IN OP
#
# PRED:
#     IN NA
#
# Result:
#
#     OP -> NA
#
# Important:
#
# This means:
#
# "When OP was missed, NA was falsely predicted in the
# same document."
#
# It does NOT mathematically prove that NA replaced OP.
# ============================================================

def create_missed_to_fp_matrix(
    true_labels,
    predicted_labels,
):

    n_labels = len(
        hierarchical_labels
    )


    matrix = np.zeros(
        (n_labels, n_labels),
        dtype=np.int64,
    )


    for i in range(
        len(true_labels)
    ):

        true_row = true_labels[i]

        pred_row = predicted_labels[i]


        missed = np.where(

            (true_row == 1)

            &

            (pred_row == 0)

        )[0]


        false_positive = np.where(

            (true_row == 0)

            &

            (pred_row == 1)

        )[0]


        for missed_idx in missed:

            for fp_idx in false_positive:

                matrix[
                    missed_idx,
                    fp_idx
                ] += 1


    return matrix


# ============================================================
# SAVE MATRIX
# ============================================================

def save_matrix(
    matrix,
    filename,
):

    df = pd.DataFrame(

        matrix,

        index=hierarchical_labels,

        columns=hierarchical_labels,
    )


    df.to_csv(

        ANALYSIS_DIR / filename,

        encoding="utf-8-sig",
    )


    return df


# ============================================================
# PLOT MATRIX
# ============================================================

def plot_matrix(
    matrix,
    filename,
    title,
    xlabel,
    ylabel,
):

    plt.figure(
        figsize=(20, 17)
    )


    sns.heatmap(

        matrix,

        annot=True,

        fmt="d",

        cmap="Blues",

        xticklabels=hierarchical_labels,

        yticklabels=hierarchical_labels,

        linewidths=0.4,

    )


    plt.xlabel(
        xlabel,
        fontsize=13,
        fontweight="bold",
    )


    plt.ylabel(
        ylabel,
        fontsize=13,
        fontweight="bold",
    )


    plt.title(
        title,
        fontsize=16,
        fontweight="bold",
    )


    plt.xticks(
        rotation=45,
        ha="right",
    )


    plt.yticks(
        rotation=0,
    )


    plt.tight_layout()


    plt.savefig(

        ANALYSIS_DIR / filename,

        dpi=400,

        bbox_inches="tight",
    )


    plt.close()


# ============================================================
# EXACT TRUE -> PREDICTED SET PATTERNS
# ============================================================
#
# This is one of the most important analyses.
#
# Example:
#
# TRUE = IN, OP
# PRED = IN, NA
#
# gets one row:
#
# IN, OP -> IN, NA : 37 documents
#
# ============================================================

def create_exact_pattern_table(
    true_labels,
    predicted_labels,
):

    counter = Counter()


    for true_row, pred_row in zip(
        true_labels,
        predicted_labels,
    ):

        true_string = label_string(
            true_row
        )

        pred_string = label_string(
            pred_row
        )


        counter[
            (
                true_string,
                pred_string,
            )
        ] += 1


    rows = []


    for (
        (
            true_string,
            pred_string,
        ),
        count,
    ) in counter.most_common():

        rows.append({

            "true_labels":
                true_string,

            "predicted_labels":
                pred_string,

            "count":
                count,

        })


    return pd.DataFrame(rows)


# ============================================================
# HIERARCHY ERROR ANALYSIS
# ============================================================

def create_hierarchy_error_table(
    true_labels,
    predicted_labels,
):

    rows = []

    for i in range(
        len(true_labels)
    ):

        true_row = true_labels[i]

        pred_row = predicted_labels[i]

        for child, parent in child_to_parent.items():

            child_idx = hierarchical_labels.index(
                child
            )

            parent_idx = hierarchical_labels.index(
                parent
            )

            child_true = (
                true_row[child_idx] == 1
            )

            parent_true = (
                true_row[parent_idx] == 1
            )

            child_pred = (
                pred_row[child_idx] == 1
            )

            parent_pred = (
                pred_row[parent_idx] == 1
            )


            # ------------------------------------------------
            # ERROR TYPE 1:
            #
            # Child is truly present and correctly predicted,
            # but its required parent is missed.
            #
            # TRUE:
            #     child  = 1
            #     parent = 1
            #
            # PRED:
            #     child  = 1
            #     parent = 0
            # ------------------------------------------------

            if (
                child_true
                and
                parent_true
                and
                child_pred
                and
                not parent_pred
            ):

                rows.append({

                    "document_index":
                        i,

                    "child":
                        child,

                    "parent":
                        parent,

                    "error_type":
                        "child_correct_parent_missed",

                    "true_child":
                        1,

                    "true_parent":
                        1,

                    "pred_child":
                        1,

                    "pred_parent":
                        0,

                })


            # ------------------------------------------------
            # ERROR TYPE 2:
            #
            # Model predicts a child without predicting its
            # required parent.
            #
            # This is a prediction-time hierarchy violation.
            #
            # It can happen whether or not the child is truly
            # present in the gold annotation.
            #
            # PRED:
            #     child  = 1
            #     parent = 0
            # ------------------------------------------------

            if (
                child_pred
                and
                not parent_pred
            ):

                rows.append({

                    "document_index":
                        i,

                    "child":
                        child,

                    "parent":
                        parent,

                    "error_type":
                        "hierarchy_violation",

                    "true_child":
                        int(child_true),

                    "true_parent":
                        int(parent_true),

                    "pred_child":
                        1,

                    "pred_parent":
                        0,

                })


            # ------------------------------------------------
            # ERROR TYPE 3:
            #
            # A TRUE child was missed while its parent was
            # correctly predicted.
            #
            # TRUE:
            #     child  = 1
            #     parent = 1
            #
            # PRED:
            #     child  = 0
            #     parent = 1
            # ------------------------------------------------

            if (
                child_true
                and
                parent_true
                and
                parent_pred
                and
                not child_pred
            ):

                rows.append({

                    "document_index":
                        i,

                    "child":
                        child,

                    "parent":
                        parent,

                    "error_type":
                        "parent_predicted_child_not_predicted",

                    "true_child":
                        1,

                    "true_parent":
                        1,

                    "pred_child":
                        0,

                    "pred_parent":
                        1,

                })


    return pd.DataFrame(
        rows
    )


# ============================================================
# HIERARCHICAL ERROR COUNTS
# ============================================================

def hierarchy_summary(
    true_labels,
    predicted_labels,
):

    total = len(
        true_labels
    )


    prediction_violations = 0

    child_correct_parent_missed = 0

    true_hierarchy_violations = 0


    violation_pairs = Counter()

    child_parent_missed_pairs = Counter()


    for i in range(total):

        true_row = true_labels[i]

        pred_row = predicted_labels[i]


        true_violations = (
            get_true_hierarchy_violations(
                true_row
            )
        )


        pred_violations = (
            get_hierarchy_violations(
                pred_row
            )
        )


        if true_violations:

            true_hierarchy_violations += 1


        if pred_violations:

            prediction_violations += 1


            for pair in pred_violations:

                violation_pairs[pair] += 1


        for child, parent in child_to_parent.items():

            child_idx = hierarchical_labels.index(
                child
            )

            parent_idx = hierarchical_labels.index(
                parent
            )


            if (

                true_row[child_idx] == 1

                and

                true_row[parent_idx] == 1

                and

                pred_row[child_idx] == 1

                and

                pred_row[parent_idx] == 0

            ):

                child_correct_parent_missed += 1

                child_parent_missed_pairs[
                    f"{child}->{parent}"
                ] += 1


    return {

        "total_documents":
            total,

        "documents_with_predicted_hierarchy_violation":
            prediction_violations,

        "percent_documents_with_predicted_hierarchy_violation":
            (
                prediction_violations / total * 100
                if total > 0
                else 0
            ),

        "documents_with_true_hierarchy_violation":
            true_hierarchy_violations,

        "child_correct_parent_missed_events":
            child_correct_parent_missed,

        "predicted_hierarchy_violation_pairs":
            dict(violation_pairs),

        "child_correct_parent_missed_pairs":
            dict(child_parent_missed_pairs),
    }


# ============================================================
# DOCUMENT-LEVEL SUMMARY
# ============================================================

def create_document_summary(
    true_labels,
    predicted_labels,
):

    total = len(
        true_labels
    )


    exact = 0

    partial = 0

    completely_wrong = 0

    nothing = 0

    any_error = 0

    hierarchy_violation = 0


    for i in range(total):

        true_set = set(
            label_set(
                true_labels[i]
            )
        )

        pred_set = set(
            label_set(
                predicted_labels[i]
            )
        )


        if true_set == pred_set:

            exact += 1

        else:

            any_error += 1


        if (
            true_set
            and
            pred_set
            and
            true_set & pred_set
            and
            true_set != pred_set
        ):

            partial += 1


        if (
            true_set
            and
            pred_set
            and
            not (true_set & pred_set)
        ):

            completely_wrong += 1


        if (
            true_set
            and
            not pred_set
        ):

            nothing += 1

        hierarchy_violations = get_hierarchy_violations(
            predicted_labels[i]
        )

        if hierarchy_violations:
            hierarchy_violation += 1



    return pd.DataFrame({

        "category": [

            "exact_match",

            "any_error",

            "partially_correct",

            "completely_wrong",

            "nothing_predicted",

             "hierarchy_violation",

        ],

        "count": [

            exact,

            any_error,

            partial,

            completely_wrong,

            nothing,

             hierarchy_violation,

        ],

        "percentage": [

            exact / total * 100,

            any_error / total * 100,

            partial / total * 100,

            completely_wrong / total * 100,

            nothing / total * 100,

            hierarchy_violation / total * 100,

        ],

    })



# ============================================================
# RUN ANALYSIS
# ============================================================

def run_analysis(
    true_labels,
    predicted_labels,
    probabilities,
    prefix,
    description,
):

    print("\n")
    print("=" * 100)
    print(description)
    print("=" * 100)


    # ========================================================
    # 1. DOCUMENT-LEVEL ERROR TABLE
    # ========================================================

    document_df = create_document_error_table(

        true_labels,

        predicted_labels,

        probabilities,

    )


    document_df.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_document_errors.csv",

        index=False,

        encoding="utf-8-sig",
    )


    # --------------------------------------------------------
    # Save only documents containing errors
    # --------------------------------------------------------

    error_documents = document_df[
        ~document_df["exact_match"]
    ]


    error_documents.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_error_documents_only.csv",

        index=False,

        encoding="utf-8-sig",
    )


    # ========================================================
    # 2. EXACT LABEL-SET PATTERNS
    # ========================================================

    pattern_df = create_exact_pattern_table(

        true_labels,

        predicted_labels,

    )


    pattern_df.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_exact_label_set_patterns.csv",

        index=False,

        encoding="utf-8-sig",
    )


    # ========================================================
    # 3. PER-LABEL SUMMARY
    # ========================================================

    label_df = create_per_label_summary(

        true_labels,

        predicted_labels,

    )


    label_df.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_per_label_summary.csv",

        index=False,

        encoding="utf-8-sig",
    )


    # ========================================================
    # 4. MISSED -> FALSE POSITIVE MATRIX
    # ========================================================

    missed_fp_matrix = create_missed_to_fp_matrix(

        true_labels,

        predicted_labels,

    )


    save_matrix(

        missed_fp_matrix,

        f"{prefix}_missed_to_false_positive.csv",

    )


    plot_matrix(

        missed_fp_matrix,

        f"{prefix}_missed_to_false_positive.png",

        (
            "Missed True Label → False-Positive Label\n"
            + description
        ),

        "False-Positive Label",

        "Missed True Label",

    )


    # ========================================================
    # 5. HIERARCHY ERRORS
    # ========================================================

    hierarchy_df = create_hierarchy_error_table(

        true_labels,

        predicted_labels,

    )


    hierarchy_df.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_hierarchy_errors.csv",

        index=False,

        encoding="utf-8-sig",
    )


    # ========================================================
    # 6. HIERARCHY SUMMARY
    # ========================================================

    hierarchy_stats = hierarchy_summary(

        true_labels,

        predicted_labels,

    )


    with open(

        ANALYSIS_DIR
        / f"{prefix}_hierarchy_summary.json",

        "w",

        encoding="utf-8",

    ) as f:

        json.dump(

            hierarchy_stats,

            f,

            indent=2,

        )


    # ========================================================
    # 7. DOCUMENT SUMMARY
    # ========================================================

    doc_summary = create_document_summary(

        true_labels,

        predicted_labels,

    )


    doc_summary.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_document_summary.csv",

        index=False,

        encoding="utf-8-sig",
    )


    # ========================================================
    # 8. PRINT SUMMARY
    # ========================================================

    print("\nDocument-level summary:")

    print(
        doc_summary.to_string(
            index=False
        )
    )


    print("\nHierarchy summary:")

    print(
        json.dumps(
            hierarchy_stats,
            indent=2,
        )
    )


    # ========================================================
    # 9. TOP ERROR PATTERNS
    # ========================================================

    print("\nTop 20 true -> predicted label-set patterns:")

    print(

        pattern_df.head(20).to_string(
            index=False
        )

    )


    print("\nTop 20 missed -> false-positive pairs:")

    pair_rows = []


    for i, true_label in enumerate(
        hierarchical_labels
    ):

        for j, predicted_label in enumerate(
            hierarchical_labels
        ):

            count = missed_fp_matrix[i, j]


            if count > 0:

                pair_rows.append({

                    "missed_label":
                        true_label,

                    "false_positive_label":
                        predicted_label,

                    "count":
                        int(count),

                })


    pair_df = pd.DataFrame(
        pair_rows
    )


    if len(pair_df) > 0:

        pair_df = pair_df.sort_values(
            "count",
            ascending=False,
        )


        print(

            pair_df.head(20).to_string(
                index=False
            )

        )


    # ========================================================
    # 10. TOP HIERARCHY VIOLATIONS
    # ========================================================

    print("\nHierarchy violation examples:")

    if len(hierarchy_df) > 0:

        print(

            hierarchy_df.head(20).to_string(
                index=False
            )

        )

    else:

        print(
            "No hierarchy violations found."
        )


# ============================================================
# RUN 1:
# ORIGINAL 0.50
# ============================================================

run_analysis(

    true_labels=test_labels,

    predicted_labels=predictions_05,

    probabilities=test_probabilities,

    prefix="threshold_0.50",

    description=(
        "Threshold = 0.50"
    ),

)


# ============================================================
# RUN 2:
# GLOBAL TUNED
# ============================================================

run_analysis(

    true_labels=test_labels,

    predicted_labels=predictions_global,

    probabilities=test_probabilities,

    prefix="global_tuned",

    description=(
        f"Global threshold = "
        f"{GLOBAL_TUNED_THRESHOLD:.3f}"
    ),

)


# ============================================================
# RUN 3:
# PER-LABEL TUNED
# ============================================================

run_analysis(

    true_labels=test_labels,

    predicted_labels=predictions_per_label,

    probabilities=test_probabilities,

    prefix="per_label_tuned",

    description=(
        "Per-label tuned thresholds"
    ),

)



# ============================================================
# SAVE THRESHOLDS
# ============================================================

threshold_df = pd.DataFrame({

    "label":
        hierarchical_labels,

    "threshold":
        threshold_array,

})


threshold_df.to_csv(

    ANALYSIS_DIR
    / "thresholds_used.csv",

    index=False,

    encoding="utf-8-sig",
)


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 100)
print("HIERARCHICAL ERROR ANALYSIS COMPLETE")
print("=" * 100)

print("\nResults saved to:")

print(
    ANALYSIS_DIR
)

print("\nThe most important files are:")

print(
    "  *_document_errors.csv"
)

print(
    "  *_error_documents_only.csv"
)

print(
    "  *_exact_label_set_patterns.csv"
)

print(
    "  *_per_label_summary.csv"
)

print(
    "  *_missed_to_false_positive.csv"
)

print(
    "  *_hierarchy_errors.csv"
)

print(
    "  *_hierarchy_summary.json"
)

print(
    "  *_document_summary.csv"
)

print("=" * 100)



LOADING TEST DATA
Logits shape: (633, 25)
Labels shape: (633, 25)
Labels: ['MT', 'LY', 'SP', 'ID', 'NA', 'HI', 'IN', 'OP', 'IP', 'it', 'ne', 'sr', 'nb', 're', 'en', 'ra', 'dtp', 'fi', 'lt', 'rv', 'ob', 'rs', 'av', 'ds', 'ed']

Parents:
['MT', 'LY', 'SP', 'ID', 'NA', 'HI', 'IN', 'OP', 'IP']

Children:
['it', 'ne', 'sr', 'nb', 're', 'en', 'ra', 'dtp', 'fi', 'lt', 'rv', 'ob', 'rs', 'av', 'ds', 'ed']


Threshold = 0.50

Document-level summary:
           category  count  percentage
        exact_match    296   46.761453
          any_error    337   53.238547
  partially_correct    302   47.709321
   completely_wrong     33    5.213270
  nothing_predicted      2    0.315956
hierarchy_violation      3    0.473934

Hierarchy summary:
{
  "total_documents": 633,
  "documents_with_predicted_hierarchy_violation": 3,
  "percent_documents_with_predicted_hierarchy_violation": 0.47393364928909953,
  "documents_with_true_hierarchy_violation": 0,
  "child_correct_parent_missed_events": 1,
  "predicte

# Flat based

## New approach codes

In [3]:
# ============================================================
# HIERARCHICAL MULTILABEL ERROR ANALYSIS
#
# Designed for:
#
#   9 parent labels
#   16 child labels
#   25 total labels
#
# The analysis explicitly distinguishes:
#
#   1. Correct labels
#   2. False positives
#   3. False negatives
#   4. Missed-label -> false-positive associations
#   5. Completely missed documents
#   6. Partially correct documents
#   7. Exact label-set matches
#   8. Hierarchical violations
#   9. Child predicted but parent missing
#  10. Parent predicted without corresponding child
#
# ============================================================


# ============================================================
# IMPORTS
# ============================================================

import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    multilabel_confusion_matrix,
)


# ============================================================
# CONFIGURATION
# ============================================================

OUTPUT_DIR = Path(
    r"C:\Users\alrazz\Downloads\Flat XLMR"
)


LOGITS_FILE = (
    OUTPUT_DIR / "test_logits.npy"
)

LABELS_FILE = (
    OUTPUT_DIR / "test_labels.npy"
)

LABEL_NAMES_FILE = (
    OUTPUT_DIR / "hierarchical_labels.json"
)


ANALYSIS_DIR = (
    OUTPUT_DIR / "hierarchical_error_analysis"
)

ANALYSIS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# THRESHOLD CONFIGURATION
# ============================================================

GLOBAL_TUNED_THRESHOLD = 0.31


PER_LABEL_THRESHOLDS = {

    "MT":  0.28,
    "LY":  0.16,
    "SP": 0.36,
    "ID":  0.29,
    "NA":  0.57,
    "HI":  0.28,
    "IN":  0.4,
    "OP":  0.35,
    "IP":  	0.38,

    "it":  0.32,
    "ne":  0.4,
    "sr":  0.28,
    "nb":  0.35,
    "re":  0.29,

    "en":  0.33,
    "ra":  0.16,
    "dtp": 0.26,
    "fi":  0.28,
    "lt":  0.37,

    "rv":  0.22,
    "ob":  0.31,
    "rs":  0.3,
    "av":  0.17,

    "ds":  0.58,
    "ed":  0.24,
}


# ============================================================
# LOAD DATA
# ============================================================

print("\n" + "=" * 100)
print("LOADING TEST DATA")
print("=" * 100)


test_logits = np.load(
    LOGITS_FILE
)

test_labels = np.load(
    LABELS_FILE
)


with open(
    LABEL_NAMES_FILE,
    "r",
    encoding="utf-8",
) as f:

    hierarchical_labels = json.load(f)


print(
    "Logits shape:",
    test_logits.shape,
)

print(
    "Labels shape:",
    test_labels.shape,
)

print(
    "Labels:",
    hierarchical_labels,
)


# ============================================================
# BASIC CHECKS
# ============================================================

if test_logits.shape != test_labels.shape:

    raise ValueError(
        "Logits and labels have different shapes."
    )


NUM_LABELS = len(
    hierarchical_labels
)


if test_logits.shape[1] != NUM_LABELS:

    raise ValueError(
        "Number of logits does not match number of labels."
    )


# ============================================================
# HIERARCHICAL STRUCTURE
# ============================================================

NUM_PARENTS = 9


parents = hierarchical_labels[
    :NUM_PARENTS
]

children = hierarchical_labels[
    NUM_PARENTS:
]


parent_indices = list(
    range(NUM_PARENTS)
)

child_indices = list(
    range(NUM_PARENTS, NUM_LABELS)
)


# ------------------------------------------------------------
# Explicit child -> parent mapping
# ------------------------------------------------------------

child_to_parent = {

    "it": "SP",

    "ne": "NA",
    "sr": "NA",
    "nb": "NA",

    "re": "HI",

    "en": "IN",
    "ra": "IN",
    "dtp": "IN",
    "fi": "IN",
    "lt": "IN",

    "rv": "OP",
    "ob": "OP",
    "rs": "OP",
    "av": "OP",

    "ds": "IP",
    "ed": "IP",
}


print("\nParents:")
print(parents)

print("\nChildren:")
print(children)


# ============================================================
# VERIFY CHILD -> PARENT MAPPING
# ============================================================

for child in children:

    if child not in child_to_parent:

        raise ValueError(
            f"No parent defined for child: {child}"
        )

    parent = child_to_parent[child]

    if parent not in parents:

        raise ValueError(
            f"Unknown parent {parent} for child {child}"
        )


# ============================================================
# SIGMOID
# ============================================================

def sigmoid(x):

    # Numerically safer implementation
    x = np.clip(
        x,
        -50,
        50,
    )

    return 1.0 / (
        1.0 + np.exp(-x)
    )


test_probabilities = sigmoid(
    test_logits
)


# ============================================================
# THRESHOLDS
# ============================================================

missing_thresholds = [

    label

    for label in hierarchical_labels

    if label not in PER_LABEL_THRESHOLDS
]


if missing_thresholds:

    raise ValueError(
        "Missing thresholds for: "
        f"{missing_thresholds}"
    )


threshold_array = np.array(

    [
        PER_LABEL_THRESHOLDS[label]
        for label in hierarchical_labels
    ],

    dtype=np.float32,
)


# ============================================================
# PREDICTIONS
# ============================================================

def make_predictions(
    probabilities,
    thresholds,
):

    thresholds = np.asarray(
        thresholds,
        dtype=np.float32,
    )


    if thresholds.ndim == 0:

        thresholds = np.repeat(
            thresholds,
            probabilities.shape[1],
        )


    return (
        probabilities
        >= thresholds.reshape(1, -1)
    ).astype(np.int32)


predictions_05 = make_predictions(
    test_probabilities,
    0.50,
)


predictions_global = make_predictions(
    test_probabilities,
    GLOBAL_TUNED_THRESHOLD,
)


predictions_per_label = make_predictions(
    test_probabilities,
    threshold_array,
)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def label_set(
    row,
    labels=hierarchical_labels,
):

    return [
        labels[i]
        for i in np.where(
            row == 1
        )[0]
    ]


def label_string(
    row,
    labels=hierarchical_labels,
):

    values = label_set(
        row,
        labels,
    )

    if not values:

        return "NONE"

    return ", ".join(values)


def set_difference_string(
    true_row,
    pred_row,
):

    true_set = set(
        label_set(true_row)
    )

    pred_set = set(
        label_set(pred_row)
    )


    missed = sorted(
        true_set - pred_set
    )

    false_positive = sorted(
        pred_set - true_set
    )


    return (
        ", ".join(missed)
        if missed
        else "NONE"
    ), (
        ", ".join(false_positive)
        if false_positive
        else "NONE"
    )


# ============================================================
# HIERARCHICAL VIOLATIONS
# ============================================================
#
# A hierarchy violation occurs when:
#
# child = 1
# parent = 0
#
# Example:
#
# fi = 1
# IN = 0
#
# This is an invalid hierarchical prediction.
# ============================================================

def get_hierarchy_violations(
    prediction_row,
):

    violations = []


    for child, parent in child_to_parent.items():

        child_idx = hierarchical_labels.index(
            child
        )

        parent_idx = hierarchical_labels.index(
            parent
        )


        if (
            prediction_row[child_idx] == 1
            and
            prediction_row[parent_idx] == 0
        ):

            violations.append(
                f"{child}->{parent}"
            )


    return violations


# ============================================================
# TRUE HIERARCHY CHECK
# ============================================================
#
# This checks whether the GOLD labels themselves are
# hierarchically consistent.
#
# Normally all violations should be zero because your
# make_hierarchical_labels() function explicitly propagates
# child labels to parents.
# ============================================================

def get_true_hierarchy_violations(
    true_row,
):

    violations = []


    for child, parent in child_to_parent.items():

        child_idx = hierarchical_labels.index(
            child
        )

        parent_idx = hierarchical_labels.index(
            parent
        )


        if (
            true_row[child_idx] == 1
            and
            true_row[parent_idx] == 0
        ):

            violations.append(
                f"{child}->{parent}"
            )


    return violations


# ============================================================
# PER-DOCUMENT ERROR ANALYSIS
# ============================================================

def create_document_error_table(
    true_labels,
    predicted_labels,
    probabilities,
):

    rows = []


    for i in range(
        len(true_labels)
    ):

        true_row = true_labels[i]

        pred_row = predicted_labels[i]

        prob_row = probabilities[i]


        true_set = set(
            label_set(true_row)
        )

        pred_set = set(
            label_set(pred_row)
        )


        correct = sorted(
            true_set & pred_set
        )

        missed = sorted(
            true_set - pred_set
        )

        false_positive = sorted(
            pred_set - true_set
        )


        true_count = len(
            true_set
        )

        predicted_count = len(
            pred_set
        )

        correct_count = len(
            correct
        )

        missed_count = len(
            missed
        )

        false_positive_count = len(
            false_positive
        )


        # ----------------------------------------------------
        # Exact set match
        # ----------------------------------------------------

        exact_match = (
            true_set == pred_set
        )


        # ----------------------------------------------------
        # Completely missed
        # ----------------------------------------------------

        nothing_predicted = (
            predicted_count == 0
        )


        # ----------------------------------------------------
        # No true labels
        # ----------------------------------------------------

        no_true_labels = (
            true_count == 0
        )


        # ----------------------------------------------------
        # Completely wrong
        #
        # There is no overlap between true and predicted
        # labels, while both contain at least one label.
        # ----------------------------------------------------

        completely_wrong = (

            true_count > 0

            and

            predicted_count > 0

            and

            correct_count == 0
        )


        # ----------------------------------------------------
        # Partially correct
        # ----------------------------------------------------

        partially_correct = (

            correct_count > 0

            and

            not exact_match
        )


        # ----------------------------------------------------
        # Hierarchical violations
        # ----------------------------------------------------

        hierarchy_violations = (
            get_hierarchy_violations(
                pred_row
            )
        )


        true_hierarchy_violations = (
            get_true_hierarchy_violations(
                true_row
            )
        )


        # ----------------------------------------------------
        # Probability of true labels
        # ----------------------------------------------------

        true_probabilities = [

            float(
                prob_row[
                    hierarchical_labels.index(label)
                ]
            )

            for label in sorted(true_set)

        ]


        # ----------------------------------------------------
        # Probability of missed labels
        # ----------------------------------------------------

        missed_probabilities = [

            float(
                prob_row[
                    hierarchical_labels.index(label)
                ]
            )

            for label in missed

        ]


        # ----------------------------------------------------
        # Probability of false positives
        # ----------------------------------------------------

        false_positive_probabilities = [

            float(
                prob_row[
                    hierarchical_labels.index(label)
                ]
            )

            for label in false_positive

        ]


        rows.append({

            "document_index": i,

            "true_labels":
                ", ".join(
                    sorted(true_set)
                )
                if true_set
                else "NONE",

            "predicted_labels":
                ", ".join(
                    sorted(pred_set)
                )
                if pred_set
                else "NONE",

            "correct_labels":
                ", ".join(correct)
                if correct
                else "NONE",

            "missed_labels":
                ", ".join(missed)
                if missed
                else "NONE",

            "false_positive_labels":
                ", ".join(false_positive)
                if false_positive
                else "NONE",

            "true_label_count":
                true_count,

            "predicted_label_count":
                predicted_count,

            "correct_label_count":
                correct_count,

            "missed_label_count":
                missed_count,

            "false_positive_count":
                false_positive_count,

            "exact_match":
                exact_match,

            "partially_correct":
                partially_correct,

            "completely_wrong":
                completely_wrong,

            "nothing_predicted":
                nothing_predicted,

            "hierarchy_violation":
                len(
                    hierarchy_violations
                ) > 0,

            "hierarchy_violations":
                "; ".join(
                    hierarchy_violations
                )
                if hierarchy_violations
                else "NONE",

            "true_hierarchy_violation":
                len(
                    true_hierarchy_violations
                ) > 0,

            "true_hierarchy_violations":
                "; ".join(
                    true_hierarchy_violations
                )
                if true_hierarchy_violations
                else "NONE",

            "true_probabilities":
                "; ".join(
                    f"{p:.4f}"
                    for p in true_probabilities
                ),

            "missed_probabilities":
                "; ".join(
                    f"{p:.4f}"
                    for p in missed_probabilities
                ),

            "false_positive_probabilities":
                "; ".join(
                    f"{p:.4f}"
                    for p in false_positive_probabilities
                ),
        })


    return pd.DataFrame(rows)


# ============================================================
# PER-LABEL TP / FP / FN / TN
# ============================================================

def create_per_label_summary(
    true_labels,
    predicted_labels,
):

    mcm = multilabel_confusion_matrix(
        true_labels,
        predicted_labels,
    )


    rows = []


    for i, label in enumerate(
        hierarchical_labels
    ):

        tn = mcm[i, 0, 0]

        fp = mcm[i, 0, 1]

        fn = mcm[i, 1, 0]

        tp = mcm[i, 1, 1]


        precision = (
            tp / (tp + fp)
            if (tp + fp) > 0
            else 0.0
        )


        recall = (
            tp / (tp + fn)
            if (tp + fn) > 0
            else 0.0
        )


        f1 = (

            2 * precision * recall
            /
            (precision + recall)

            if (precision + recall) > 0

            else 0.0
        )


        rows.append({

            "label": label,

            "TP": int(tp),

            "FP": int(fp),

            "FN": int(fn),

            "TN": int(tn),

            "precision":
                precision,

            "recall":
                recall,

            "F1":
                f1,

            "support":
                int(tp + fn),

        })


    return pd.DataFrame(rows)


# ============================================================
# MISSED -> FALSE POSITIVE ASSOCIATION
# ============================================================
#
# Example:
#
# TRUE:
#     IN OP
#
# PRED:
#     IN NA
#
# Result:
#
#     OP -> NA
#
# Important:
#
# This means:
#
# "When OP was missed, NA was falsely predicted in the
# same document."
#
# It does NOT mathematically prove that NA replaced OP.
# ============================================================

def create_missed_to_fp_matrix(
    true_labels,
    predicted_labels,
):

    n_labels = len(
        hierarchical_labels
    )


    matrix = np.zeros(
        (n_labels, n_labels),
        dtype=np.int64,
    )


    for i in range(
        len(true_labels)
    ):

        true_row = true_labels[i]

        pred_row = predicted_labels[i]


        missed = np.where(

            (true_row == 1)

            &

            (pred_row == 0)

        )[0]


        false_positive = np.where(

            (true_row == 0)

            &

            (pred_row == 1)

        )[0]


        for missed_idx in missed:

            for fp_idx in false_positive:

                matrix[
                    missed_idx,
                    fp_idx
                ] += 1


    return matrix


# ============================================================
# SAVE MATRIX
# ============================================================

def save_matrix(
    matrix,
    filename,
):

    df = pd.DataFrame(

        matrix,

        index=hierarchical_labels,

        columns=hierarchical_labels,
    )


    df.to_csv(

        ANALYSIS_DIR / filename,

        encoding="utf-8-sig",
    )


    return df


# ============================================================
# PLOT MATRIX
# ============================================================

def plot_matrix(
    matrix,
    filename,
    title,
    xlabel,
    ylabel,
):

    plt.figure(
        figsize=(20, 17)
    )


    sns.heatmap(

        matrix,

        annot=True,

        fmt="d",

        cmap="Blues",

        xticklabels=hierarchical_labels,

        yticklabels=hierarchical_labels,

        linewidths=0.4,

    )


    plt.xlabel(
        xlabel,
        fontsize=13,
        fontweight="bold",
    )


    plt.ylabel(
        ylabel,
        fontsize=13,
        fontweight="bold",
    )


    plt.title(
        title,
        fontsize=16,
        fontweight="bold",
    )


    plt.xticks(
        rotation=45,
        ha="right",
    )


    plt.yticks(
        rotation=0,
    )


    plt.tight_layout()


    plt.savefig(

        ANALYSIS_DIR / filename,

        dpi=400,

        bbox_inches="tight",
    )


    plt.close()


# ============================================================
# EXACT TRUE -> PREDICTED SET PATTERNS
# ============================================================
#
# This is one of the most important analyses.
#
# Example:
#
# TRUE = IN, OP
# PRED = IN, NA
#
# gets one row:
#
# IN, OP -> IN, NA : 37 documents
#
# ============================================================

def create_exact_pattern_table(
    true_labels,
    predicted_labels,
):

    counter = Counter()


    for true_row, pred_row in zip(
        true_labels,
        predicted_labels,
    ):

        true_string = label_string(
            true_row
        )

        pred_string = label_string(
            pred_row
        )


        counter[
            (
                true_string,
                pred_string,
            )
        ] += 1


    rows = []


    for (
        (
            true_string,
            pred_string,
        ),
        count,
    ) in counter.most_common():

        rows.append({

            "true_labels":
                true_string,

            "predicted_labels":
                pred_string,

            "count":
                count,

        })


    return pd.DataFrame(rows)


# ============================================================
# HIERARCHY ERROR ANALYSIS
# ============================================================

def create_hierarchy_error_table(
    true_labels,
    predicted_labels,
):

    rows = []

    for i in range(
        len(true_labels)
    ):

        true_row = true_labels[i]

        pred_row = predicted_labels[i]

        for child, parent in child_to_parent.items():

            child_idx = hierarchical_labels.index(
                child
            )

            parent_idx = hierarchical_labels.index(
                parent
            )

            child_true = (
                true_row[child_idx] == 1
            )

            parent_true = (
                true_row[parent_idx] == 1
            )

            child_pred = (
                pred_row[child_idx] == 1
            )

            parent_pred = (
                pred_row[parent_idx] == 1
            )


            # ------------------------------------------------
            # ERROR TYPE 1:
            #
            # Child is truly present and correctly predicted,
            # but its required parent is missed.
            #
            # TRUE:
            #     child  = 1
            #     parent = 1
            #
            # PRED:
            #     child  = 1
            #     parent = 0
            # ------------------------------------------------

            if (
                child_true
                and
                parent_true
                and
                child_pred
                and
                not parent_pred
            ):

                rows.append({

                    "document_index":
                        i,

                    "child":
                        child,

                    "parent":
                        parent,

                    "error_type":
                        "child_correct_parent_missed",

                    "true_child":
                        1,

                    "true_parent":
                        1,

                    "pred_child":
                        1,

                    "pred_parent":
                        0,

                })


            # ------------------------------------------------
            # ERROR TYPE 2:
            #
            # Model predicts a child without predicting its
            # required parent.
            #
            # This is a prediction-time hierarchy violation.
            #
            # It can happen whether or not the child is truly
            # present in the gold annotation.
            #
            # PRED:
            #     child  = 1
            #     parent = 0
            # ------------------------------------------------

            if (
                child_pred
                and
                not parent_pred
            ):

                rows.append({

                    "document_index":
                        i,

                    "child":
                        child,

                    "parent":
                        parent,

                    "error_type":
                        "hierarchy_violation",

                    "true_child":
                        int(child_true),

                    "true_parent":
                        int(parent_true),

                    "pred_child":
                        1,

                    "pred_parent":
                        0,

                })


            # ------------------------------------------------
            # ERROR TYPE 3:
            #
            # A TRUE child was missed while its parent was
            # correctly predicted.
            #
            # TRUE:
            #     child  = 1
            #     parent = 1
            #
            # PRED:
            #     child  = 0
            #     parent = 1
            # ------------------------------------------------

            if (
                child_true
                and
                parent_true
                and
                parent_pred
                and
                not child_pred
            ):

                rows.append({

                    "document_index":
                        i,

                    "child":
                        child,

                    "parent":
                        parent,

                    "error_type":
                        "parent_predicted_child_not_predicted",

                    "true_child":
                        1,

                    "true_parent":
                        1,

                    "pred_child":
                        0,

                    "pred_parent":
                        1,

                })


    return pd.DataFrame(
        rows
    )

# ============================================================
# HIERARCHICAL ERROR COUNTS
# ============================================================

def hierarchy_summary(
    true_labels,
    predicted_labels,
):

    total = len(
        true_labels
    )


    prediction_violations = 0

    child_correct_parent_missed = 0

    true_hierarchy_violations = 0


    violation_pairs = Counter()

    child_parent_missed_pairs = Counter()


    for i in range(total):

        true_row = true_labels[i]

        pred_row = predicted_labels[i]


        true_violations = (
            get_true_hierarchy_violations(
                true_row
            )
        )


        pred_violations = (
            get_hierarchy_violations(
                pred_row
            )
        )


        if true_violations:

            true_hierarchy_violations += 1


        if pred_violations:

            prediction_violations += 1


            for pair in pred_violations:

                violation_pairs[pair] += 1


        for child, parent in child_to_parent.items():

            child_idx = hierarchical_labels.index(
                child
            )

            parent_idx = hierarchical_labels.index(
                parent
            )


            if (

                true_row[child_idx] == 1

                and

                true_row[parent_idx] == 1

                and

                pred_row[child_idx] == 1

                and

                pred_row[parent_idx] == 0

            ):

                child_correct_parent_missed += 1

                child_parent_missed_pairs[
                    f"{child}->{parent}"
                ] += 1


    return {

        "total_documents":
            total,

        "documents_with_predicted_hierarchy_violation":
            prediction_violations,

        "percent_documents_with_predicted_hierarchy_violation":
            (
                prediction_violations / total * 100
                if total > 0
                else 0
            ),

        "documents_with_true_hierarchy_violation":
            true_hierarchy_violations,

        "child_correct_parent_missed_events":
            child_correct_parent_missed,

        "predicted_hierarchy_violation_pairs":
            dict(violation_pairs),

        "child_correct_parent_missed_pairs":
            dict(child_parent_missed_pairs),
    }


# ============================================================
# DOCUMENT-LEVEL SUMMARY
# ============================================================

def create_document_summary(
    true_labels,
    predicted_labels,
):

    total = len(
        true_labels
    )


    exact = 0

    partial = 0

    completely_wrong = 0

    nothing = 0

    any_error = 0

    hierarchy_violation = 0


    for i in range(total):

        true_set = set(
            label_set(
                true_labels[i]
            )
        )

        pred_set = set(
            label_set(
                predicted_labels[i]
            )
        )


        if true_set == pred_set:

            exact += 1

        else:

            any_error += 1


        if (
            true_set
            and
            pred_set
            and
            true_set & pred_set
            and
            true_set != pred_set
        ):

            partial += 1


        if (
            true_set
            and
            pred_set
            and
            not (true_set & pred_set)
        ):

            completely_wrong += 1


        if (
            true_set
            and
            not pred_set
        ):

            nothing += 1

        hierarchy_violations = get_hierarchy_violations(
            predicted_labels[i]
        )

        if hierarchy_violations:
            hierarchy_violation += 1



    return pd.DataFrame({

        "category": [

            "exact_match",

            "any_error",

            "partially_correct",

            "completely_wrong",

            "nothing_predicted",

             "hierarchy_violation",

        ],

        "count": [

            exact,

            any_error,

            partial,

            completely_wrong,

            nothing,

             hierarchy_violation,

        ],

        "percentage": [

            exact / total * 100,

            any_error / total * 100,

            partial / total * 100,

            completely_wrong / total * 100,

            nothing / total * 100,

            hierarchy_violation / total * 100,

        ],

    })


# ============================================================
# RUN ANALYSIS
# ============================================================

def run_analysis(
    true_labels,
    predicted_labels,
    probabilities,
    prefix,
    description,
):

    print("\n")
    print("=" * 100)
    print(description)
    print("=" * 100)


    # ========================================================
    # 1. DOCUMENT-LEVEL ERROR TABLE
    # ========================================================

    document_df = create_document_error_table(

        true_labels,

        predicted_labels,

        probabilities,

    )


    document_df.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_document_errors.csv",

        index=False,

        encoding="utf-8-sig",
    )


    # --------------------------------------------------------
    # Save only documents containing errors
    # --------------------------------------------------------

    error_documents = document_df[
        ~document_df["exact_match"]
    ]


    error_documents.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_error_documents_only.csv",

        index=False,

        encoding="utf-8-sig",
    )


    # ========================================================
    # 2. EXACT LABEL-SET PATTERNS
    # ========================================================

    pattern_df = create_exact_pattern_table(

        true_labels,

        predicted_labels,

    )


    pattern_df.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_exact_label_set_patterns.csv",

        index=False,

        encoding="utf-8-sig",
    )


    # ========================================================
    # 3. PER-LABEL SUMMARY
    # ========================================================

    label_df = create_per_label_summary(

        true_labels,

        predicted_labels,

    )


    label_df.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_per_label_summary.csv",

        index=False,

        encoding="utf-8-sig",
    )


    # ========================================================
    # 4. MISSED -> FALSE POSITIVE MATRIX
    # ========================================================

    missed_fp_matrix = create_missed_to_fp_matrix(

        true_labels,

        predicted_labels,

    )


    save_matrix(

        missed_fp_matrix,

        f"{prefix}_missed_to_false_positive.csv",

    )


    plot_matrix(

        missed_fp_matrix,

        f"{prefix}_missed_to_false_positive.png",

        (
            "Missed True Label → False-Positive Label\n"
            + description
        ),

        "False-Positive Label",

        "Missed True Label",

    )


    # ========================================================
    # 5. HIERARCHY ERRORS
    # ========================================================

    hierarchy_df = create_hierarchy_error_table(

        true_labels,

        predicted_labels,

    )


    hierarchy_df.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_hierarchy_errors.csv",

        index=False,

        encoding="utf-8-sig",
    )


    # ========================================================
    # 6. HIERARCHY SUMMARY
    # ========================================================

    hierarchy_stats = hierarchy_summary(

        true_labels,

        predicted_labels,

    )


    with open(

        ANALYSIS_DIR
        / f"{prefix}_hierarchy_summary.json",

        "w",

        encoding="utf-8",

    ) as f:

        json.dump(

            hierarchy_stats,

            f,

            indent=2,

        )


    # ========================================================
    # 7. DOCUMENT SUMMARY
    # ========================================================

    doc_summary = create_document_summary(

        true_labels,

        predicted_labels,

    )


    doc_summary.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_document_summary.csv",

        index=False,

        encoding="utf-8-sig",
    )


    # ========================================================
    # 8. PRINT SUMMARY
    # ========================================================

    print("\nDocument-level summary:")

    print(
        doc_summary.to_string(
            index=False
        )
    )


    print("\nHierarchy summary:")

    print(
        json.dumps(
            hierarchy_stats,
            indent=2,
        )
    )


    # ========================================================
    # 9. TOP ERROR PATTERNS
    # ========================================================

    print("\nTop 20 true -> predicted label-set patterns:")

    print(

        pattern_df.head(20).to_string(
            index=False
        )

    )


    print("\nTop 20 missed -> false-positive pairs:")

    pair_rows = []


    for i, true_label in enumerate(
        hierarchical_labels
    ):

        for j, predicted_label in enumerate(
            hierarchical_labels
        ):

            count = missed_fp_matrix[i, j]


            if count > 0:

                pair_rows.append({

                    "missed_label":
                        true_label,

                    "false_positive_label":
                        predicted_label,

                    "count":
                        int(count),

                })


    pair_df = pd.DataFrame(
        pair_rows
    )


    if len(pair_df) > 0:

        pair_df = pair_df.sort_values(
            "count",
            ascending=False,
        )


        print(

            pair_df.head(20).to_string(
                index=False
            )

        )


    # ========================================================
    # 10. TOP HIERARCHY VIOLATIONS
    # ========================================================

    print("\nHierarchy violation examples:")

    if len(hierarchy_df) > 0:

        print(

            hierarchy_df.head(20).to_string(
                index=False
            )

        )

    else:

        print(
            "No hierarchy violations found."
        )


# ============================================================
# RUN 1:
# ORIGINAL 0.50
# ============================================================

run_analysis(

    true_labels=test_labels,

    predicted_labels=predictions_05,

    probabilities=test_probabilities,

    prefix="threshold_0.50",

    description=(
        "Threshold = 0.50"
    ),

)


# ============================================================
# RUN 2:
# GLOBAL TUNED
# ============================================================

run_analysis(

    true_labels=test_labels,

    predicted_labels=predictions_global,

    probabilities=test_probabilities,

    prefix="global_tuned",

    description=(
        f"Global threshold = "
        f"{GLOBAL_TUNED_THRESHOLD:.3f}"
    ),

)


# ============================================================
# RUN 3:
# PER-LABEL TUNED
# ============================================================

run_analysis(

    true_labels=test_labels,

    predicted_labels=predictions_per_label,

    probabilities=test_probabilities,

    prefix="per_label_tuned",

    description=(
        "Per-label tuned thresholds"
    ),

)


# ============================================================
# SAVE THRESHOLDS
# ============================================================

threshold_df = pd.DataFrame({

    "label":
        hierarchical_labels,

    "threshold":
        threshold_array,

})


threshold_df.to_csv(

    ANALYSIS_DIR
    / "thresholds_used.csv",

    index=False,

    encoding="utf-8-sig",
)


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 100)
print("HIERARCHICAL ERROR ANALYSIS COMPLETE")
print("=" * 100)

print("\nResults saved to:")

print(
    ANALYSIS_DIR
)

print("\nThe most important files are:")

print(
    "  *_document_errors.csv"
)

print(
    "  *_error_documents_only.csv"
)

print(
    "  *_exact_label_set_patterns.csv"
)

print(
    "  *_per_label_summary.csv"
)

print(
    "  *_missed_to_false_positive.csv"
)

print(
    "  *_hierarchy_errors.csv"
)

print(
    "  *_hierarchy_summary.json"
)

print(
    "  *_document_summary.csv"
)

print("=" * 100)



LOADING TEST DATA
Logits shape: (633, 25)
Labels shape: (633, 25)
Labels: ['MT', 'LY', 'SP', 'ID', 'NA', 'HI', 'IN', 'OP', 'IP', 'it', 'ne', 'sr', 'nb', 're', 'en', 'ra', 'dtp', 'fi', 'lt', 'rv', 'ob', 'rs', 'av', 'ds', 'ed']

Parents:
['MT', 'LY', 'SP', 'ID', 'NA', 'HI', 'IN', 'OP', 'IP']

Children:
['it', 'ne', 'sr', 'nb', 're', 'en', 'ra', 'dtp', 'fi', 'lt', 'rv', 'ob', 'rs', 'av', 'ds', 'ed']


Threshold = 0.50

Document-level summary:
           category  count  percentage
        exact_match    268   42.338073
          any_error    365   57.661927
  partially_correct    320   50.552923
   completely_wrong     38    6.003160
  nothing_predicted      7    1.105845
hierarchy_violation      0    0.000000

Hierarchy summary:
{
  "total_documents": 633,
  "documents_with_predicted_hierarchy_violation": 0,
  "percent_documents_with_predicted_hierarchy_violation": 0.0,
  "documents_with_true_hierarchy_violation": 0,
  "child_correct_parent_missed_events": 0,
  "predicted_hierarchy_viol

# Parent Conditioned

## New approach codes

In [4]:
# ============================================================
# HIERARCHICAL MULTILABEL ERROR ANALYSIS
#
# Designed for:
#
#   9 parent labels
#   16 child labels
#   25 total labels
#
# The analysis explicitly distinguishes:
#
#   1. Correct labels
#   2. False positives
#   3. False negatives
#   4. Missed-label -> false-positive associations
#   5. Completely missed documents
#   6. Partially correct documents
#   7. Exact label-set matches
#   8. Hierarchical violations
#   9. Child predicted but parent missing
#  10. Parent predicted without corresponding child
#
# ============================================================


# ============================================================
# IMPORTS
# ============================================================

import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    multilabel_confusion_matrix,
)


# ============================================================
# CONFIGURATION
# ============================================================

OUTPUT_DIR = Path(
    r"C:\Users\alrazz\Downloads\Parent_Conditioned_Hierarchica"
)


LOGITS_FILE = (
    OUTPUT_DIR / "test_logits.npy"
)

LABELS_FILE = (
    OUTPUT_DIR / "test_labels.npy"
)

LABEL_NAMES_FILE = (
    OUTPUT_DIR / "hierarchical_labels.json"
)


ANALYSIS_DIR = (
    OUTPUT_DIR / "hierarchical_error_analysis"
)

ANALYSIS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# THRESHOLD CONFIGURATION
# ============================================================

GLOBAL_TUNED_THRESHOLD = 0.28


PER_LABEL_THRESHOLDS = {

    "MT":  0.11,
    "LY":  0.23,
    "SP": 0.36,
    "ID":  0.26,
    "NA":  0.40,
    "HI":  0.39,
    "IN":  0.52,
    "OP":  0.39,
    "IP":  	0.41,

    "it":  0.31,
    "ne":  0.40,
    "sr":   0.33,
    "nb":  0.13,
    "re":  0.28,

    "en":  0.50,
    "ra":  0.10,
    "dtp": 0.33,
    "fi":  0.53,
    "lt":  0.28,

    "rv":  0.23,
    "ob":  0.30,
    "rs":  0.39,
    "av":  0.60,

    "ds":  0.80,
    "ed":  0.24,
}


# ============================================================
# LOAD DATA
# ============================================================

print("\n" + "=" * 100)
print("LOADING TEST DATA")
print("=" * 100)


test_logits = np.load(
    LOGITS_FILE
)

test_labels = np.load(
    LABELS_FILE
)


with open(
    LABEL_NAMES_FILE,
    "r",
    encoding="utf-8",
) as f:

    hierarchical_labels = json.load(f)


print(
    "Logits shape:",
    test_logits.shape,
)

print(
    "Labels shape:",
    test_labels.shape,
)

print(
    "Labels:",
    hierarchical_labels,
)


# ============================================================
# BASIC CHECKS
# ============================================================

if test_logits.shape != test_labels.shape:

    raise ValueError(
        "Logits and labels have different shapes."
    )


NUM_LABELS = len(
    hierarchical_labels
)


if test_logits.shape[1] != NUM_LABELS:

    raise ValueError(
        "Number of logits does not match number of labels."
    )


# ============================================================
# HIERARCHICAL STRUCTURE
# ============================================================

NUM_PARENTS = 9


parents = hierarchical_labels[
    :NUM_PARENTS
]

children = hierarchical_labels[
    NUM_PARENTS:
]


parent_indices = list(
    range(NUM_PARENTS)
)

child_indices = list(
    range(NUM_PARENTS, NUM_LABELS)
)


# ------------------------------------------------------------
# Explicit child -> parent mapping
# ------------------------------------------------------------

child_to_parent = {

    "it": "SP",

    "ne": "NA",
    "sr": "NA",
    "nb": "NA",

    "re": "HI",

    "en": "IN",
    "ra": "IN",
    "dtp": "IN",
    "fi": "IN",
    "lt": "IN",

    "rv": "OP",
    "ob": "OP",
    "rs": "OP",
    "av": "OP",

    "ds": "IP",
    "ed": "IP",
}


print("\nParents:")
print(parents)

print("\nChildren:")
print(children)


# ============================================================
# VERIFY CHILD -> PARENT MAPPING
# ============================================================

for child in children:

    if child not in child_to_parent:

        raise ValueError(
            f"No parent defined for child: {child}"
        )

    parent = child_to_parent[child]

    if parent not in parents:

        raise ValueError(
            f"Unknown parent {parent} for child {child}"
        )


# ============================================================
# SIGMOID
# ============================================================

def sigmoid(x):

    # Numerically safer implementation
    x = np.clip(
        x,
        -50,
        50,
    )

    return 1.0 / (
        1.0 + np.exp(-x)
    )


test_probabilities = sigmoid(
    test_logits
)


# ============================================================
# THRESHOLDS
# ============================================================

missing_thresholds = [

    label

    for label in hierarchical_labels

    if label not in PER_LABEL_THRESHOLDS
]


if missing_thresholds:

    raise ValueError(
        "Missing thresholds for: "
        f"{missing_thresholds}"
    )


threshold_array = np.array(

    [
        PER_LABEL_THRESHOLDS[label]
        for label in hierarchical_labels
    ],

    dtype=np.float32,
)


# ============================================================
# PREDICTIONS
# ============================================================

def make_predictions(
    probabilities,
    thresholds,
):

    thresholds = np.asarray(
        thresholds,
        dtype=np.float32,
    )


    if thresholds.ndim == 0:

        thresholds = np.repeat(
            thresholds,
            probabilities.shape[1],
        )


    return (
        probabilities
        >= thresholds.reshape(1, -1)
    ).astype(np.int32)


predictions_05 = make_predictions(
    test_probabilities,
    0.50,
)


predictions_global = make_predictions(
    test_probabilities,
    GLOBAL_TUNED_THRESHOLD,
)


predictions_per_label = make_predictions(
    test_probabilities,
    threshold_array,
)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def label_set(
    row,
    labels=hierarchical_labels,
):

    return [
        labels[i]
        for i in np.where(
            row == 1
        )[0]
    ]


def label_string(
    row,
    labels=hierarchical_labels,
):

    values = label_set(
        row,
        labels,
    )

    if not values:

        return "NONE"

    return ", ".join(values)


def set_difference_string(
    true_row,
    pred_row,
):

    true_set = set(
        label_set(true_row)
    )

    pred_set = set(
        label_set(pred_row)
    )


    missed = sorted(
        true_set - pred_set
    )

    false_positive = sorted(
        pred_set - true_set
    )


    return (
        ", ".join(missed)
        if missed
        else "NONE"
    ), (
        ", ".join(false_positive)
        if false_positive
        else "NONE"
    )


# ============================================================
# HIERARCHICAL VIOLATIONS
# ============================================================
#
# A hierarchy violation occurs when:
#
# child = 1
# parent = 0
#
# Example:
#
# fi = 1
# IN = 0
#
# This is an invalid hierarchical prediction.
# ============================================================

def get_hierarchy_violations(
    prediction_row,
):

    violations = []


    for child, parent in child_to_parent.items():

        child_idx = hierarchical_labels.index(
            child
        )

        parent_idx = hierarchical_labels.index(
            parent
        )


        if (
            prediction_row[child_idx] == 1
            and
            prediction_row[parent_idx] == 0
        ):

            violations.append(
                f"{child}->{parent}"
            )


    return violations


# ============================================================
# TRUE HIERARCHY CHECK
# ============================================================
#
# This checks whether the GOLD labels themselves are
# hierarchically consistent.
#
# Normally all violations should be zero because your
# make_hierarchical_labels() function explicitly propagates
# child labels to parents.
# ============================================================

def get_true_hierarchy_violations(
    true_row,
):

    violations = []


    for child, parent in child_to_parent.items():

        child_idx = hierarchical_labels.index(
            child
        )

        parent_idx = hierarchical_labels.index(
            parent
        )


        if (
            true_row[child_idx] == 1
            and
            true_row[parent_idx] == 0
        ):

            violations.append(
                f"{child}->{parent}"
            )


    return violations


# ============================================================
# PER-DOCUMENT ERROR ANALYSIS
# ============================================================

def create_document_error_table(
    true_labels,
    predicted_labels,
    probabilities,
):

    rows = []


    for i in range(
        len(true_labels)
    ):

        true_row = true_labels[i]

        pred_row = predicted_labels[i]

        prob_row = probabilities[i]


        true_set = set(
            label_set(true_row)
        )

        pred_set = set(
            label_set(pred_row)
        )


        correct = sorted(
            true_set & pred_set
        )

        missed = sorted(
            true_set - pred_set
        )

        false_positive = sorted(
            pred_set - true_set
        )


        true_count = len(
            true_set
        )

        predicted_count = len(
            pred_set
        )

        correct_count = len(
            correct
        )

        missed_count = len(
            missed
        )

        false_positive_count = len(
            false_positive
        )


        # ----------------------------------------------------
        # Exact set match
        # ----------------------------------------------------

        exact_match = (
            true_set == pred_set
        )


        # ----------------------------------------------------
        # Completely missed
        # ----------------------------------------------------

        nothing_predicted = (
            predicted_count == 0
        )


        # ----------------------------------------------------
        # No true labels
        # ----------------------------------------------------

        no_true_labels = (
            true_count == 0
        )


        # ----------------------------------------------------
        # Completely wrong
        #
        # There is no overlap between true and predicted
        # labels, while both contain at least one label.
        # ----------------------------------------------------

        completely_wrong = (

            true_count > 0

            and

            predicted_count > 0

            and

            correct_count == 0
        )


        # ----------------------------------------------------
        # Partially correct
        # ----------------------------------------------------

        partially_correct = (

            correct_count > 0

            and

            not exact_match
        )


        # ----------------------------------------------------
        # Hierarchical violations
        # ----------------------------------------------------

        hierarchy_violations = (
            get_hierarchy_violations(
                pred_row
            )
        )


        true_hierarchy_violations = (
            get_true_hierarchy_violations(
                true_row
            )
        )


        # ----------------------------------------------------
        # Probability of true labels
        # ----------------------------------------------------

        true_probabilities = [

            float(
                prob_row[
                    hierarchical_labels.index(label)
                ]
            )

            for label in sorted(true_set)

        ]


        # ----------------------------------------------------
        # Probability of missed labels
        # ----------------------------------------------------

        missed_probabilities = [

            float(
                prob_row[
                    hierarchical_labels.index(label)
                ]
            )

            for label in missed

        ]


        # ----------------------------------------------------
        # Probability of false positives
        # ----------------------------------------------------

        false_positive_probabilities = [

            float(
                prob_row[
                    hierarchical_labels.index(label)
                ]
            )

            for label in false_positive

        ]


        rows.append({

            "document_index": i,

            "true_labels":
                ", ".join(
                    sorted(true_set)
                )
                if true_set
                else "NONE",

            "predicted_labels":
                ", ".join(
                    sorted(pred_set)
                )
                if pred_set
                else "NONE",

            "correct_labels":
                ", ".join(correct)
                if correct
                else "NONE",

            "missed_labels":
                ", ".join(missed)
                if missed
                else "NONE",

            "false_positive_labels":
                ", ".join(false_positive)
                if false_positive
                else "NONE",

            "true_label_count":
                true_count,

            "predicted_label_count":
                predicted_count,

            "correct_label_count":
                correct_count,

            "missed_label_count":
                missed_count,

            "false_positive_count":
                false_positive_count,

            "exact_match":
                exact_match,

            "partially_correct":
                partially_correct,

            "completely_wrong":
                completely_wrong,

            "nothing_predicted":
                nothing_predicted,

            "hierarchy_violation":
                len(
                    hierarchy_violations
                ) > 0,

            "hierarchy_violations":
                "; ".join(
                    hierarchy_violations
                )
                if hierarchy_violations
                else "NONE",

            "true_hierarchy_violation":
                len(
                    true_hierarchy_violations
                ) > 0,

            "true_hierarchy_violations":
                "; ".join(
                    true_hierarchy_violations
                )
                if true_hierarchy_violations
                else "NONE",

            "true_probabilities":
                "; ".join(
                    f"{p:.4f}"
                    for p in true_probabilities
                ),

            "missed_probabilities":
                "; ".join(
                    f"{p:.4f}"
                    for p in missed_probabilities
                ),

            "false_positive_probabilities":
                "; ".join(
                    f"{p:.4f}"
                    for p in false_positive_probabilities
                ),
        })


    return pd.DataFrame(rows)


# ============================================================
# PER-LABEL TP / FP / FN / TN
# ============================================================

def create_per_label_summary(
    true_labels,
    predicted_labels,
):

    mcm = multilabel_confusion_matrix(
        true_labels,
        predicted_labels,
    )


    rows = []


    for i, label in enumerate(
        hierarchical_labels
    ):

        tn = mcm[i, 0, 0]

        fp = mcm[i, 0, 1]

        fn = mcm[i, 1, 0]

        tp = mcm[i, 1, 1]


        precision = (
            tp / (tp + fp)
            if (tp + fp) > 0
            else 0.0
        )


        recall = (
            tp / (tp + fn)
            if (tp + fn) > 0
            else 0.0
        )


        f1 = (

            2 * precision * recall
            /
            (precision + recall)

            if (precision + recall) > 0

            else 0.0
        )


        rows.append({

            "label": label,

            "TP": int(tp),

            "FP": int(fp),

            "FN": int(fn),

            "TN": int(tn),

            "precision":
                precision,

            "recall":
                recall,

            "F1":
                f1,

            "support":
                int(tp + fn),

        })


    return pd.DataFrame(rows)


# ============================================================
# MISSED -> FALSE POSITIVE ASSOCIATION
# ============================================================
#
# Example:
#
# TRUE:
#     IN OP
#
# PRED:
#     IN NA
#
# Result:
#
#     OP -> NA
#
# Important:
#
# This means:
#
# "When OP was missed, NA was falsely predicted in the
# same document."
#
# It does NOT mathematically prove that NA replaced OP.
# ============================================================

def create_missed_to_fp_matrix(
    true_labels,
    predicted_labels,
):

    n_labels = len(
        hierarchical_labels
    )


    matrix = np.zeros(
        (n_labels, n_labels),
        dtype=np.int64,
    )


    for i in range(
        len(true_labels)
    ):

        true_row = true_labels[i]

        pred_row = predicted_labels[i]


        missed = np.where(

            (true_row == 1)

            &

            (pred_row == 0)

        )[0]


        false_positive = np.where(

            (true_row == 0)

            &

            (pred_row == 1)

        )[0]


        for missed_idx in missed:

            for fp_idx in false_positive:

                matrix[
                    missed_idx,
                    fp_idx
                ] += 1


    return matrix


# ============================================================
# SAVE MATRIX
# ============================================================

def save_matrix(
    matrix,
    filename,
):

    df = pd.DataFrame(

        matrix,

        index=hierarchical_labels,

        columns=hierarchical_labels,
    )


    df.to_csv(

        ANALYSIS_DIR / filename,

        encoding="utf-8-sig",
    )


    return df


# ============================================================
# PLOT MATRIX
# ============================================================

def plot_matrix(
    matrix,
    filename,
    title,
    xlabel,
    ylabel,
):

    plt.figure(
        figsize=(20, 17)
    )


    sns.heatmap(

        matrix,

        annot=True,

        fmt="d",

        cmap="Blues",

        xticklabels=hierarchical_labels,

        yticklabels=hierarchical_labels,

        linewidths=0.4,

    )


    plt.xlabel(
        xlabel,
        fontsize=13,
        fontweight="bold",
    )


    plt.ylabel(
        ylabel,
        fontsize=13,
        fontweight="bold",
    )


    plt.title(
        title,
        fontsize=16,
        fontweight="bold",
    )


    plt.xticks(
        rotation=45,
        ha="right",
    )


    plt.yticks(
        rotation=0,
    )


    plt.tight_layout()


    plt.savefig(

        ANALYSIS_DIR / filename,

        dpi=400,

        bbox_inches="tight",
    )


    plt.close()


# ============================================================
# EXACT TRUE -> PREDICTED SET PATTERNS
# ============================================================
#
# This is one of the most important analyses.
#
# Example:
#
# TRUE = IN, OP
# PRED = IN, NA
#
# gets one row:
#
# IN, OP -> IN, NA : 37 documents
#
# ============================================================

def create_exact_pattern_table(
    true_labels,
    predicted_labels,
):

    counter = Counter()


    for true_row, pred_row in zip(
        true_labels,
        predicted_labels,
    ):

        true_string = label_string(
            true_row
        )

        pred_string = label_string(
            pred_row
        )


        counter[
            (
                true_string,
                pred_string,
            )
        ] += 1


    rows = []


    for (
        (
            true_string,
            pred_string,
        ),
        count,
    ) in counter.most_common():

        rows.append({

            "true_labels":
                true_string,

            "predicted_labels":
                pred_string,

            "count":
                count,

        })


    return pd.DataFrame(rows)


# ============================================================
# HIERARCHY ERROR ANALYSIS
# ============================================================

def create_hierarchy_error_table(
    true_labels,
    predicted_labels,
):

    rows = []

    for i in range(
        len(true_labels)
    ):

        true_row = true_labels[i]

        pred_row = predicted_labels[i]

        for child, parent in child_to_parent.items():

            child_idx = hierarchical_labels.index(
                child
            )

            parent_idx = hierarchical_labels.index(
                parent
            )

            child_true = (
                true_row[child_idx] == 1
            )

            parent_true = (
                true_row[parent_idx] == 1
            )

            child_pred = (
                pred_row[child_idx] == 1
            )

            parent_pred = (
                pred_row[parent_idx] == 1
            )


            # ------------------------------------------------
            # ERROR TYPE 1:
            #
            # Child is truly present and correctly predicted,
            # but its required parent is missed.
            #
            # TRUE:
            #     child  = 1
            #     parent = 1
            #
            # PRED:
            #     child  = 1
            #     parent = 0
            # ------------------------------------------------

            if (
                child_true
                and
                parent_true
                and
                child_pred
                and
                not parent_pred
            ):

                rows.append({

                    "document_index":
                        i,

                    "child":
                        child,

                    "parent":
                        parent,

                    "error_type":
                        "child_correct_parent_missed",

                    "true_child":
                        1,

                    "true_parent":
                        1,

                    "pred_child":
                        1,

                    "pred_parent":
                        0,

                })


            # ------------------------------------------------
            # ERROR TYPE 2:
            #
            # Model predicts a child without predicting its
            # required parent.
            #
            # This is a prediction-time hierarchy violation.
            #
            # It can happen whether or not the child is truly
            # present in the gold annotation.
            #
            # PRED:
            #     child  = 1
            #     parent = 0
            # ------------------------------------------------

            if (
                child_pred
                and
                not parent_pred
            ):

                rows.append({

                    "document_index":
                        i,

                    "child":
                        child,

                    "parent":
                        parent,

                    "error_type":
                        "hierarchy_violation",

                    "true_child":
                        int(child_true),

                    "true_parent":
                        int(parent_true),

                    "pred_child":
                        1,

                    "pred_parent":
                        0,

                })


            # ------------------------------------------------
            # ERROR TYPE 3:
            #
            # A TRUE child was missed while its parent was
            # correctly predicted.
            #
            # TRUE:
            #     child  = 1
            #     parent = 1
            #
            # PRED:
            #     child  = 0
            #     parent = 1
            # ------------------------------------------------

            if (
                child_true
                and
                parent_true
                and
                parent_pred
                and
                not child_pred
            ):

                rows.append({

                    "document_index":
                        i,

                    "child":
                        child,

                    "parent":
                        parent,

                    "error_type":
                        "parent_predicted_child_not_predicted",

                    "true_child":
                        1,

                    "true_parent":
                        1,

                    "pred_child":
                        0,

                    "pred_parent":
                        1,

                })


    return pd.DataFrame(
        rows
    )



# ============================================================
# HIERARCHICAL ERROR COUNTS
# ============================================================

def hierarchy_summary(
    true_labels,
    predicted_labels,
):

    total = len(
        true_labels
    )


    prediction_violations = 0

    child_correct_parent_missed = 0

    true_hierarchy_violations = 0


    violation_pairs = Counter()

    child_parent_missed_pairs = Counter()


    for i in range(total):

        true_row = true_labels[i]

        pred_row = predicted_labels[i]


        true_violations = (
            get_true_hierarchy_violations(
                true_row
            )
        )


        pred_violations = (
            get_hierarchy_violations(
                pred_row
            )
        )


        if true_violations:

            true_hierarchy_violations += 1


        if pred_violations:

            prediction_violations += 1


            for pair in pred_violations:

                violation_pairs[pair] += 1


        for child, parent in child_to_parent.items():

            child_idx = hierarchical_labels.index(
                child
            )

            parent_idx = hierarchical_labels.index(
                parent
            )


            if (

                true_row[child_idx] == 1

                and

                true_row[parent_idx] == 1

                and

                pred_row[child_idx] == 1

                and

                pred_row[parent_idx] == 0

            ):

                child_correct_parent_missed += 1

                child_parent_missed_pairs[
                    f"{child}->{parent}"
                ] += 1


    return {

        "total_documents":
            total,

        "documents_with_predicted_hierarchy_violation":
            prediction_violations,

        "percent_documents_with_predicted_hierarchy_violation":
            (
                prediction_violations / total * 100
                if total > 0
                else 0
            ),

        "documents_with_true_hierarchy_violation":
            true_hierarchy_violations,

        "child_correct_parent_missed_events":
            child_correct_parent_missed,

        "predicted_hierarchy_violation_pairs":
            dict(violation_pairs),

        "child_correct_parent_missed_pairs":
            dict(child_parent_missed_pairs),
    }


# ============================================================
# DOCUMENT-LEVEL SUMMARY
# ============================================================

def create_document_summary(
    true_labels,
    predicted_labels,
):

    total = len(
        true_labels
    )


    exact = 0

    partial = 0

    completely_wrong = 0

    nothing = 0

    any_error = 0

    hierarchy_violation = 0


    for i in range(total):

        true_set = set(
            label_set(
                true_labels[i]
            )
        )

        pred_set = set(
            label_set(
                predicted_labels[i]
            )
        )


        if true_set == pred_set:

            exact += 1

        else:

            any_error += 1


        if (
            true_set
            and
            pred_set
            and
            true_set & pred_set
            and
            true_set != pred_set
        ):

            partial += 1


        if (
            true_set
            and
            pred_set
            and
            not (true_set & pred_set)
        ):

            completely_wrong += 1


        if (
            true_set
            and
            not pred_set
        ):

            nothing += 1

        hierarchy_violations = get_hierarchy_violations(
            predicted_labels[i]
        )

        if hierarchy_violations:
            hierarchy_violation += 1



    return pd.DataFrame({

        "category": [

            "exact_match",

            "any_error",

            "partially_correct",

            "completely_wrong",

            "nothing_predicted",

             "hierarchy_violation",

        ],

        "count": [

            exact,

            any_error,

            partial,

            completely_wrong,

            nothing,

             hierarchy_violation,

        ],

        "percentage": [

            exact / total * 100,

            any_error / total * 100,

            partial / total * 100,

            completely_wrong / total * 100,

            nothing / total * 100,

            hierarchy_violation / total * 100,

        ],

    })


# ============================================================
# RUN ANALYSIS
# ============================================================

def run_analysis(
    true_labels,
    predicted_labels,
    probabilities,
    prefix,
    description,
):

    print("\n")
    print("=" * 100)
    print(description)
    print("=" * 100)


    # ========================================================
    # 1. DOCUMENT-LEVEL ERROR TABLE
    # ========================================================

    document_df = create_document_error_table(

        true_labels,

        predicted_labels,

        probabilities,

    )


    document_df.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_document_errors.csv",

        index=False,

        encoding="utf-8-sig",
    )


    # --------------------------------------------------------
    # Save only documents containing errors
    # --------------------------------------------------------

    error_documents = document_df[
        ~document_df["exact_match"]
    ]


    error_documents.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_error_documents_only.csv",

        index=False,

        encoding="utf-8-sig",
    )


    # ========================================================
    # 2. EXACT LABEL-SET PATTERNS
    # ========================================================

    pattern_df = create_exact_pattern_table(

        true_labels,

        predicted_labels,

    )


    pattern_df.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_exact_label_set_patterns.csv",

        index=False,

        encoding="utf-8-sig",
    )


    # ========================================================
    # 3. PER-LABEL SUMMARY
    # ========================================================

    label_df = create_per_label_summary(

        true_labels,

        predicted_labels,

    )


    label_df.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_per_label_summary.csv",

        index=False,

        encoding="utf-8-sig",
    )


    # ========================================================
    # 4. MISSED -> FALSE POSITIVE MATRIX
    # ========================================================

    missed_fp_matrix = create_missed_to_fp_matrix(

        true_labels,

        predicted_labels,

    )


    save_matrix(

        missed_fp_matrix,

        f"{prefix}_missed_to_false_positive.csv",

    )


    plot_matrix(

        missed_fp_matrix,

        f"{prefix}_missed_to_false_positive.png",

        (
            "Missed True Label → False-Positive Label\n"
            + description
        ),

        "False-Positive Label",

        "Missed True Label",

    )


    # ========================================================
    # 5. HIERARCHY ERRORS
    # ========================================================

    hierarchy_df = create_hierarchy_error_table(

        true_labels,

        predicted_labels,

    )


    hierarchy_df.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_hierarchy_errors.csv",

        index=False,

        encoding="utf-8-sig",
    )


    # ========================================================
    # 6. HIERARCHY SUMMARY
    # ========================================================

    hierarchy_stats = hierarchy_summary(

        true_labels,

        predicted_labels,

    )


    with open(

        ANALYSIS_DIR
        / f"{prefix}_hierarchy_summary.json",

        "w",

        encoding="utf-8",

    ) as f:

        json.dump(

            hierarchy_stats,

            f,

            indent=2,

        )


    # ========================================================
    # 7. DOCUMENT SUMMARY
    # ========================================================

    doc_summary = create_document_summary(

        true_labels,

        predicted_labels,

    )


    doc_summary.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_document_summary.csv",

        index=False,

        encoding="utf-8-sig",
    )


    # ========================================================
    # 8. PRINT SUMMARY
    # ========================================================

    print("\nDocument-level summary:")

    print(
        doc_summary.to_string(
            index=False
        )
    )


    print("\nHierarchy summary:")

    print(
        json.dumps(
            hierarchy_stats,
            indent=2,
        )
    )


    # ========================================================
    # 9. TOP ERROR PATTERNS
    # ========================================================

    print("\nTop 20 true -> predicted label-set patterns:")

    print(

        pattern_df.head(20).to_string(
            index=False
        )

    )


    print("\nTop 20 missed -> false-positive pairs:")

    pair_rows = []


    for i, true_label in enumerate(
        hierarchical_labels
    ):

        for j, predicted_label in enumerate(
            hierarchical_labels
        ):

            count = missed_fp_matrix[i, j]


            if count > 0:

                pair_rows.append({

                    "missed_label":
                        true_label,

                    "false_positive_label":
                        predicted_label,

                    "count":
                        int(count),

                })


    pair_df = pd.DataFrame(
        pair_rows
    )


    if len(pair_df) > 0:

        pair_df = pair_df.sort_values(
            "count",
            ascending=False,
        )


        print(

            pair_df.head(20).to_string(
                index=False
            )

        )


    # ========================================================
    # 10. TOP HIERARCHY VIOLATIONS
    # ========================================================

    print("\nHierarchy violation examples:")

    if len(hierarchy_df) > 0:

        print(

            hierarchy_df.head(20).to_string(
                index=False
            )

        )

    else:

        print(
            "No hierarchy violations found."
        )


# ============================================================
# RUN 1:
# ORIGINAL 0.50
# ============================================================

run_analysis(

    true_labels=test_labels,

    predicted_labels=predictions_05,

    probabilities=test_probabilities,

    prefix="threshold_0.50",

    description=(
        "Threshold = 0.50"
    ),

)


# ============================================================
# RUN 2:
# GLOBAL TUNED
# ============================================================

run_analysis(

    true_labels=test_labels,

    predicted_labels=predictions_global,

    probabilities=test_probabilities,

    prefix="global_tuned",

    description=(
        f"Global threshold = "
        f"{GLOBAL_TUNED_THRESHOLD:.3f}"
    ),

)


# ============================================================
# RUN 3:
# PER-LABEL TUNED
# ============================================================

run_analysis(

    true_labels=test_labels,

    predicted_labels=predictions_per_label,

    probabilities=test_probabilities,

    prefix="per_label_tuned",

    description=(
        "Per-label tuned thresholds"
    ),

)


# ============================================================
# SAVE THRESHOLDS
# ============================================================

threshold_df = pd.DataFrame({

    "label":
        hierarchical_labels,

    "threshold":
        threshold_array,

})


threshold_df.to_csv(

    ANALYSIS_DIR
    / "thresholds_used.csv",

    index=False,

    encoding="utf-8-sig",
)


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 100)
print("HIERARCHICAL ERROR ANALYSIS COMPLETE")
print("=" * 100)

print("\nResults saved to:")

print(
    ANALYSIS_DIR
)

print("\nThe most important files are:")

print(
    "  *_document_errors.csv"
)

print(
    "  *_error_documents_only.csv"
)

print(
    "  *_exact_label_set_patterns.csv"
)

print(
    "  *_per_label_summary.csv"
)

print(
    "  *_missed_to_false_positive.csv"
)

print(
    "  *_hierarchy_errors.csv"
)

print(
    "  *_hierarchy_summary.json"
)

print(
    "  *_document_summary.csv"
)

print("=" * 100)



LOADING TEST DATA
Logits shape: (633, 25)
Labels shape: (633, 25)
Labels: ['MT', 'LY', 'SP', 'ID', 'NA', 'HI', 'IN', 'OP', 'IP', 'it', 'ne', 'sr', 'nb', 're', 'en', 'ra', 'dtp', 'fi', 'lt', 'rv', 'ob', 'rs', 'av', 'ds', 'ed']

Parents:
['MT', 'LY', 'SP', 'ID', 'NA', 'HI', 'IN', 'OP', 'IP']

Children:
['it', 'ne', 'sr', 'nb', 're', 'en', 'ra', 'dtp', 'fi', 'lt', 'rv', 'ob', 'rs', 'av', 'ds', 'ed']


Threshold = 0.50

Document-level summary:
           category  count  percentage
        exact_match    277   43.759874
          any_error    356   56.240126
  partially_correct    302   47.709321
   completely_wrong     39    6.161137
  nothing_predicted     15    2.369668
hierarchy_violation     12    1.895735

Hierarchy summary:
{
  "total_documents": 633,
  "documents_with_predicted_hierarchy_violation": 12,
  "percent_documents_with_predicted_hierarchy_violation": 1.8957345971563981,
  "documents_with_true_hierarchy_violation": 0,
  "child_correct_parent_missed_events": 8,
  "predicte

# Parent Shared Expert

## New approach codes

In [5]:
# ============================================================
# HIERARCHICAL MULTILABEL ERROR ANALYSIS
#
# Designed for:
#
#   9 parent labels
#   16 child labels
#   25 total labels
#
# The analysis explicitly distinguishes:
#
#   1. Correct labels
#   2. False positives
#   3. False negatives
#   4. Missed-label -> false-positive associations
#   5. Completely missed documents
#   6. Partially correct documents
#   7. Exact label-set matches
#   8. Hierarchical violations
#   9. Child predicted but parent missing
#  10. Parent predicted without corresponding child
#
# ============================================================


# ============================================================
# IMPORTS
# ============================================================

import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    multilabel_confusion_matrix,
)


# ============================================================
# CONFIGURATION
# ============================================================

OUTPUT_DIR = Path(
    r"C:\Users\alrazz\Downloads\Parent Shared Expert"
)


LOGITS_FILE = (
    OUTPUT_DIR / "test_logits.npy"
)

LABELS_FILE = (
    OUTPUT_DIR / "test_labels.npy"
)

LABEL_NAMES_FILE = (
    OUTPUT_DIR / "hierarchical_labels.json"
)


ANALYSIS_DIR = (
    OUTPUT_DIR / "hierarchical_error_analysis"
)

ANALYSIS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# THRESHOLD CONFIGURATION
# ============================================================

GLOBAL_TUNED_THRESHOLD = 0.24


PER_LABEL_THRESHOLDS = {

    "MT":  0.07,
    "LY":  0.56,
    "SP": 0.26,
    "ID":  0.41,
    "NA":  0.44,
    "HI":  0.21,
    "IN":  0.44,
    "OP":  0.10,
    "IP":  	0.31,

    "it":  0.36,
    "ne":  0.49,
    "sr":   0.28,
    "nb":  0.25,
    "re":  0.38,

    "en":  0.24,
    "ra":  0.14,
    "dtp": 0.25,
    "fi":  0.22,
    "lt":  0.34,

    "rv":  0.33,
    "ob":  0.29,
    "rs":  0.24,
    "av":  0.26,

    "ds":  0.57,
    "ed":  0.41,
}


# ============================================================
# LOAD DATA
# ============================================================

print("\n" + "=" * 100)
print("LOADING TEST DATA")
print("=" * 100)


test_logits = np.load(
    LOGITS_FILE
)

test_labels = np.load(
    LABELS_FILE
)


with open(
    LABEL_NAMES_FILE,
    "r",
    encoding="utf-8",
) as f:

    hierarchical_labels = json.load(f)


print(
    "Logits shape:",
    test_logits.shape,
)

print(
    "Labels shape:",
    test_labels.shape,
)

print(
    "Labels:",
    hierarchical_labels,
)


# ============================================================
# BASIC CHECKS
# ============================================================

if test_logits.shape != test_labels.shape:

    raise ValueError(
        "Logits and labels have different shapes."
    )


NUM_LABELS = len(
    hierarchical_labels
)


if test_logits.shape[1] != NUM_LABELS:

    raise ValueError(
        "Number of logits does not match number of labels."
    )


# ============================================================
# HIERARCHICAL STRUCTURE
# ============================================================

NUM_PARENTS = 9


parents = hierarchical_labels[
    :NUM_PARENTS
]

children = hierarchical_labels[
    NUM_PARENTS:
]


parent_indices = list(
    range(NUM_PARENTS)
)

child_indices = list(
    range(NUM_PARENTS, NUM_LABELS)
)


# ------------------------------------------------------------
# Explicit child -> parent mapping
# ------------------------------------------------------------

child_to_parent = {

    "it": "SP",

    "ne": "NA",
    "sr": "NA",
    "nb": "NA",

    "re": "HI",

    "en": "IN",
    "ra": "IN",
    "dtp": "IN",
    "fi": "IN",
    "lt": "IN",

    "rv": "OP",
    "ob": "OP",
    "rs": "OP",
    "av": "OP",

    "ds": "IP",
    "ed": "IP",
}


print("\nParents:")
print(parents)

print("\nChildren:")
print(children)


# ============================================================
# VERIFY CHILD -> PARENT MAPPING
# ============================================================

for child in children:

    if child not in child_to_parent:

        raise ValueError(
            f"No parent defined for child: {child}"
        )

    parent = child_to_parent[child]

    if parent not in parents:

        raise ValueError(
            f"Unknown parent {parent} for child {child}"
        )


# ============================================================
# SIGMOID
# ============================================================

def sigmoid(x):

    # Numerically safer implementation
    x = np.clip(
        x,
        -50,
        50,
    )

    return 1.0 / (
        1.0 + np.exp(-x)
    )


test_probabilities = sigmoid(
    test_logits
)


# ============================================================
# THRESHOLDS
# ============================================================

missing_thresholds = [

    label

    for label in hierarchical_labels

    if label not in PER_LABEL_THRESHOLDS
]


if missing_thresholds:

    raise ValueError(
        "Missing thresholds for: "
        f"{missing_thresholds}"
    )


threshold_array = np.array(

    [
        PER_LABEL_THRESHOLDS[label]
        for label in hierarchical_labels
    ],

    dtype=np.float32,
)


# ============================================================
# PREDICTIONS
# ============================================================

def make_predictions(
    probabilities,
    thresholds,
):

    thresholds = np.asarray(
        thresholds,
        dtype=np.float32,
    )


    if thresholds.ndim == 0:

        thresholds = np.repeat(
            thresholds,
            probabilities.shape[1],
        )


    return (
        probabilities
        >= thresholds.reshape(1, -1)
    ).astype(np.int32)


predictions_05 = make_predictions(
    test_probabilities,
    0.50,
)


predictions_global = make_predictions(
    test_probabilities,
    GLOBAL_TUNED_THRESHOLD,
)


predictions_per_label = make_predictions(
    test_probabilities,
    threshold_array,
)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def label_set(
    row,
    labels=hierarchical_labels,
):

    return [
        labels[i]
        for i in np.where(
            row == 1
        )[0]
    ]


def label_string(
    row,
    labels=hierarchical_labels,
):

    values = label_set(
        row,
        labels,
    )

    if not values:

        return "NONE"

    return ", ".join(values)


def set_difference_string(
    true_row,
    pred_row,
):

    true_set = set(
        label_set(true_row)
    )

    pred_set = set(
        label_set(pred_row)
    )


    missed = sorted(
        true_set - pred_set
    )

    false_positive = sorted(
        pred_set - true_set
    )


    return (
        ", ".join(missed)
        if missed
        else "NONE"
    ), (
        ", ".join(false_positive)
        if false_positive
        else "NONE"
    )


# ============================================================
# HIERARCHICAL VIOLATIONS
# ============================================================
#
# A hierarchy violation occurs when:
#
# child = 1
# parent = 0
#
# Example:
#
# fi = 1
# IN = 0
#
# This is an invalid hierarchical prediction.
# ============================================================

def get_hierarchy_violations(
    prediction_row,
):

    violations = []


    for child, parent in child_to_parent.items():

        child_idx = hierarchical_labels.index(
            child
        )

        parent_idx = hierarchical_labels.index(
            parent
        )


        if (
            prediction_row[child_idx] == 1
            and
            prediction_row[parent_idx] == 0
        ):

            violations.append(
                f"{child}->{parent}"
            )


    return violations


# ============================================================
# TRUE HIERARCHY CHECK
# ============================================================
#
# This checks whether the GOLD labels themselves are
# hierarchically consistent.
#
# Normally all violations should be zero because your
# make_hierarchical_labels() function explicitly propagates
# child labels to parents.
# ============================================================

def get_true_hierarchy_violations(
    true_row,
):

    violations = []


    for child, parent in child_to_parent.items():

        child_idx = hierarchical_labels.index(
            child
        )

        parent_idx = hierarchical_labels.index(
            parent
        )


        if (
            true_row[child_idx] == 1
            and
            true_row[parent_idx] == 0
        ):

            violations.append(
                f"{child}->{parent}"
            )


    return violations


# ============================================================
# PER-DOCUMENT ERROR ANALYSIS
# ============================================================

def create_document_error_table(
    true_labels,
    predicted_labels,
    probabilities,
):

    rows = []


    for i in range(
        len(true_labels)
    ):

        true_row = true_labels[i]

        pred_row = predicted_labels[i]

        prob_row = probabilities[i]


        true_set = set(
            label_set(true_row)
        )

        pred_set = set(
            label_set(pred_row)
        )


        correct = sorted(
            true_set & pred_set
        )

        missed = sorted(
            true_set - pred_set
        )

        false_positive = sorted(
            pred_set - true_set
        )


        true_count = len(
            true_set
        )

        predicted_count = len(
            pred_set
        )

        correct_count = len(
            correct
        )

        missed_count = len(
            missed
        )

        false_positive_count = len(
            false_positive
        )


        # ----------------------------------------------------
        # Exact set match
        # ----------------------------------------------------

        exact_match = (
            true_set == pred_set
        )


        # ----------------------------------------------------
        # Completely missed
        # ----------------------------------------------------

        nothing_predicted = (
            predicted_count == 0
        )


        # ----------------------------------------------------
        # No true labels
        # ----------------------------------------------------

        no_true_labels = (
            true_count == 0
        )


        # ----------------------------------------------------
        # Completely wrong
        #
        # There is no overlap between true and predicted
        # labels, while both contain at least one label.
        # ----------------------------------------------------

        completely_wrong = (

            true_count > 0

            and

            predicted_count > 0

            and

            correct_count == 0
        )


        # ----------------------------------------------------
        # Partially correct
        # ----------------------------------------------------

        partially_correct = (

            correct_count > 0

            and

            not exact_match
        )


        # ----------------------------------------------------
        # Hierarchical violations
        # ----------------------------------------------------

        hierarchy_violations = (
            get_hierarchy_violations(
                pred_row
            )
        )


        true_hierarchy_violations = (
            get_true_hierarchy_violations(
                true_row
            )
        )


        # ----------------------------------------------------
        # Probability of true labels
        # ----------------------------------------------------

        true_probabilities = [

            float(
                prob_row[
                    hierarchical_labels.index(label)
                ]
            )

            for label in sorted(true_set)

        ]


        # ----------------------------------------------------
        # Probability of missed labels
        # ----------------------------------------------------

        missed_probabilities = [

            float(
                prob_row[
                    hierarchical_labels.index(label)
                ]
            )

            for label in missed

        ]


        # ----------------------------------------------------
        # Probability of false positives
        # ----------------------------------------------------

        false_positive_probabilities = [

            float(
                prob_row[
                    hierarchical_labels.index(label)
                ]
            )

            for label in false_positive

        ]


        rows.append({

            "document_index": i,

            "true_labels":
                ", ".join(
                    sorted(true_set)
                )
                if true_set
                else "NONE",

            "predicted_labels":
                ", ".join(
                    sorted(pred_set)
                )
                if pred_set
                else "NONE",

            "correct_labels":
                ", ".join(correct)
                if correct
                else "NONE",

            "missed_labels":
                ", ".join(missed)
                if missed
                else "NONE",

            "false_positive_labels":
                ", ".join(false_positive)
                if false_positive
                else "NONE",

            "true_label_count":
                true_count,

            "predicted_label_count":
                predicted_count,

            "correct_label_count":
                correct_count,

            "missed_label_count":
                missed_count,

            "false_positive_count":
                false_positive_count,

            "exact_match":
                exact_match,

            "partially_correct":
                partially_correct,

            "completely_wrong":
                completely_wrong,

            "nothing_predicted":
                nothing_predicted,

            "hierarchy_violation":
                len(
                    hierarchy_violations
                ) > 0,

            "hierarchy_violations":
                "; ".join(
                    hierarchy_violations
                )
                if hierarchy_violations
                else "NONE",

            "true_hierarchy_violation":
                len(
                    true_hierarchy_violations
                ) > 0,

            "true_hierarchy_violations":
                "; ".join(
                    true_hierarchy_violations
                )
                if true_hierarchy_violations
                else "NONE",

            "true_probabilities":
                "; ".join(
                    f"{p:.4f}"
                    for p in true_probabilities
                ),

            "missed_probabilities":
                "; ".join(
                    f"{p:.4f}"
                    for p in missed_probabilities
                ),

            "false_positive_probabilities":
                "; ".join(
                    f"{p:.4f}"
                    for p in false_positive_probabilities
                ),
        })


    return pd.DataFrame(rows)


# ============================================================
# PER-LABEL TP / FP / FN / TN
# ============================================================

def create_per_label_summary(
    true_labels,
    predicted_labels,
):

    mcm = multilabel_confusion_matrix(
        true_labels,
        predicted_labels,
    )


    rows = []


    for i, label in enumerate(
        hierarchical_labels
    ):

        tn = mcm[i, 0, 0]

        fp = mcm[i, 0, 1]

        fn = mcm[i, 1, 0]

        tp = mcm[i, 1, 1]


        precision = (
            tp / (tp + fp)
            if (tp + fp) > 0
            else 0.0
        )


        recall = (
            tp / (tp + fn)
            if (tp + fn) > 0
            else 0.0
        )


        f1 = (

            2 * precision * recall
            /
            (precision + recall)

            if (precision + recall) > 0

            else 0.0
        )


        rows.append({

            "label": label,

            "TP": int(tp),

            "FP": int(fp),

            "FN": int(fn),

            "TN": int(tn),

            "precision":
                precision,

            "recall":
                recall,

            "F1":
                f1,

            "support":
                int(tp + fn),

        })


    return pd.DataFrame(rows)


# ============================================================
# MISSED -> FALSE POSITIVE ASSOCIATION
# ============================================================
#
# Example:
#
# TRUE:
#     IN OP
#
# PRED:
#     IN NA
#
# Result:
#
#     OP -> NA
#
# Important:
#
# This means:
#
# "When OP was missed, NA was falsely predicted in the
# same document."
#
# It does NOT mathematically prove that NA replaced OP.
# ============================================================

def create_missed_to_fp_matrix(
    true_labels,
    predicted_labels,
):

    n_labels = len(
        hierarchical_labels
    )


    matrix = np.zeros(
        (n_labels, n_labels),
        dtype=np.int64,
    )


    for i in range(
        len(true_labels)
    ):

        true_row = true_labels[i]

        pred_row = predicted_labels[i]


        missed = np.where(

            (true_row == 1)

            &

            (pred_row == 0)

        )[0]


        false_positive = np.where(

            (true_row == 0)

            &

            (pred_row == 1)

        )[0]


        for missed_idx in missed:

            for fp_idx in false_positive:

                matrix[
                    missed_idx,
                    fp_idx
                ] += 1


    return matrix


# ============================================================
# SAVE MATRIX
# ============================================================

def save_matrix(
    matrix,
    filename,
):

    df = pd.DataFrame(

        matrix,

        index=hierarchical_labels,

        columns=hierarchical_labels,
    )


    df.to_csv(

        ANALYSIS_DIR / filename,

        encoding="utf-8-sig",
    )


    return df


# ============================================================
# PLOT MATRIX
# ============================================================

def plot_matrix(
    matrix,
    filename,
    title,
    xlabel,
    ylabel,
):

    plt.figure(
        figsize=(20, 17)
    )


    sns.heatmap(

        matrix,

        annot=True,

        fmt="d",

        cmap="Blues",

        xticklabels=hierarchical_labels,

        yticklabels=hierarchical_labels,

        linewidths=0.4,

    )


    plt.xlabel(
        xlabel,
        fontsize=13,
        fontweight="bold",
    )


    plt.ylabel(
        ylabel,
        fontsize=13,
        fontweight="bold",
    )


    plt.title(
        title,
        fontsize=16,
        fontweight="bold",
    )


    plt.xticks(
        rotation=45,
        ha="right",
    )


    plt.yticks(
        rotation=0,
    )


    plt.tight_layout()


    plt.savefig(

        ANALYSIS_DIR / filename,

        dpi=400,

        bbox_inches="tight",
    )


    plt.close()


# ============================================================
# EXACT TRUE -> PREDICTED SET PATTERNS
# ============================================================
#
# This is one of the most important analyses.
#
# Example:
#
# TRUE = IN, OP
# PRED = IN, NA
#
# gets one row:
#
# IN, OP -> IN, NA : 37 documents
#
# ============================================================

def create_exact_pattern_table(
    true_labels,
    predicted_labels,
):

    counter = Counter()


    for true_row, pred_row in zip(
        true_labels,
        predicted_labels,
    ):

        true_string = label_string(
            true_row
        )

        pred_string = label_string(
            pred_row
        )


        counter[
            (
                true_string,
                pred_string,
            )
        ] += 1


    rows = []


    for (
        (
            true_string,
            pred_string,
        ),
        count,
    ) in counter.most_common():

        rows.append({

            "true_labels":
                true_string,

            "predicted_labels":
                pred_string,

            "count":
                count,

        })


    return pd.DataFrame(rows)


# ============================================================
# HIERARCHY ERROR ANALYSIS
# ============================================================

def create_hierarchy_error_table(
    true_labels,
    predicted_labels,
):

    rows = []

    for i in range(
        len(true_labels)
    ):

        true_row = true_labels[i]

        pred_row = predicted_labels[i]

        for child, parent in child_to_parent.items():

            child_idx = hierarchical_labels.index(
                child
            )

            parent_idx = hierarchical_labels.index(
                parent
            )

            child_true = (
                true_row[child_idx] == 1
            )

            parent_true = (
                true_row[parent_idx] == 1
            )

            child_pred = (
                pred_row[child_idx] == 1
            )

            parent_pred = (
                pred_row[parent_idx] == 1
            )


            # ------------------------------------------------
            # ERROR TYPE 1:
            #
            # Child is truly present and correctly predicted,
            # but its required parent is missed.
            #
            # TRUE:
            #     child  = 1
            #     parent = 1
            #
            # PRED:
            #     child  = 1
            #     parent = 0
            # ------------------------------------------------

            if (
                child_true
                and
                parent_true
                and
                child_pred
                and
                not parent_pred
            ):

                rows.append({

                    "document_index":
                        i,

                    "child":
                        child,

                    "parent":
                        parent,

                    "error_type":
                        "child_correct_parent_missed",

                    "true_child":
                        1,

                    "true_parent":
                        1,

                    "pred_child":
                        1,

                    "pred_parent":
                        0,

                })


            # ------------------------------------------------
            # ERROR TYPE 2:
            #
            # Model predicts a child without predicting its
            # required parent.
            #
            # This is a prediction-time hierarchy violation.
            #
            # It can happen whether or not the child is truly
            # present in the gold annotation.
            #
            # PRED:
            #     child  = 1
            #     parent = 0
            # ------------------------------------------------

            if (
                child_pred
                and
                not parent_pred
            ):

                rows.append({

                    "document_index":
                        i,

                    "child":
                        child,

                    "parent":
                        parent,

                    "error_type":
                        "hierarchy_violation",

                    "true_child":
                        int(child_true),

                    "true_parent":
                        int(parent_true),

                    "pred_child":
                        1,

                    "pred_parent":
                        0,

                })


            # ------------------------------------------------
            # ERROR TYPE 3:
            #
            # A TRUE child was missed while its parent was
            # correctly predicted.
            #
            # TRUE:
            #     child  = 1
            #     parent = 1
            #
            # PRED:
            #     child  = 0
            #     parent = 1
            # ------------------------------------------------

            if (
                child_true
                and
                parent_true
                and
                parent_pred
                and
                not child_pred
            ):

                rows.append({

                    "document_index":
                        i,

                    "child":
                        child,

                    "parent":
                        parent,

                    "error_type":
                        "parent_predicted_child_not_predicted",

                    "true_child":
                        1,

                    "true_parent":
                        1,

                    "pred_child":
                        0,

                    "pred_parent":
                        1,

                })


    return pd.DataFrame(
        rows
    )


# ============================================================
# HIERARCHICAL ERROR COUNTS
# ============================================================

def hierarchy_summary(
    true_labels,
    predicted_labels,
):

    total = len(
        true_labels
    )


    prediction_violations = 0

    child_correct_parent_missed = 0

    true_hierarchy_violations = 0


    violation_pairs = Counter()

    child_parent_missed_pairs = Counter()


    for i in range(total):

        true_row = true_labels[i]

        pred_row = predicted_labels[i]


        true_violations = (
            get_true_hierarchy_violations(
                true_row
            )
        )


        pred_violations = (
            get_hierarchy_violations(
                pred_row
            )
        )


        if true_violations:

            true_hierarchy_violations += 1


        if pred_violations:

            prediction_violations += 1


            for pair in pred_violations:

                violation_pairs[pair] += 1


        for child, parent in child_to_parent.items():

            child_idx = hierarchical_labels.index(
                child
            )

            parent_idx = hierarchical_labels.index(
                parent
            )


            if (

                true_row[child_idx] == 1

                and

                true_row[parent_idx] == 1

                and

                pred_row[child_idx] == 1

                and

                pred_row[parent_idx] == 0

            ):

                child_correct_parent_missed += 1

                child_parent_missed_pairs[
                    f"{child}->{parent}"
                ] += 1


    return {

        "total_documents":
            total,

        "documents_with_predicted_hierarchy_violation":
            prediction_violations,

        "percent_documents_with_predicted_hierarchy_violation":
            (
                prediction_violations / total * 100
                if total > 0
                else 0
            ),

        "documents_with_true_hierarchy_violation":
            true_hierarchy_violations,

        "child_correct_parent_missed_events":
            child_correct_parent_missed,

        "predicted_hierarchy_violation_pairs":
            dict(violation_pairs),

        "child_correct_parent_missed_pairs":
            dict(child_parent_missed_pairs),
    }


# ============================================================
# DOCUMENT-LEVEL SUMMARY
# ============================================================

def create_document_summary(
    true_labels,
    predicted_labels,
):

    total = len(
        true_labels
    )


    exact = 0

    partial = 0

    completely_wrong = 0

    nothing = 0

    any_error = 0

    hierarchy_violation = 0


    for i in range(total):

        true_set = set(
            label_set(
                true_labels[i]
            )
        )

        pred_set = set(
            label_set(
                predicted_labels[i]
            )
        )


        if true_set == pred_set:

            exact += 1

        else:

            any_error += 1


        if (
            true_set
            and
            pred_set
            and
            true_set & pred_set
            and
            true_set != pred_set
        ):

            partial += 1


        if (
            true_set
            and
            pred_set
            and
            not (true_set & pred_set)
        ):

            completely_wrong += 1


        if (
            true_set
            and
            not pred_set
        ):

            nothing += 1

        hierarchy_violations = get_hierarchy_violations(
            predicted_labels[i]
        )

        if hierarchy_violations:
            hierarchy_violation += 1



    return pd.DataFrame({

        "category": [

            "exact_match",

            "any_error",

            "partially_correct",

            "completely_wrong",

            "nothing_predicted",

             "hierarchy_violation",

        ],

        "count": [

            exact,

            any_error,

            partial,

            completely_wrong,

            nothing,

             hierarchy_violation,

        ],

        "percentage": [

            exact / total * 100,

            any_error / total * 100,

            partial / total * 100,

            completely_wrong / total * 100,

            nothing / total * 100,

            hierarchy_violation / total * 100,

        ],

    })


# ============================================================
# RUN ANALYSIS
# ============================================================

def run_analysis(
    true_labels,
    predicted_labels,
    probabilities,
    prefix,
    description,
):

    print("\n")
    print("=" * 100)
    print(description)
    print("=" * 100)


    # ========================================================
    # 1. DOCUMENT-LEVEL ERROR TABLE
    # ========================================================

    document_df = create_document_error_table(

        true_labels,

        predicted_labels,

        probabilities,

    )


    document_df.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_document_errors.csv",

        index=False,

        encoding="utf-8-sig",
    )


    # --------------------------------------------------------
    # Save only documents containing errors
    # --------------------------------------------------------

    error_documents = document_df[
        ~document_df["exact_match"]
    ]


    error_documents.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_error_documents_only.csv",

        index=False,

        encoding="utf-8-sig",
    )


    # ========================================================
    # 2. EXACT LABEL-SET PATTERNS
    # ========================================================

    pattern_df = create_exact_pattern_table(

        true_labels,

        predicted_labels,

    )


    pattern_df.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_exact_label_set_patterns.csv",

        index=False,

        encoding="utf-8-sig",
    )


    # ========================================================
    # 3. PER-LABEL SUMMARY
    # ========================================================

    label_df = create_per_label_summary(

        true_labels,

        predicted_labels,

    )


    label_df.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_per_label_summary.csv",

        index=False,

        encoding="utf-8-sig",
    )


    # ========================================================
    # 4. MISSED -> FALSE POSITIVE MATRIX
    # ========================================================

    missed_fp_matrix = create_missed_to_fp_matrix(

        true_labels,

        predicted_labels,

    )


    save_matrix(

        missed_fp_matrix,

        f"{prefix}_missed_to_false_positive.csv",

    )


    plot_matrix(

        missed_fp_matrix,

        f"{prefix}_missed_to_false_positive.png",

        (
            "Missed True Label → False-Positive Label\n"
            + description
        ),

        "False-Positive Label",

        "Missed True Label",

    )


    # ========================================================
    # 5. HIERARCHY ERRORS
    # ========================================================

    hierarchy_df = create_hierarchy_error_table(

        true_labels,

        predicted_labels,

    )


    hierarchy_df.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_hierarchy_errors.csv",

        index=False,

        encoding="utf-8-sig",
    )


    # ========================================================
    # 6. HIERARCHY SUMMARY
    # ========================================================

    hierarchy_stats = hierarchy_summary(

        true_labels,

        predicted_labels,

    )


    with open(

        ANALYSIS_DIR
        / f"{prefix}_hierarchy_summary.json",

        "w",

        encoding="utf-8",

    ) as f:

        json.dump(

            hierarchy_stats,

            f,

            indent=2,

        )


    # ========================================================
    # 7. DOCUMENT SUMMARY
    # ========================================================

    doc_summary = create_document_summary(

        true_labels,

        predicted_labels,

    )


    doc_summary.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_document_summary.csv",

        index=False,

        encoding="utf-8-sig",
    )


    # ========================================================
    # 8. PRINT SUMMARY
    # ========================================================

    print("\nDocument-level summary:")

    print(
        doc_summary.to_string(
            index=False
        )
    )


    print("\nHierarchy summary:")

    print(
        json.dumps(
            hierarchy_stats,
            indent=2,
        )
    )


    # ========================================================
    # 9. TOP ERROR PATTERNS
    # ========================================================

    print("\nTop 20 true -> predicted label-set patterns:")

    print(

        pattern_df.head(20).to_string(
            index=False
        )

    )


    print("\nTop 20 missed -> false-positive pairs:")

    pair_rows = []


    for i, true_label in enumerate(
        hierarchical_labels
    ):

        for j, predicted_label in enumerate(
            hierarchical_labels
        ):

            count = missed_fp_matrix[i, j]


            if count > 0:

                pair_rows.append({

                    "missed_label":
                        true_label,

                    "false_positive_label":
                        predicted_label,

                    "count":
                        int(count),

                })


    pair_df = pd.DataFrame(
        pair_rows
    )


    if len(pair_df) > 0:

        pair_df = pair_df.sort_values(
            "count",
            ascending=False,
        )


        print(

            pair_df.head(20).to_string(
                index=False
            )

        )


    # ========================================================
    # 10. TOP HIERARCHY VIOLATIONS
    # ========================================================

    print("\nHierarchy violation examples:")

    if len(hierarchy_df) > 0:

        print(

            hierarchy_df.head(20).to_string(
                index=False
            )

        )

    else:

        print(
            "No hierarchy violations found."
        )


# ============================================================
# RUN 1:
# ORIGINAL 0.50
# ============================================================

run_analysis(

    true_labels=test_labels,

    predicted_labels=predictions_05,

    probabilities=test_probabilities,

    prefix="threshold_0.50",

    description=(
        "Threshold = 0.50"
    ),

)


# ============================================================
# RUN 2:
# GLOBAL TUNED
# ============================================================

run_analysis(

    true_labels=test_labels,

    predicted_labels=predictions_global,

    probabilities=test_probabilities,

    prefix="global_tuned",

    description=(
        f"Global threshold = "
        f"{GLOBAL_TUNED_THRESHOLD:.3f}"
    ),

)


# ============================================================
# RUN 3:
# PER-LABEL TUNED
# ============================================================

run_analysis(

    true_labels=test_labels,

    predicted_labels=predictions_per_label,

    probabilities=test_probabilities,

    prefix="per_label_tuned",

    description=(
        "Per-label tuned thresholds"
    ),

)


# ============================================================
# SAVE THRESHOLDS
# ============================================================

threshold_df = pd.DataFrame({

    "label":
        hierarchical_labels,

    "threshold":
        threshold_array,

})


threshold_df.to_csv(

    ANALYSIS_DIR
    / "thresholds_used.csv",

    index=False,

    encoding="utf-8-sig",
)


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 100)
print("HIERARCHICAL ERROR ANALYSIS COMPLETE")
print("=" * 100)

print("\nResults saved to:")

print(
    ANALYSIS_DIR
)

print("\nThe most important files are:")

print(
    "  *_document_errors.csv"
)

print(
    "  *_error_documents_only.csv"
)

print(
    "  *_exact_label_set_patterns.csv"
)

print(
    "  *_per_label_summary.csv"
)

print(
    "  *_missed_to_false_positive.csv"
)

print(
    "  *_hierarchy_errors.csv"
)

print(
    "  *_hierarchy_summary.json"
)

print(
    "  *_document_summary.csv"
)

print("=" * 100)



LOADING TEST DATA
Logits shape: (633, 25)
Labels shape: (633, 25)
Labels: ['MT', 'LY', 'SP', 'ID', 'NA', 'HI', 'IN', 'OP', 'IP', 'it', 'ne', 'sr', 'nb', 're', 'en', 'ra', 'dtp', 'fi', 'lt', 'rv', 'ob', 'rs', 'av', 'ds', 'ed']

Parents:
['MT', 'LY', 'SP', 'ID', 'NA', 'HI', 'IN', 'OP', 'IP']

Children:
['it', 'ne', 'sr', 'nb', 're', 'en', 'ra', 'dtp', 'fi', 'lt', 'rv', 'ob', 'rs', 'av', 'ds', 'ed']


Threshold = 0.50

Document-level summary:
           category  count  percentage
        exact_match    240   37.914692
          any_error    393   62.085308
  partially_correct    361   57.030016
   completely_wrong     30    4.739336
  nothing_predicted      2    0.315956
hierarchy_violation      0    0.000000

Hierarchy summary:
{
  "total_documents": 633,
  "documents_with_predicted_hierarchy_violation": 0,
  "percent_documents_with_predicted_hierarchy_violation": 0.0,
  "documents_with_true_hierarchy_violation": 0,
  "child_correct_parent_missed_events": 0,
  "predicted_hierarchy_viol

# Golden Route

## New approach codes

In [6]:
# ============================================================
# HIERARCHICAL MULTILABEL ERROR ANALYSIS
#
# Designed for:
#
#   9 parent labels
#   16 child labels
#   25 total labels
#
# The analysis explicitly distinguishes:
#
#   1. Correct labels
#   2. False positives
#   3. False negatives
#   4. Missed-label -> false-positive associations
#   5. Completely missed documents
#   6. Partially correct documents
#   7. Exact label-set matches
#   8. Hierarchical violations
#   9. Child predicted but parent missing
#  10. Parent predicted while a specific child is missed
#
# IMPORTANT:
#
# This script ANALYZES independently thresholded predictions.
# It does NOT automatically repair/enforce the hierarchy.
#
# ============================================================


# ============================================================
# IMPORTS
# ============================================================

import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    multilabel_confusion_matrix,
)


# ============================================================
# CONFIGURATION
# ============================================================

OUTPUT_DIR = Path(
    r"C:\Users\alrazz\Downloads\Gold_routing"
)


LOGITS_FILE = (
    OUTPUT_DIR / "test_logits.npy"
)

LABELS_FILE = (
    OUTPUT_DIR / "test_labels.npy"
)

LABEL_NAMES_FILE = (
    OUTPUT_DIR / "hierarchical_labels.json"
)


ANALYSIS_DIR = (
    OUTPUT_DIR / "hierarchical_error_analysis"
)

ANALYSIS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# THRESHOLD CONFIGURATION
# ============================================================

PARENT_THRESHOLD = 0.50

GLOBAL_CHILD_THRESHOLD = 0.32


# ------------------------------------------------------------
# Per-label tuned thresholds
# ------------------------------------------------------------

PER_LABEL_THRESHOLDS = {

    # --------------------------------------------------------
    # Parents
    # --------------------------------------------------------

    "MT": 0.50,
    "LY": 0.50,
    "SP": 0.50,
    "ID": 0.50,
    "NA": 0.50,
    "HI": 0.50,
    "IN": 0.50,
    "OP": 0.50,
    "IP": 0.50,

    # --------------------------------------------------------
    # Children
    # --------------------------------------------------------

    "it": 0.67,

    "ne": 0.60,
    "sr": 0.77,
    "nb": 0.21,

    "re": 0.05,

    "en": 0.27,
    "ra": 0.31,
    "dtp": 0.29,
    "fi": 0.28,
    "lt": 0.23,

    "rv": 0.48,
    "ob": 0.36,
    "rs": 0.22,
    "av": 0.41,

    "ds": 0.95,
    "ed": 0.05,
}


# ============================================================
# LOAD DATA
# ============================================================

print("\n" + "=" * 100)
print("LOADING TEST DATA")
print("=" * 100)


test_logits = np.load(
    LOGITS_FILE
)

test_labels = np.load(
    LABELS_FILE
)


with open(
    LABEL_NAMES_FILE,
    "r",
    encoding="utf-8",
) as f:

    hierarchical_labels = json.load(f)


print(
    "Logits shape:",
    test_logits.shape,
)

print(
    "Labels shape:",
    test_labels.shape,
)

print(
    "Labels:",
    hierarchical_labels,
)


# ============================================================
# BASIC CHECKS
# ============================================================

if test_logits.shape != test_labels.shape:

    raise ValueError(
        "Logits and labels have different shapes."
    )


NUM_LABELS = len(
    hierarchical_labels
)


if test_logits.shape[1] != NUM_LABELS:

    raise ValueError(
        "Number of logits does not match number of labels."
    )


# ============================================================
# HIERARCHICAL STRUCTURE
# ============================================================

NUM_PARENTS = 9


parents = hierarchical_labels[
    :NUM_PARENTS
]

children = hierarchical_labels[
    NUM_PARENTS:
]


parent_indices = list(
    range(NUM_PARENTS)
)

child_indices = list(
    range(NUM_PARENTS, NUM_LABELS)
)


# ------------------------------------------------------------
# Explicit child -> parent mapping
# ------------------------------------------------------------

child_to_parent = {

    "it": "SP",

    "ne": "NA",
    "sr": "NA",
    "nb": "NA",

    "re": "HI",

    "en": "IN",
    "ra": "IN",
    "dtp": "IN",
    "fi": "IN",
    "lt": "IN",

    "rv": "OP",
    "ob": "OP",
    "rs": "OP",
    "av": "OP",

    "ds": "IP",
    "ed": "IP",
}


print("\nParents:")
print(parents)

print("\nChildren:")
print(children)


# ============================================================
# VERIFY CHILD -> PARENT MAPPING
# ============================================================

for child in children:

    if child not in child_to_parent:

        raise ValueError(
            f"No parent defined for child: {child}"
        )

    parent = child_to_parent[child]

    if parent not in parents:

        raise ValueError(
            f"Unknown parent {parent} for child {child}"
        )


# ------------------------------------------------------------
# Verify that all configured thresholds exist
# ------------------------------------------------------------

missing_thresholds = [

    label

    for label in hierarchical_labels

    if label not in PER_LABEL_THRESHOLDS
]


if missing_thresholds:

    raise ValueError(
        "Missing per-label thresholds for: "
        f"{missing_thresholds}"
    )


# ============================================================
# SIGMOID
# ============================================================

def sigmoid(x):

    # Numerically safer implementation
    x = np.clip(
        x,
        -50,
        50,
    )

    return 1.0 / (
        1.0 + np.exp(-x)
    )


test_probabilities = sigmoid(
    test_logits
)


# ============================================================
# THRESHOLD ARRAYS
# ============================================================

# ------------------------------------------------------------
# 1. All labels at 0.50
# ------------------------------------------------------------

threshold_array_050 = np.full(
    NUM_LABELS,
    0.50,
    dtype=np.float32,
)


# ------------------------------------------------------------
# 2. Parent = 0.50
#    Child = global 0.32
# ------------------------------------------------------------

threshold_array_parent_child_global = np.array(

    [
        PARENT_THRESHOLD
        if i < NUM_PARENTS
        else GLOBAL_CHILD_THRESHOLD

        for i in range(NUM_LABELS)
    ],

    dtype=np.float32,
)


# ------------------------------------------------------------
# 3. Individually tuned thresholds
# ------------------------------------------------------------

threshold_array_per_label = np.array(

    [
        PER_LABEL_THRESHOLDS[label]

        for label in hierarchical_labels
    ],

    dtype=np.float32,
)


# ============================================================
# PRINT THRESHOLDS
# ============================================================

print("\n" + "=" * 100)
print("THRESHOLDS")
print("=" * 100)

print(
    f"\n{'Label':>8} | "
    f"{'0.50':>8} | "
    f"{'Parent/Child':>15} | "
    f"{'Per-label':>12}"
)

print("-" * 55)


for label, t1, t2, t3 in zip(

    hierarchical_labels,

    threshold_array_050,

    threshold_array_parent_child_global,

    threshold_array_per_label,

):

    print(

        f"{label:>8} | "
        f"{t1:>8.2f} | "
        f"{t2:>15.2f} | "
        f"{t3:>12.2f}"

    )


# ============================================================
# PREDICTIONS
# ============================================================

def make_predictions(
    probabilities,
    thresholds,
):

    thresholds = np.asarray(
        thresholds,
        dtype=np.float32,
    )


    if thresholds.ndim == 0:

        thresholds = np.repeat(
            thresholds,
            probabilities.shape[1],
        )


    if len(thresholds) != probabilities.shape[1]:

        raise ValueError(
            "Number of thresholds does not match "
            "number of labels."
        )


    return (

        probabilities
        >= thresholds.reshape(1, -1)

    ).astype(np.int32)


# ------------------------------------------------------------
# Run 1: Original 0.50
# ------------------------------------------------------------

predictions_05 = make_predictions(

    test_probabilities,

    threshold_array_050,

)


# ------------------------------------------------------------
# Run 2: Parent 0.50 / Child global 0.32
# ------------------------------------------------------------

predictions_parent_child_global = make_predictions(

    test_probabilities,

    threshold_array_parent_child_global,

)


# ------------------------------------------------------------
# Run 3: Per-label tuned
# ------------------------------------------------------------

predictions_per_label = make_predictions(

    test_probabilities,

    threshold_array_per_label,

)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def label_set(
    row,
    labels=hierarchical_labels,
):

    return [

        labels[i]

        for i in np.where(
            row == 1
        )[0]

    ]


def label_string(
    row,
    labels=hierarchical_labels,
):

    values = label_set(
        row,
        labels,
    )


    if not values:

        return "NONE"


    return ", ".join(values)


def set_difference_string(
    true_row,
    pred_row,
):

    true_set = set(
        label_set(true_row)
    )

    pred_set = set(
        label_set(pred_row)
    )


    missed = sorted(
        true_set - pred_set
    )

    false_positive = sorted(
        pred_set - true_set
    )


    return (

        ", ".join(missed)
        if missed
        else "NONE"

    ), (

        ", ".join(false_positive)
        if false_positive
        else "NONE"

    )


# ============================================================
# HIERARCHICAL VIOLATIONS
# ============================================================
#
# A prediction hierarchy violation occurs when:
#
#     child = 1
#     parent = 0
#
# Example:
#
#     fi = 1
#     IN = 0
#
# This is an invalid hierarchical prediction.
#
# NOTE:
#
# The script does NOT automatically repair this.
# It only identifies the violation.
# ============================================================

def get_hierarchy_violations(
    prediction_row,
):

    violations = []


    for child, parent in child_to_parent.items():

        child_idx = hierarchical_labels.index(
            child
        )

        parent_idx = hierarchical_labels.index(
            parent
        )


        if (

            prediction_row[child_idx] == 1

            and

            prediction_row[parent_idx] == 0

        ):

            violations.append(
                f"{child}->{parent}"
            )


    return violations


# ============================================================
# TRUE HIERARCHY CHECK
# ============================================================
#
# Checks whether GOLD labels are hierarchically consistent.
#
# Normally all violations should be zero because the gold-label
# construction propagated child labels to their parents.
# ============================================================

def get_true_hierarchy_violations(
    true_row,
):

    violations = []


    for child, parent in child_to_parent.items():

        child_idx = hierarchical_labels.index(
            child
        )

        parent_idx = hierarchical_labels.index(
            parent
        )


        if (

            true_row[child_idx] == 1

            and

            true_row[parent_idx] == 0

        ):

            violations.append(
                f"{child}->{parent}"
            )


    return violations


# ============================================================
# PER-DOCUMENT ERROR ANALYSIS
# ============================================================

def create_document_error_table(
    true_labels,
    predicted_labels,
    probabilities,
):

    rows = []


    for i in range(
        len(true_labels)
    ):

        true_row = true_labels[i]

        pred_row = predicted_labels[i]

        prob_row = probabilities[i]


        true_set = set(
            label_set(true_row)
        )

        pred_set = set(
            label_set(pred_row)
        )


        correct = sorted(
            true_set & pred_set
        )

        missed = sorted(
            true_set - pred_set
        )

        false_positive = sorted(
            pred_set - true_set
        )


        true_count = len(
            true_set
        )

        predicted_count = len(
            pred_set
        )

        correct_count = len(
            correct
        )

        missed_count = len(
            missed
        )

        false_positive_count = len(
            false_positive
        )


        # ----------------------------------------------------
        # Exact set match
        # ----------------------------------------------------

        exact_match = (
            true_set == pred_set
        )


        # ----------------------------------------------------
        # Completely missed
        #
        # True labels exist, but nothing was predicted.
        # ----------------------------------------------------

        completely_missed = (

            true_count > 0

            and

            predicted_count == 0

        )


        # ----------------------------------------------------
        # No true labels
        # ----------------------------------------------------

        no_true_labels = (
            true_count == 0
        )


        # ----------------------------------------------------
        # Completely wrong
        #
        # Both true and predicted labels exist,
        # but there is no overlap.
        # ----------------------------------------------------

        completely_wrong = (

            true_count > 0

            and

            predicted_count > 0

            and

            correct_count == 0

        )


        # ----------------------------------------------------
        # Partially correct
        # ----------------------------------------------------

        partially_correct = (

            correct_count > 0

            and

            not exact_match

        )


        # ----------------------------------------------------
        # Hierarchical violations
        # ----------------------------------------------------

        hierarchy_violations = (

            get_hierarchy_violations(
                pred_row
            )

        )


        true_hierarchy_violations = (

            get_true_hierarchy_violations(
                true_row
            )

        )


        # ----------------------------------------------------
        # Probability of true labels
        # ----------------------------------------------------

        true_probabilities = [

            float(

                prob_row[
                    hierarchical_labels.index(label)
                ]

            )

            for label in sorted(true_set)

        ]


        # ----------------------------------------------------
        # Probability of missed labels
        # ----------------------------------------------------

        missed_probabilities = [

            float(

                prob_row[
                    hierarchical_labels.index(label)
                ]

            )

            for label in missed

        ]


        # ----------------------------------------------------
        # Probability of false positives
        # ----------------------------------------------------

        false_positive_probabilities = [

            float(

                prob_row[
                    hierarchical_labels.index(label)
                ]

            )

            for label in false_positive

        ]


        rows.append({

            "document_index":
                i,

            "true_labels":
                ", ".join(
                    sorted(true_set)
                )
                if true_set
                else "NONE",

            "predicted_labels":
                ", ".join(
                    sorted(pred_set)
                )
                if pred_set
                else "NONE",

            "correct_labels":
                ", ".join(correct)
                if correct
                else "NONE",

            "missed_labels":
                ", ".join(missed)
                if missed
                else "NONE",

            "false_positive_labels":
                ", ".join(false_positive)
                if false_positive
                else "NONE",

            "true_label_count":
                true_count,

            "predicted_label_count":
                predicted_count,

            "correct_label_count":
                correct_count,

            "missed_label_count":
                missed_count,

            "false_positive_count":
                false_positive_count,

            "exact_match":
                exact_match,

            "partially_correct":
                partially_correct,

            "completely_wrong":
                completely_wrong,

            "completely_missed":
                completely_missed,

            "no_true_labels":
                no_true_labels,

            "hierarchy_violation":
                len(
                    hierarchy_violations
                ) > 0,

            "hierarchy_violations":
                "; ".join(
                    hierarchy_violations
                )
                if hierarchy_violations
                else "NONE",

            "true_hierarchy_violation":
                len(
                    true_hierarchy_violations
                ) > 0,

            "true_hierarchy_violations":
                "; ".join(
                    true_hierarchy_violations
                )
                if true_hierarchy_violations
                else "NONE",

            "true_probabilities":
                "; ".join(
                    f"{p:.4f}"
                    for p in true_probabilities
                ),

            "missed_probabilities":
                "; ".join(
                    f"{p:.4f}"
                    for p in missed_probabilities
                ),

            "false_positive_probabilities":
                "; ".join(
                    f"{p:.4f}"
                    for p in false_positive_probabilities
                ),

        })


    return pd.DataFrame(rows)


# ============================================================
# PER-LABEL TP / FP / FN / TN
# ============================================================

def create_per_label_summary(
    true_labels,
    predicted_labels,
):

    mcm = multilabel_confusion_matrix(

        true_labels,

        predicted_labels,

    )


    rows = []


    for i, label in enumerate(
        hierarchical_labels
    ):

        tn = mcm[i, 0, 0]

        fp = mcm[i, 0, 1]

        fn = mcm[i, 1, 0]

        tp = mcm[i, 1, 1]


        precision = (

            tp / (tp + fp)

            if (tp + fp) > 0

            else 0.0

        )


        recall = (

            tp / (tp + fn)

            if (tp + fn) > 0

            else 0.0

        )


        f1 = (

            2 * precision * recall
            /
            (precision + recall)

            if (precision + recall) > 0

            else 0.0

        )


        rows.append({

            "label":
                label,

            "TP":
                int(tp),

            "FP":
                int(fp),

            "FN":
                int(fn),

            "TN":
                int(tn),

            "precision":
                precision,

            "recall":
                recall,

            "F1":
                f1,

            "support":
                int(tp + fn),

        })


    return pd.DataFrame(rows)


# ============================================================
# MISSED -> FALSE POSITIVE ASSOCIATION
# ============================================================
#
# Example:
#
# TRUE:
#     IN OP
#
# PRED:
#     IN NA
#
# Result:
#
#     OP -> NA
#
# Interpretation:
#
# "When OP was missed, NA was falsely predicted in the
# same document."
#
# This does NOT mathematically prove that NA replaced OP.
# ============================================================

def create_missed_to_fp_matrix(
    true_labels,
    predicted_labels,
):

    n_labels = len(
        hierarchical_labels
    )


    matrix = np.zeros(

        (n_labels, n_labels),

        dtype=np.int64,

    )


    for i in range(
        len(true_labels)
    ):

        true_row = true_labels[i]

        pred_row = predicted_labels[i]


        missed = np.where(

            (true_row == 1)

            &

            (pred_row == 0)

        )[0]


        false_positive = np.where(

            (true_row == 0)

            &

            (pred_row == 1)

        )[0]


        for missed_idx in missed:

            for fp_idx in false_positive:

                matrix[
                    missed_idx,
                    fp_idx
                ] += 1


    return matrix


# ============================================================
# SAVE MATRIX
# ============================================================

def save_matrix(
    matrix,
    filename,
):

    df = pd.DataFrame(

        matrix,

        index=hierarchical_labels,

        columns=hierarchical_labels,

    )


    df.to_csv(

        ANALYSIS_DIR / filename,

        encoding="utf-8-sig",

    )


    return df


# ============================================================
# PLOT MATRIX
# ============================================================

def plot_matrix(
    matrix,
    filename,
    title,
    xlabel,
    ylabel,
):

    plt.figure(
        figsize=(20, 17)
    )


    sns.heatmap(

        matrix,

        annot=True,

        fmt="d",

        cmap="Blues",

        xticklabels=hierarchical_labels,

        yticklabels=hierarchical_labels,

        linewidths=0.4,

    )


    plt.xlabel(

        xlabel,

        fontsize=13,

        fontweight="bold",

    )


    plt.ylabel(

        ylabel,

        fontsize=13,

        fontweight="bold",

    )


    plt.title(

        title,

        fontsize=16,

        fontweight="bold",

    )


    plt.xticks(

        rotation=45,

        ha="right",

    )


    plt.yticks(

        rotation=0,

    )


    plt.tight_layout()


    plt.savefig(

        ANALYSIS_DIR / filename,

        dpi=400,

        bbox_inches="tight",

    )


    plt.close()


# ============================================================
# EXACT TRUE -> PREDICTED SET PATTERNS
# ============================================================
#
# Example:
#
# TRUE = IN, OP
# PRED = IN, NA
#
# gets one row:
#
# IN, OP -> IN, NA : 37 documents
# ============================================================

def create_exact_pattern_table(
    true_labels,
    predicted_labels,
):

    counter = Counter()


    for true_row, pred_row in zip(

        true_labels,

        predicted_labels,

    ):

        true_string = label_string(
            true_row
        )

        pred_string = label_string(
            pred_row
        )


        counter[
            (
                true_string,
                pred_string,
            )
        ] += 1


    rows = []


    for (

        (

            true_string,

            pred_string,

        ),

        count,

    ) in counter.most_common():


        rows.append({

            "true_labels":
                true_string,

            "predicted_labels":
                pred_string,

            "count":
                count,

        })


    return pd.DataFrame(rows)


# ============================================================
# HIERARCHY ERROR ANALYSIS
# ============================================================

def create_hierarchy_error_table(
    true_labels,
    predicted_labels,
):

    rows = []

    for i in range(
        len(true_labels)
    ):

        true_row = true_labels[i]

        pred_row = predicted_labels[i]

        for child, parent in child_to_parent.items():

            child_idx = hierarchical_labels.index(
                child
            )

            parent_idx = hierarchical_labels.index(
                parent
            )

            child_true = (
                true_row[child_idx] == 1
            )

            parent_true = (
                true_row[parent_idx] == 1
            )

            child_pred = (
                pred_row[child_idx] == 1
            )

            parent_pred = (
                pred_row[parent_idx] == 1
            )


            # ------------------------------------------------
            # ERROR TYPE 1:
            #
            # Child is truly present and correctly predicted,
            # but its required parent is missed.
            #
            # TRUE:
            #     child  = 1
            #     parent = 1
            #
            # PRED:
            #     child  = 1
            #     parent = 0
            # ------------------------------------------------

            if (
                child_true
                and
                parent_true
                and
                child_pred
                and
                not parent_pred
            ):

                rows.append({

                    "document_index":
                        i,

                    "child":
                        child,

                    "parent":
                        parent,

                    "error_type":
                        "child_correct_parent_missed",

                    "true_child":
                        1,

                    "true_parent":
                        1,

                    "pred_child":
                        1,

                    "pred_parent":
                        0,

                })


            # ------------------------------------------------
            # ERROR TYPE 2:
            #
            # Model predicts a child without predicting its
            # required parent.
            #
            # This is a prediction-time hierarchy violation.
            #
            # It can happen whether or not the child is truly
            # present in the gold annotation.
            #
            # PRED:
            #     child  = 1
            #     parent = 0
            # ------------------------------------------------

            if (
                child_pred
                and
                not parent_pred
            ):

                rows.append({

                    "document_index":
                        i,

                    "child":
                        child,

                    "parent":
                        parent,

                    "error_type":
                        "hierarchy_violation",

                    "true_child":
                        int(child_true),

                    "true_parent":
                        int(parent_true),

                    "pred_child":
                        1,

                    "pred_parent":
                        0,

                })


            # ------------------------------------------------
            # ERROR TYPE 3:
            #
            # A TRUE child was missed while its parent was
            # correctly predicted.
            #
            # TRUE:
            #     child  = 1
            #     parent = 1
            #
            # PRED:
            #     child  = 0
            #     parent = 1
            # ------------------------------------------------

            if (
                child_true
                and
                parent_true
                and
                parent_pred
                and
                not child_pred
            ):

                rows.append({

                    "document_index":
                        i,

                    "child":
                        child,

                    "parent":
                        parent,

                    "error_type":
                        "parent_predicted_child_not_predicted",

                    "true_child":
                        1,

                    "true_parent":
                        1,

                    "pred_child":
                        0,

                    "pred_parent":
                        1,

                })


    return pd.DataFrame(
        rows
    )


# ============================================================
# HIERARCHICAL ERROR COUNTS
# ============================================================

def hierarchy_summary(
    true_labels,
    predicted_labels,
):

    total = len(
        true_labels
    )


    prediction_violations = 0

    child_correct_parent_missed = 0

    parent_correct_child_missed = 0

    true_hierarchy_violations = 0


    violation_pairs = Counter()

    child_parent_missed_pairs = Counter()

    parent_child_missed_pairs = Counter()


    for i in range(total):

        true_row = true_labels[i]

        pred_row = predicted_labels[i]


        true_violations = (
            get_true_hierarchy_violations(
                true_row
            )
        )


        pred_violations = (
            get_hierarchy_violations(
                pred_row
            )
        )


        if true_violations:

            true_hierarchy_violations += 1


        if pred_violations:

            prediction_violations += 1


            for pair in pred_violations:

                violation_pairs[pair] += 1


        for child, parent in child_to_parent.items():

            child_idx = hierarchical_labels.index(
                child
            )

            parent_idx = hierarchical_labels.index(
                parent
            )


            child_true = (
                true_row[child_idx] == 1
            )

            parent_true = (
                true_row[parent_idx] == 1
            )

            child_pred = (
                pred_row[child_idx] == 1
            )

            parent_pred = (
                pred_row[parent_idx] == 1
            )


            # ------------------------------------------------
            # Correct child, missed parent
            # ------------------------------------------------

            if (

                child_true

                and

                parent_true

                and

                child_pred

                and

                not parent_pred

            ):

                child_correct_parent_missed += 1

                child_parent_missed_pairs[
                    f"{child}->{parent}"
                ] += 1


            # ------------------------------------------------
            # Correct parent, missed true child
            # ------------------------------------------------

            if (

                child_true

                and

                parent_true

                and

                parent_pred

                and

                not child_pred

            ):

                parent_correct_child_missed += 1

                parent_child_missed_pairs[
                    f"{parent}->{child}"
                ] += 1


    return {

        "total_documents":
            total,

        "documents_with_predicted_hierarchy_violation":
            prediction_violations,

        "percent_documents_with_predicted_hierarchy_violation":
            (
                prediction_violations / total * 100
                if total > 0
                else 0
            ),

        "documents_with_true_hierarchy_violation":
            true_hierarchy_violations,

        "child_correct_parent_missed_events":
            child_correct_parent_missed,

        "parent_correct_child_missed_events":
            parent_correct_child_missed,

        "predicted_hierarchy_violation_pairs":
            dict(violation_pairs),

        "child_correct_parent_missed_pairs":
            dict(child_parent_missed_pairs),

        "parent_correct_child_missed_pairs":
            dict(parent_child_missed_pairs),

    }


# ============================================================
# DOCUMENT-LEVEL SUMMARY
# ============================================================

def create_document_summary(
    true_labels,
    predicted_labels,
):

    total = len(
        true_labels
    )


    exact = 0

    partial = 0

    completely_wrong = 0

    completely_missed = 0

    any_error = 0

    hierarchy_violation = 0


    for i in range(total):

        true_set = set(
            label_set(
                true_labels[i]
            )
        )

        pred_set = set(
            label_set(
                predicted_labels[i]
            )
        )


        # ----------------------------------------------------
        # Exact match
        # ----------------------------------------------------

        if true_set == pred_set:

            exact += 1

        else:

            any_error += 1


        # ----------------------------------------------------
        # Partially correct
        # ----------------------------------------------------

        if (

            true_set

            and

            pred_set

            and

            true_set & pred_set

            and

            true_set != pred_set

        ):

            partial += 1


        # ----------------------------------------------------
        # Completely wrong
        # ----------------------------------------------------

        if (

            true_set

            and

            pred_set

            and

            not (true_set & pred_set)

        ):

            completely_wrong += 1


        # ----------------------------------------------------
        # Completely missed
        # ----------------------------------------------------

        if (

            true_set

            and

            not pred_set

        ):

            completely_missed += 1


        # ----------------------------------------------------
        # Hierarchy violation
        # ----------------------------------------------------

        hierarchy_violations = (
            get_hierarchy_violations(
                predicted_labels[i]
            )
        )


        if hierarchy_violations:

            hierarchy_violation += 1


    if total > 0:

        percentages = [

            exact / total * 100,

            any_error / total * 100,

            partial / total * 100,

            completely_wrong / total * 100,

            completely_missed / total * 100,

            hierarchy_violation / total * 100,

        ]

    else:

        percentages = [
            0.0
            for _ in range(6)
        ]


    return pd.DataFrame({

        "category": [

            "exact_match",

            "any_error",

            "partially_correct",

            "completely_wrong",

            "completely_missed",

            "hierarchy_violation",

        ],

        "count": [

            exact,

            any_error,

            partial,

            completely_wrong,

            completely_missed,

            hierarchy_violation,

        ],

        "percentage": percentages,

    })


# ============================================================
# RUN ANALYSIS
# ============================================================

def run_analysis(
    true_labels,
    predicted_labels,
    probabilities,
    prefix,
    description,
):

    print("\n")
    print("=" * 100)
    print(description)
    print("=" * 100)


    # ========================================================
    # 1. DOCUMENT-LEVEL ERROR TABLE
    # ========================================================

    document_df = create_document_error_table(

        true_labels,

        predicted_labels,

        probabilities,

    )


    document_df.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_document_errors.csv",

        index=False,

        encoding="utf-8-sig",

    )


    # --------------------------------------------------------
    # Save only documents containing errors
    # --------------------------------------------------------

    error_documents = document_df[

        ~document_df["exact_match"]

    ]


    error_documents.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_error_documents_only.csv",

        index=False,

        encoding="utf-8-sig",

    )


    # ========================================================
    # 2. EXACT LABEL-SET PATTERNS
    # ========================================================

    pattern_df = create_exact_pattern_table(

        true_labels,

        predicted_labels,

    )


    pattern_df.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_exact_label_set_patterns.csv",

        index=False,

        encoding="utf-8-sig",

    )


    # ========================================================
    # 3. PER-LABEL SUMMARY
    # ========================================================

    label_df = create_per_label_summary(

        true_labels,

        predicted_labels,

    )


    label_df.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_per_label_summary.csv",

        index=False,

        encoding="utf-8-sig",

    )


    # ========================================================
    # 4. MISSED -> FALSE POSITIVE MATRIX
    # ========================================================

    missed_fp_matrix = create_missed_to_fp_matrix(

        true_labels,

        predicted_labels,

    )


    save_matrix(

        missed_fp_matrix,

        f"{prefix}_missed_to_false_positive.csv",

    )


    plot_matrix(

        missed_fp_matrix,

        f"{prefix}_missed_to_false_positive.png",

        (
            "Missed True Label → False-Positive Label\n"
            + description
        ),

        "False-Positive Label",

        "Missed True Label",

    )


    # ========================================================
    # 5. HIERARCHY ERRORS
    # ========================================================

    hierarchy_df = create_hierarchy_error_table(

        true_labels,

        predicted_labels,

    )


    hierarchy_df.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_hierarchy_errors.csv",

        index=False,

        encoding="utf-8-sig",

    )


    # ========================================================
    # 6. HIERARCHY SUMMARY
    # ========================================================

    hierarchy_stats = hierarchy_summary(

        true_labels,

        predicted_labels,

    )


    with open(

        ANALYSIS_DIR
        / f"{prefix}_hierarchy_summary.json",

        "w",

        encoding="utf-8",

    ) as f:

        json.dump(

            hierarchy_stats,

            f,

            indent=2,

        )


    # ========================================================
    # 7. DOCUMENT SUMMARY
    # ========================================================

    doc_summary = create_document_summary(

        true_labels,

        predicted_labels,

    )


    doc_summary.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_document_summary.csv",

        index=False,

        encoding="utf-8-sig",

    )


    # ========================================================
    # 8. PRINT SUMMARY
    # ========================================================

    print("\nDocument-level summary:")

    print(

        doc_summary.to_string(
            index=False
        )

    )


    print("\nHierarchy summary:")

    print(

        json.dumps(

            hierarchy_stats,

            indent=2,

        )

    )


    # ========================================================
    # 9. TOP ERROR PATTERNS
    # ========================================================

    print(
        "\nTop 20 true -> predicted label-set patterns:"
    )


    print(

        pattern_df.head(20).to_string(
            index=False
        )

    )


    # ========================================================
    # 10. TOP MISSED -> FP PAIRS
    # ========================================================

    print(
        "\nTop 20 missed -> false-positive pairs:"
    )


    pair_rows = []


    for i, true_label in enumerate(
        hierarchical_labels
    ):

        for j, predicted_label in enumerate(
            hierarchical_labels
        ):

            count = missed_fp_matrix[i, j]


            if count > 0:

                pair_rows.append({

                    "missed_label":
                        true_label,

                    "false_positive_label":
                        predicted_label,

                    "count":
                        int(count),

                })


    pair_df = pd.DataFrame(
        pair_rows
    )


    if len(pair_df) > 0:

        pair_df = pair_df.sort_values(

            "count",

            ascending=False,

        )


        print(

            pair_df.head(20).to_string(
                index=False
            )

        )

    else:

        print(
            "No missed -> false-positive pairs found."
        )


    # ========================================================
    # 11. TOP HIERARCHY ERRORS
    # ========================================================

    print(
        "\nHierarchy error examples:"
    )


    if len(hierarchy_df) > 0:

        print(

            hierarchy_df.head(20).to_string(
                index=False
            )

        )

    else:

        print(
            "No hierarchy errors found."
        )


# ============================================================
# RUN 1:
# ORIGINAL 0.50
# ============================================================

run_analysis(

    true_labels=test_labels,

    predicted_labels=predictions_05,

    probabilities=test_probabilities,

    prefix="threshold_0.50",

    description=(
        "Threshold = 0.50"
    ),

)


# ============================================================
# RUN 2:
# PARENT 0.50 / GLOBAL CHILD 0.32
# ============================================================

run_analysis(

    true_labels=test_labels,

    predicted_labels=predictions_parent_child_global,

    probabilities=test_probabilities,

    prefix="parent_050_child_global_032",

    description=(

        "Parent threshold = 0.50; "
        "global child threshold = 0.32"

    ),

)


# ============================================================
# RUN 3:
# PER-LABEL TUNED
# ============================================================

run_analysis(

    true_labels=test_labels,

    predicted_labels=predictions_per_label,

    probabilities=test_probabilities,

    prefix="per_label_tuned",

    description=(
        "Per-label tuned thresholds"
    ),

)


# ============================================================
# SAVE THRESHOLDS
# ============================================================

threshold_df = pd.DataFrame({

    "label":
        hierarchical_labels,

    "threshold_0.50":
        threshold_array_050,

    "parent_050_child_global_032":
        threshold_array_parent_child_global,

    "per_label_tuned":
        threshold_array_per_label,

})


threshold_df.to_csv(

    ANALYSIS_DIR
    / "thresholds_used.csv",

    index=False,

    encoding="utf-8-sig",

)


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 100)
print("HIERARCHICAL ERROR ANALYSIS COMPLETE")
print("=" * 100)


print("\nResults saved to:")

print(
    ANALYSIS_DIR
)


print("\nThe most important files are:")


print(
    "  *_document_errors.csv"
)

print(
    "  *_error_documents_only.csv"
)

print(
    "  *_exact_label_set_patterns.csv"
)

print(
    "  *_per_label_summary.csv"
)

print(
    "  *_missed_to_false_positive.csv"
)

print(
    "  *_missed_to_false_positive.png"
)

print(
    "  *_hierarchy_errors.csv"
)

print(
    "  *_hierarchy_summary.json"
)

print(
    "  *_document_summary.csv"
)

print(
    "  thresholds_used.csv"
)


print("=" * 100)



LOADING TEST DATA
Logits shape: (633, 25)
Labels shape: (633, 25)
Labels: ['MT', 'LY', 'SP', 'ID', 'NA', 'HI', 'IN', 'OP', 'IP', 'it', 'ne', 'sr', 'nb', 're', 'en', 'ra', 'dtp', 'fi', 'lt', 'rv', 'ob', 'rs', 'av', 'ds', 'ed']

Parents:
['MT', 'LY', 'SP', 'ID', 'NA', 'HI', 'IN', 'OP', 'IP']

Children:
['it', 'ne', 'sr', 'nb', 're', 'en', 'ra', 'dtp', 'fi', 'lt', 'rv', 'ob', 'rs', 'av', 'ds', 'ed']

THRESHOLDS

   Label |     0.50 |    Parent/Child |    Per-label
-------------------------------------------------------
      MT |     0.50 |            0.50 |         0.50
      LY |     0.50 |            0.50 |         0.50
      SP |     0.50 |            0.50 |         0.50
      ID |     0.50 |            0.50 |         0.50
      NA |     0.50 |            0.50 |         0.50
      HI |     0.50 |            0.50 |         0.50
      IN |     0.50 |            0.50 |         0.50
      OP |     0.50 |            0.50 |         0.50
      IP |     0.50 |            0.50 |         0.50


# Hard Route

## New approach codes

In [8]:
# ============================================================
# HIERARCHICAL MULTILABEL ERROR ANALYSIS
#
# Designed for:
#
#   9 parent labels
#   16 child labels
#   25 total labels
#
# The analysis explicitly distinguishes:
#
#   1. Correct labels
#   2. False positives
#   3. False negatives
#   4. Missed-label -> false-positive associations
#   5. Completely missed documents
#   6. Partially correct documents
#   7. Exact label-set matches
#   8. Hierarchical violations
#   9. Child predicted but parent missing
#  10. Parent predicted while a specific child is missed
#
# IMPORTANT:
#
# This script ANALYZES independently thresholded predictions.
# It does NOT automatically repair/enforce the hierarchy.
#
# ============================================================


# ============================================================
# IMPORTS
# ============================================================

import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    multilabel_confusion_matrix,
)


# ============================================================
# CONFIGURATION
# ============================================================

OUTPUT_DIR = Path(
    r"C:\Users\alrazz\Downloads\Hard_routing"
)


LOGITS_FILE = (
    OUTPUT_DIR / "test_logits.npy"
)

LABELS_FILE = (
    OUTPUT_DIR / "test_labels.npy"
)

LABEL_NAMES_FILE = (
    OUTPUT_DIR / "hierarchical_labels.json"
)


ANALYSIS_DIR = (
    OUTPUT_DIR / "hierarchical_error_analysis"
)

ANALYSIS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# THRESHOLD CONFIGURATION
# ============================================================

PARENT_THRESHOLD = 0.50

GLOBAL_CHILD_THRESHOLD = 0.35


# ------------------------------------------------------------
# Per-label tuned thresholds
# ------------------------------------------------------------

PER_LABEL_THRESHOLDS = {

    # --------------------------------------------------------
    # Parents
    # --------------------------------------------------------

    "MT": 0.50,
    "LY": 0.50,
    "SP": 0.50,
    "ID": 0.50,
    "NA": 0.50,
    "HI": 0.50,
    "IN": 0.50,
    "OP": 0.50,
    "IP": 0.50,

    # --------------------------------------------------------
    # Children
    # --------------------------------------------------------

    "it": 0.84,

    "ne": 0.52,
    "sr": 0.36,
    "nb": 0.42,

    "re": 0.95,

    "en": 0.39,
    "ra": 0.21,
    "dtp": 0.35,
    "fi": 0.27,
    "lt": 0.35,

    "rv": 0.55,
    "ob": 0.33,
    "rs": 0.33,
    "av": 0.35,

    "ds": 0.82,
    "ed": 0.13,
}


# ============================================================
# LOAD DATA
# ============================================================

print("\n" + "=" * 100)
print("LOADING TEST DATA")
print("=" * 100)


test_logits = np.load(
    LOGITS_FILE
)

test_labels = np.load(
    LABELS_FILE
)


with open(
    LABEL_NAMES_FILE,
    "r",
    encoding="utf-8",
) as f:

    hierarchical_labels = json.load(f)


print(
    "Logits shape:",
    test_logits.shape,
)

print(
    "Labels shape:",
    test_labels.shape,
)

print(
    "Labels:",
    hierarchical_labels,
)


# ============================================================
# BASIC CHECKS
# ============================================================

if test_logits.shape != test_labels.shape:

    raise ValueError(
        "Logits and labels have different shapes."
    )


NUM_LABELS = len(
    hierarchical_labels
)


if test_logits.shape[1] != NUM_LABELS:

    raise ValueError(
        "Number of logits does not match number of labels."
    )


# ============================================================
# HIERARCHICAL STRUCTURE
# ============================================================

NUM_PARENTS = 9


parents = hierarchical_labels[
    :NUM_PARENTS
]

children = hierarchical_labels[
    NUM_PARENTS:
]


parent_indices = list(
    range(NUM_PARENTS)
)

child_indices = list(
    range(NUM_PARENTS, NUM_LABELS)
)


# ------------------------------------------------------------
# Explicit child -> parent mapping
# ------------------------------------------------------------

child_to_parent = {

    "it": "SP",

    "ne": "NA",
    "sr": "NA",
    "nb": "NA",

    "re": "HI",

    "en": "IN",
    "ra": "IN",
    "dtp": "IN",
    "fi": "IN",
    "lt": "IN",

    "rv": "OP",
    "ob": "OP",
    "rs": "OP",
    "av": "OP",

    "ds": "IP",
    "ed": "IP",
}


print("\nParents:")
print(parents)

print("\nChildren:")
print(children)


# ============================================================
# VERIFY CHILD -> PARENT MAPPING
# ============================================================

for child in children:

    if child not in child_to_parent:

        raise ValueError(
            f"No parent defined for child: {child}"
        )

    parent = child_to_parent[child]

    if parent not in parents:

        raise ValueError(
            f"Unknown parent {parent} for child {child}"
        )


# ------------------------------------------------------------
# Verify that all configured thresholds exist
# ------------------------------------------------------------

missing_thresholds = [

    label

    for label in hierarchical_labels

    if label not in PER_LABEL_THRESHOLDS
]


if missing_thresholds:

    raise ValueError(
        "Missing per-label thresholds for: "
        f"{missing_thresholds}"
    )


# ============================================================
# SIGMOID
# ============================================================

def sigmoid(x):

    # Numerically safer implementation
    x = np.clip(
        x,
        -50,
        50,
    )

    return 1.0 / (
        1.0 + np.exp(-x)
    )


test_probabilities = sigmoid(
    test_logits
)


# ============================================================
# THRESHOLD ARRAYS
# ============================================================

# ------------------------------------------------------------
# 1. All labels at 0.50
# ------------------------------------------------------------

threshold_array_050 = np.full(
    NUM_LABELS,
    0.50,
    dtype=np.float32,
)


# ------------------------------------------------------------
# 2. Parent = 0.50
#    Child = global 0.32
# ------------------------------------------------------------

threshold_array_parent_child_global = np.array(

    [
        PARENT_THRESHOLD
        if i < NUM_PARENTS
        else GLOBAL_CHILD_THRESHOLD

        for i in range(NUM_LABELS)
    ],

    dtype=np.float32,
)


# ------------------------------------------------------------
# 3. Individually tuned thresholds
# ------------------------------------------------------------

threshold_array_per_label = np.array(

    [
        PER_LABEL_THRESHOLDS[label]

        for label in hierarchical_labels
    ],

    dtype=np.float32,
)


# ============================================================
# PRINT THRESHOLDS
# ============================================================

print("\n" + "=" * 100)
print("THRESHOLDS")
print("=" * 100)

print(
    f"\n{'Label':>8} | "
    f"{'0.50':>8} | "
    f"{'Parent/Child':>15} | "
    f"{'Per-label':>12}"
)

print("-" * 55)


for label, t1, t2, t3 in zip(

    hierarchical_labels,

    threshold_array_050,

    threshold_array_parent_child_global,

    threshold_array_per_label,

):

    print(

        f"{label:>8} | "
        f"{t1:>8.2f} | "
        f"{t2:>15.2f} | "
        f"{t3:>12.2f}"

    )


# ============================================================
# PREDICTIONS
# ============================================================

def make_predictions(
    probabilities,
    thresholds,
):

    thresholds = np.asarray(
        thresholds,
        dtype=np.float32,
    )


    if thresholds.ndim == 0:

        thresholds = np.repeat(
            thresholds,
            probabilities.shape[1],
        )


    if len(thresholds) != probabilities.shape[1]:

        raise ValueError(
            "Number of thresholds does not match "
            "number of labels."
        )


    return (

        probabilities
        >= thresholds.reshape(1, -1)

    ).astype(np.int32)


# ------------------------------------------------------------
# Run 1: Original 0.50
# ------------------------------------------------------------

predictions_05 = make_predictions(

    test_probabilities,

    threshold_array_050,

)


# ------------------------------------------------------------
# Run 2: Parent 0.50 / Child global 0.32
# ------------------------------------------------------------

predictions_parent_child_global = make_predictions(

    test_probabilities,

    threshold_array_parent_child_global,

)


# ------------------------------------------------------------
# Run 3: Per-label tuned
# ------------------------------------------------------------

predictions_per_label = make_predictions(

    test_probabilities,

    threshold_array_per_label,

)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def label_set(
    row,
    labels=hierarchical_labels,
):

    return [

        labels[i]

        for i in np.where(
            row == 1
        )[0]

    ]


def label_string(
    row,
    labels=hierarchical_labels,
):

    values = label_set(
        row,
        labels,
    )


    if not values:

        return "NONE"


    return ", ".join(values)


def set_difference_string(
    true_row,
    pred_row,
):

    true_set = set(
        label_set(true_row)
    )

    pred_set = set(
        label_set(pred_row)
    )


    missed = sorted(
        true_set - pred_set
    )

    false_positive = sorted(
        pred_set - true_set
    )


    return (

        ", ".join(missed)
        if missed
        else "NONE"

    ), (

        ", ".join(false_positive)
        if false_positive
        else "NONE"

    )


# ============================================================
# HIERARCHICAL VIOLATIONS
# ============================================================
#
# A prediction hierarchy violation occurs when:
#
#     child = 1
#     parent = 0
#
# Example:
#
#     fi = 1
#     IN = 0
#
# This is an invalid hierarchical prediction.
#
# NOTE:
#
# The script does NOT automatically repair this.
# It only identifies the violation.
# ============================================================

def get_hierarchy_violations(
    prediction_row,
):

    violations = []


    for child, parent in child_to_parent.items():

        child_idx = hierarchical_labels.index(
            child
        )

        parent_idx = hierarchical_labels.index(
            parent
        )


        if (

            prediction_row[child_idx] == 1

            and

            prediction_row[parent_idx] == 0

        ):

            violations.append(
                f"{child}->{parent}"
            )


    return violations


# ============================================================
# TRUE HIERARCHY CHECK
# ============================================================
#
# Checks whether GOLD labels are hierarchically consistent.
#
# Normally all violations should be zero because the gold-label
# construction propagated child labels to their parents.
# ============================================================

def get_true_hierarchy_violations(
    true_row,
):

    violations = []


    for child, parent in child_to_parent.items():

        child_idx = hierarchical_labels.index(
            child
        )

        parent_idx = hierarchical_labels.index(
            parent
        )


        if (

            true_row[child_idx] == 1

            and

            true_row[parent_idx] == 0

        ):

            violations.append(
                f"{child}->{parent}"
            )


    return violations


# ============================================================
# PER-DOCUMENT ERROR ANALYSIS
# ============================================================

def create_document_error_table(
    true_labels,
    predicted_labels,
    probabilities,
):

    rows = []


    for i in range(
        len(true_labels)
    ):

        true_row = true_labels[i]

        pred_row = predicted_labels[i]

        prob_row = probabilities[i]


        true_set = set(
            label_set(true_row)
        )

        pred_set = set(
            label_set(pred_row)
        )


        correct = sorted(
            true_set & pred_set
        )

        missed = sorted(
            true_set - pred_set
        )

        false_positive = sorted(
            pred_set - true_set
        )


        true_count = len(
            true_set
        )

        predicted_count = len(
            pred_set
        )

        correct_count = len(
            correct
        )

        missed_count = len(
            missed
        )

        false_positive_count = len(
            false_positive
        )


        # ----------------------------------------------------
        # Exact set match
        # ----------------------------------------------------

        exact_match = (
            true_set == pred_set
        )


        # ----------------------------------------------------
        # Completely missed
        #
        # True labels exist, but nothing was predicted.
        # ----------------------------------------------------

        completely_missed = (

            true_count > 0

            and

            predicted_count == 0

        )


        # ----------------------------------------------------
        # No true labels
        # ----------------------------------------------------

        no_true_labels = (
            true_count == 0
        )


        # ----------------------------------------------------
        # Completely wrong
        #
        # Both true and predicted labels exist,
        # but there is no overlap.
        # ----------------------------------------------------

        completely_wrong = (

            true_count > 0

            and

            predicted_count > 0

            and

            correct_count == 0

        )


        # ----------------------------------------------------
        # Partially correct
        # ----------------------------------------------------

        partially_correct = (

            correct_count > 0

            and

            not exact_match

        )


        # ----------------------------------------------------
        # Hierarchical violations
        # ----------------------------------------------------

        hierarchy_violations = (

            get_hierarchy_violations(
                pred_row
            )

        )


        true_hierarchy_violations = (

            get_true_hierarchy_violations(
                true_row
            )

        )


        # ----------------------------------------------------
        # Probability of true labels
        # ----------------------------------------------------

        true_probabilities = [

            float(

                prob_row[
                    hierarchical_labels.index(label)
                ]

            )

            for label in sorted(true_set)

        ]


        # ----------------------------------------------------
        # Probability of missed labels
        # ----------------------------------------------------

        missed_probabilities = [

            float(

                prob_row[
                    hierarchical_labels.index(label)
                ]

            )

            for label in missed

        ]


        # ----------------------------------------------------
        # Probability of false positives
        # ----------------------------------------------------

        false_positive_probabilities = [

            float(

                prob_row[
                    hierarchical_labels.index(label)
                ]

            )

            for label in false_positive

        ]


        rows.append({

            "document_index":
                i,

            "true_labels":
                ", ".join(
                    sorted(true_set)
                )
                if true_set
                else "NONE",

            "predicted_labels":
                ", ".join(
                    sorted(pred_set)
                )
                if pred_set
                else "NONE",

            "correct_labels":
                ", ".join(correct)
                if correct
                else "NONE",

            "missed_labels":
                ", ".join(missed)
                if missed
                else "NONE",

            "false_positive_labels":
                ", ".join(false_positive)
                if false_positive
                else "NONE",

            "true_label_count":
                true_count,

            "predicted_label_count":
                predicted_count,

            "correct_label_count":
                correct_count,

            "missed_label_count":
                missed_count,

            "false_positive_count":
                false_positive_count,

            "exact_match":
                exact_match,

            "partially_correct":
                partially_correct,

            "completely_wrong":
                completely_wrong,

            "completely_missed":
                completely_missed,

            "no_true_labels":
                no_true_labels,

            "hierarchy_violation":
                len(
                    hierarchy_violations
                ) > 0,

            "hierarchy_violations":
                "; ".join(
                    hierarchy_violations
                )
                if hierarchy_violations
                else "NONE",

            "true_hierarchy_violation":
                len(
                    true_hierarchy_violations
                ) > 0,

            "true_hierarchy_violations":
                "; ".join(
                    true_hierarchy_violations
                )
                if true_hierarchy_violations
                else "NONE",

            "true_probabilities":
                "; ".join(
                    f"{p:.4f}"
                    for p in true_probabilities
                ),

            "missed_probabilities":
                "; ".join(
                    f"{p:.4f}"
                    for p in missed_probabilities
                ),

            "false_positive_probabilities":
                "; ".join(
                    f"{p:.4f}"
                    for p in false_positive_probabilities
                ),

        })


    return pd.DataFrame(rows)


# ============================================================
# PER-LABEL TP / FP / FN / TN
# ============================================================

def create_per_label_summary(
    true_labels,
    predicted_labels,
):

    mcm = multilabel_confusion_matrix(

        true_labels,

        predicted_labels,

    )


    rows = []


    for i, label in enumerate(
        hierarchical_labels
    ):

        tn = mcm[i, 0, 0]

        fp = mcm[i, 0, 1]

        fn = mcm[i, 1, 0]

        tp = mcm[i, 1, 1]


        precision = (

            tp / (tp + fp)

            if (tp + fp) > 0

            else 0.0

        )


        recall = (

            tp / (tp + fn)

            if (tp + fn) > 0

            else 0.0

        )


        f1 = (

            2 * precision * recall
            /
            (precision + recall)

            if (precision + recall) > 0

            else 0.0

        )


        rows.append({

            "label":
                label,

            "TP":
                int(tp),

            "FP":
                int(fp),

            "FN":
                int(fn),

            "TN":
                int(tn),

            "precision":
                precision,

            "recall":
                recall,

            "F1":
                f1,

            "support":
                int(tp + fn),

        })


    return pd.DataFrame(rows)


# ============================================================
# MISSED -> FALSE POSITIVE ASSOCIATION
# ============================================================
#
# Example:
#
# TRUE:
#     IN OP
#
# PRED:
#     IN NA
#
# Result:
#
#     OP -> NA
#
# Interpretation:
#
# "When OP was missed, NA was falsely predicted in the
# same document."
#
# This does NOT mathematically prove that NA replaced OP.
# ============================================================

def create_missed_to_fp_matrix(
    true_labels,
    predicted_labels,
):

    n_labels = len(
        hierarchical_labels
    )


    matrix = np.zeros(

        (n_labels, n_labels),

        dtype=np.int64,

    )


    for i in range(
        len(true_labels)
    ):

        true_row = true_labels[i]

        pred_row = predicted_labels[i]


        missed = np.where(

            (true_row == 1)

            &

            (pred_row == 0)

        )[0]


        false_positive = np.where(

            (true_row == 0)

            &

            (pred_row == 1)

        )[0]


        for missed_idx in missed:

            for fp_idx in false_positive:

                matrix[
                    missed_idx,
                    fp_idx
                ] += 1


    return matrix


# ============================================================
# SAVE MATRIX
# ============================================================

def save_matrix(
    matrix,
    filename,
):

    df = pd.DataFrame(

        matrix,

        index=hierarchical_labels,

        columns=hierarchical_labels,

    )


    df.to_csv(

        ANALYSIS_DIR / filename,

        encoding="utf-8-sig",

    )


    return df


# ============================================================
# PLOT MATRIX
# ============================================================

def plot_matrix(
    matrix,
    filename,
    title,
    xlabel,
    ylabel,
):

    plt.figure(
        figsize=(20, 17)
    )


    sns.heatmap(

        matrix,

        annot=True,

        fmt="d",

        cmap="Blues",

        xticklabels=hierarchical_labels,

        yticklabels=hierarchical_labels,

        linewidths=0.4,

    )


    plt.xlabel(

        xlabel,

        fontsize=13,

        fontweight="bold",

    )


    plt.ylabel(

        ylabel,

        fontsize=13,

        fontweight="bold",

    )


    plt.title(

        title,

        fontsize=16,

        fontweight="bold",

    )


    plt.xticks(

        rotation=45,

        ha="right",

    )


    plt.yticks(

        rotation=0,

    )


    plt.tight_layout()


    plt.savefig(

        ANALYSIS_DIR / filename,

        dpi=400,

        bbox_inches="tight",

    )


    plt.close()


# ============================================================
# EXACT TRUE -> PREDICTED SET PATTERNS
# ============================================================
#
# Example:
#
# TRUE = IN, OP
# PRED = IN, NA
#
# gets one row:
#
# IN, OP -> IN, NA : 37 documents
# ============================================================

def create_exact_pattern_table(
    true_labels,
    predicted_labels,
):

    counter = Counter()


    for true_row, pred_row in zip(

        true_labels,

        predicted_labels,

    ):

        true_string = label_string(
            true_row
        )

        pred_string = label_string(
            pred_row
        )


        counter[
            (
                true_string,
                pred_string,
            )
        ] += 1


    rows = []


    for (

        (

            true_string,

            pred_string,

        ),

        count,

    ) in counter.most_common():


        rows.append({

            "true_labels":
                true_string,

            "predicted_labels":
                pred_string,

            "count":
                count,

        })


    return pd.DataFrame(rows)


def create_hierarchy_error_table(
    true_labels,
    predicted_labels,
):

    rows = []

    for i in range(
        len(true_labels)
    ):

        true_row = true_labels[i]

        pred_row = predicted_labels[i]

        for child, parent in child_to_parent.items():

            child_idx = hierarchical_labels.index(
                child
            )

            parent_idx = hierarchical_labels.index(
                parent
            )

            child_true = (
                true_row[child_idx] == 1
            )

            parent_true = (
                true_row[parent_idx] == 1
            )

            child_pred = (
                pred_row[child_idx] == 1
            )

            parent_pred = (
                pred_row[parent_idx] == 1
            )


            # ------------------------------------------------
            # ERROR TYPE 1:
            #
            # Child is truly present and correctly predicted,
            # but its required parent is missed.
            #
            # TRUE:
            #     child  = 1
            #     parent = 1
            #
            # PRED:
            #     child  = 1
            #     parent = 0
            # ------------------------------------------------

            if (
                child_true
                and
                parent_true
                and
                child_pred
                and
                not parent_pred
            ):

                rows.append({

                    "document_index":
                        i,

                    "child":
                        child,

                    "parent":
                        parent,

                    "error_type":
                        "child_correct_parent_missed",

                    "true_child":
                        1,

                    "true_parent":
                        1,

                    "pred_child":
                        1,

                    "pred_parent":
                        0,

                })


            # ------------------------------------------------
            # ERROR TYPE 2:
            #
            # Model predicts a child without predicting its
            # required parent.
            #
            # This is a prediction-time hierarchy violation.
            #
            # It can happen whether or not the child is truly
            # present in the gold annotation.
            #
            # PRED:
            #     child  = 1
            #     parent = 0
            # ------------------------------------------------

            if (
                child_pred
                and
                not parent_pred
            ):

                rows.append({

                    "document_index":
                        i,

                    "child":
                        child,

                    "parent":
                        parent,

                    "error_type":
                        "hierarchy_violation",

                    "true_child":
                        int(child_true),

                    "true_parent":
                        int(parent_true),

                    "pred_child":
                        1,

                    "pred_parent":
                        0,

                })


            # ------------------------------------------------
            # ERROR TYPE 3:
            #
            # A TRUE child was missed while its parent was
            # correctly predicted.
            #
            # TRUE:
            #     child  = 1
            #     parent = 1
            #
            # PRED:
            #     child  = 0
            #     parent = 1
            # ------------------------------------------------

            if (
                child_true
                and
                parent_true
                and
                parent_pred
                and
                not child_pred
            ):

                rows.append({

                    "document_index":
                        i,

                    "child":
                        child,

                    "parent":
                        parent,

                    "error_type":
                        "parent_predicted_child_not_predicted",

                    "true_child":
                        1,

                    "true_parent":
                        1,

                    "pred_child":
                        0,

                    "pred_parent":
                        1,

                })

    fi_idx = hierarchical_labels.index("fi")
    in_idx = hierarchical_labels.index("IN")

    print("True fi:", np.sum(true_labels[:, fi_idx] == 1))
    print("True IN:", np.sum(true_labels[:, in_idx] == 1))

    print(
        "True fi + IN:",
        np.sum(
            (true_labels[:, fi_idx] == 1)
            &
            (true_labels[:, in_idx] == 1)
        )
    )

    print(
        "True fi + IN, predicted fi missed:",
        np.sum(
            (true_labels[:, fi_idx] == 1)
            &
            (true_labels[:, in_idx] == 1)
            &
            (predicted_labels[:, fi_idx] == 0)
            &
            (predicted_labels[:, in_idx] == 1)
        )
    )   
    return pd.DataFrame(
        rows
    )



# ============================================================
# HIERARCHICAL ERROR COUNTS
# ============================================================

def hierarchy_summary(
    true_labels,
    predicted_labels,
):

    total = len(
        true_labels
    )


    prediction_violations = 0

    child_correct_parent_missed = 0

    parent_correct_child_missed = 0

    true_hierarchy_violations = 0


    violation_pairs = Counter()

    child_parent_missed_pairs = Counter()

    parent_child_missed_pairs = Counter()


    for i in range(total):

        true_row = true_labels[i]

        pred_row = predicted_labels[i]


        true_violations = (
            get_true_hierarchy_violations(
                true_row
            )
        )


        pred_violations = (
            get_hierarchy_violations(
                pred_row
            )
        )


        if true_violations:

            true_hierarchy_violations += 1


        if pred_violations:

            prediction_violations += 1


            for pair in pred_violations:

                violation_pairs[pair] += 1


        for child, parent in child_to_parent.items():

            child_idx = hierarchical_labels.index(
                child
            )

            parent_idx = hierarchical_labels.index(
                parent
            )


            child_true = (
                true_row[child_idx] == 1
            )

            parent_true = (
                true_row[parent_idx] == 1
            )

            child_pred = (
                pred_row[child_idx] == 1
            )

            parent_pred = (
                pred_row[parent_idx] == 1
            )


            # ------------------------------------------------
            # Correct child, missed parent
            # ------------------------------------------------

            if (

                child_true

                and

                parent_true

                and

                child_pred

                and

                not parent_pred

            ):

                child_correct_parent_missed += 1

                child_parent_missed_pairs[
                    f"{child}->{parent}"
                ] += 1


            # ------------------------------------------------
            # Correct parent, missed true child
            # ------------------------------------------------

            if (

                child_true

                and

                parent_true

                and

                parent_pred

                and

                not child_pred

            ):

                parent_correct_child_missed += 1

                parent_child_missed_pairs[
                    f"{parent}->{child}"
                ] += 1


    return {

        "total_documents":
            total,

        "documents_with_predicted_hierarchy_violation":
            prediction_violations,

        "percent_documents_with_predicted_hierarchy_violation":
            (
                prediction_violations / total * 100
                if total > 0
                else 0
            ),

        "documents_with_true_hierarchy_violation":
            true_hierarchy_violations,

        "child_correct_parent_missed_events":
            child_correct_parent_missed,

        "parent_correct_child_missed_events":
            parent_correct_child_missed,

        "predicted_hierarchy_violation_pairs":
            dict(violation_pairs),

        "child_correct_parent_missed_pairs":
            dict(child_parent_missed_pairs),

        "parent_correct_child_missed_pairs":
            dict(parent_child_missed_pairs),

    }


# ============================================================
# DOCUMENT-LEVEL SUMMARY
# ============================================================

def create_document_summary(
    true_labels,
    predicted_labels,
):

    total = len(
        true_labels
    )


    exact = 0

    partial = 0

    completely_wrong = 0

    completely_missed = 0

    any_error = 0

    hierarchy_violation = 0


    for i in range(total):

        true_set = set(
            label_set(
                true_labels[i]
            )
        )

        pred_set = set(
            label_set(
                predicted_labels[i]
            )
        )


        # ----------------------------------------------------
        # Exact match
        # ----------------------------------------------------

        if true_set == pred_set:

            exact += 1

        else:

            any_error += 1


        # ----------------------------------------------------
        # Partially correct
        # ----------------------------------------------------

        if (

            true_set

            and

            pred_set

            and

            true_set & pred_set

            and

            true_set != pred_set

        ):

            partial += 1


        # ----------------------------------------------------
        # Completely wrong
        # ----------------------------------------------------

        if (

            true_set

            and

            pred_set

            and

            not (true_set & pred_set)

        ):

            completely_wrong += 1


        # ----------------------------------------------------
        # Completely missed
        # ----------------------------------------------------

        if (

            true_set

            and

            not pred_set

        ):

            completely_missed += 1


        # ----------------------------------------------------
        # Hierarchy violation
        # ----------------------------------------------------

        hierarchy_violations = (
            get_hierarchy_violations(
                predicted_labels[i]
            )
        )


        if hierarchy_violations:

            hierarchy_violation += 1


    if total > 0:

        percentages = [

            exact / total * 100,

            any_error / total * 100,

            partial / total * 100,

            completely_wrong / total * 100,

            completely_missed / total * 100,

            hierarchy_violation / total * 100,

        ]

    else:

        percentages = [
            0.0
            for _ in range(6)
        ]


    return pd.DataFrame({

        "category": [

            "exact_match",

            "any_error",

            "partially_correct",

            "completely_wrong",

            "completely_missed",

            "hierarchy_violation",

        ],

        "count": [

            exact,

            any_error,

            partial,

            completely_wrong,

            completely_missed,

            hierarchy_violation,

        ],

        "percentage": percentages,

    })


# ============================================================
# RUN ANALYSIS
# ============================================================

def run_analysis(
    true_labels,
    predicted_labels,
    probabilities,
    prefix,
    description,
):

    print("\n")
    print("=" * 100)
    print(description)
    print("=" * 100)


    # ========================================================
    # 1. DOCUMENT-LEVEL ERROR TABLE
    # ========================================================

    document_df = create_document_error_table(

        true_labels,

        predicted_labels,

        probabilities,

    )


    document_df.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_document_errors.csv",

        index=False,

        encoding="utf-8-sig",

    )


    # --------------------------------------------------------
    # Save only documents containing errors
    # --------------------------------------------------------

    error_documents = document_df[

        ~document_df["exact_match"]

    ]


    error_documents.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_error_documents_only.csv",

        index=False,

        encoding="utf-8-sig",

    )


    # ========================================================
    # 2. EXACT LABEL-SET PATTERNS
    # ========================================================

    pattern_df = create_exact_pattern_table(

        true_labels,

        predicted_labels,

    )


    pattern_df.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_exact_label_set_patterns.csv",

        index=False,

        encoding="utf-8-sig",

    )


    # ========================================================
    # 3. PER-LABEL SUMMARY
    # ========================================================

    label_df = create_per_label_summary(

        true_labels,

        predicted_labels,

    )


    label_df.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_per_label_summary.csv",

        index=False,

        encoding="utf-8-sig",

    )


    # ========================================================
    # 4. MISSED -> FALSE POSITIVE MATRIX
    # ========================================================

    missed_fp_matrix = create_missed_to_fp_matrix(

        true_labels,

        predicted_labels,

    )


    save_matrix(

        missed_fp_matrix,

        f"{prefix}_missed_to_false_positive.csv",

    )


    plot_matrix(

        missed_fp_matrix,

        f"{prefix}_missed_to_false_positive.png",

        (
            "Missed True Label → False-Positive Label\n"
            + description
        ),

        "False-Positive Label",

        "Missed True Label",

    )


    # ========================================================
    # 5. HIERARCHY ERRORS
    # ========================================================

    hierarchy_df = create_hierarchy_error_table(

        true_labels,

        predicted_labels,

    )


    hierarchy_df.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_hierarchy_errors.csv",

        index=False,

        encoding="utf-8-sig",

    )


    # ========================================================
    # 6. HIERARCHY SUMMARY
    # ========================================================

    hierarchy_stats = hierarchy_summary(

        true_labels,

        predicted_labels,

    )


    with open(

        ANALYSIS_DIR
        / f"{prefix}_hierarchy_summary.json",

        "w",

        encoding="utf-8",

    ) as f:

        json.dump(

            hierarchy_stats,

            f,

            indent=2,

        )


    # ========================================================
    # 7. DOCUMENT SUMMARY
    # ========================================================

    doc_summary = create_document_summary(

        true_labels,

        predicted_labels,

    )


    doc_summary.to_csv(

        ANALYSIS_DIR
        / f"{prefix}_document_summary.csv",

        index=False,

        encoding="utf-8-sig",

    )


    # ========================================================
    # 8. PRINT SUMMARY
    # ========================================================

    print("\nDocument-level summary:")

    print(

        doc_summary.to_string(
            index=False
        )

    )


    print("\nHierarchy summary:")

    print(

        json.dumps(

            hierarchy_stats,

            indent=2,

        )

    )


    # ========================================================
    # 9. TOP ERROR PATTERNS
    # ========================================================

    print(
        "\nTop 20 true -> predicted label-set patterns:"
    )


    print(

        pattern_df.head(20).to_string(
            index=False
        )

    )


    # ========================================================
    # 10. TOP MISSED -> FP PAIRS
    # ========================================================

    print(
        "\nTop 20 missed -> false-positive pairs:"
    )


    pair_rows = []


    for i, true_label in enumerate(
        hierarchical_labels
    ):

        for j, predicted_label in enumerate(
            hierarchical_labels
        ):

            count = missed_fp_matrix[i, j]


            if count > 0:

                pair_rows.append({

                    "missed_label":
                        true_label,

                    "false_positive_label":
                        predicted_label,

                    "count":
                        int(count),

                })


    pair_df = pd.DataFrame(
        pair_rows
    )


    if len(pair_df) > 0:

        pair_df = pair_df.sort_values(

            "count",

            ascending=False,

        )


        print(

            pair_df.head(20).to_string(
                index=False
            )

        )

    else:

        print(
            "No missed -> false-positive pairs found."
        )


    # ========================================================
    # 11. TOP HIERARCHY ERRORS
    # ========================================================

    print(
        "\nHierarchy error examples:"
    )


    if len(hierarchy_df) > 0:

        print(

            hierarchy_df.head(20).to_string(
                index=False
            )

        )

    else:

        print(
            "No hierarchy errors found."
        )


# ============================================================
# RUN 1:
# ORIGINAL 0.50
# ============================================================

run_analysis(

    true_labels=test_labels,

    predicted_labels=predictions_05,

    probabilities=test_probabilities,

    prefix="threshold_0.50",

    description=(
        "Threshold = 0.50"
    ),

)


# ============================================================
# RUN 2:
# PARENT 0.50 / GLOBAL CHILD 0.32
# ============================================================

run_analysis(

    true_labels=test_labels,

    predicted_labels=predictions_parent_child_global,

    probabilities=test_probabilities,

    prefix="parent_050_child_global_032",

    description=(

        "Parent threshold = 0.50; "
        "global child threshold = 0.32"

    ),

)


# ============================================================
# RUN 3:
# PER-LABEL TUNED
# ============================================================

run_analysis(

    true_labels=test_labels,

    predicted_labels=predictions_per_label,

    probabilities=test_probabilities,

    prefix="per_label_tuned",

    description=(
        "Per-label tuned thresholds"
    ),

)


# ============================================================
# SAVE THRESHOLDS
# ============================================================

threshold_df = pd.DataFrame({

    "label":
        hierarchical_labels,

    "threshold_0.50":
        threshold_array_050,

    "parent_050_child_global_032":
        threshold_array_parent_child_global,

    "per_label_tuned":
        threshold_array_per_label,

})


threshold_df.to_csv(

    ANALYSIS_DIR
    / "thresholds_used.csv",

    index=False,

    encoding="utf-8-sig",

)


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 100)
print("HIERARCHICAL ERROR ANALYSIS COMPLETE")
print("=" * 100)


print("\nResults saved to:")

print(
    ANALYSIS_DIR
)


print("\nThe most important files are:")


print(
    "  *_document_errors.csv"
)

print(
    "  *_error_documents_only.csv"
)

print(
    "  *_exact_label_set_patterns.csv"
)

print(
    "  *_per_label_summary.csv"
)

print(
    "  *_missed_to_false_positive.csv"
)

print(
    "  *_missed_to_false_positive.png"
)

print(
    "  *_hierarchy_errors.csv"
)

print(
    "  *_hierarchy_summary.json"
)

print(
    "  *_document_summary.csv"
)

print(
    "  thresholds_used.csv"
)


print("=" * 100)



LOADING TEST DATA
Logits shape: (633, 25)
Labels shape: (633, 25)
Labels: ['MT', 'LY', 'SP', 'ID', 'NA', 'HI', 'IN', 'OP', 'IP', 'it', 'ne', 'sr', 'nb', 're', 'en', 'ra', 'dtp', 'fi', 'lt', 'rv', 'ob', 'rs', 'av', 'ds', 'ed']

Parents:
['MT', 'LY', 'SP', 'ID', 'NA', 'HI', 'IN', 'OP', 'IP']

Children:
['it', 'ne', 'sr', 'nb', 're', 'en', 'ra', 'dtp', 'fi', 'lt', 'rv', 'ob', 'rs', 'av', 'ds', 'ed']

THRESHOLDS

   Label |     0.50 |    Parent/Child |    Per-label
-------------------------------------------------------
      MT |     0.50 |            0.50 |         0.50
      LY |     0.50 |            0.50 |         0.50
      SP |     0.50 |            0.50 |         0.50
      ID |     0.50 |            0.50 |         0.50
      NA |     0.50 |            0.50 |         0.50
      HI |     0.50 |            0.50 |         0.50
      IN |     0.50 |            0.50 |         0.50
      OP |     0.50 |            0.50 |         0.50
      IP |     0.50 |            0.50 |         0.50
